# Enhanced Federated Learning Cycle for DeepFake Detection

**Thesis Project — Google Colab Runtime**

# IGNORE BELOW FOR NOW, TO BE UPDATE AS TFF IS NO LONGER USED AND IS REPLACED BY FLOWER

## Modules

| # | Module | Description |
|---|--------|-------------|
| 1 | `enhanced_client_selection.py` | Multi-criteria client selection |
| 2 | `update_validation.py` | Update validation & contribution weighing |
| 3 | `knowledge_distillation.py` | Server-side knowledge distillation |
| 4 | `client_reputation_ledger.py` | Persistent reputation ledger |
| 5 | `evaluation_metrics.py` | Evaluation metrics & reporting |
| 6 | `federated_learning_cycle.py` | Main FL orchestrator (pure Keras) |
| 7 | `tff_data_utils.py` | TFF data management wrapper |
| 8 | `tff_learning_process.py` | TFF model wrapping & process builder |
| 9 | `tff_federated_cycle.py` | TFF-based FL cycle orchestrator |

## Architecture

Flower handles the **core federated computation** (model broadcast →
client-side local training → data-weighted FedAvg). Our thesis
enhancements operate as a **post-aggregation refinement layer**:

1. **Client Selection** (Part 1) → selects participants
2. **TFF Round** → FedAvg on selected clients
3. **Update Validation** (Part 2) → contribution-weighted re-aggregation
4. **Knowledge Distillation** (Part 3) → refine with ensemble KD
5. **Reputation Update** (Part 4) → feed gains into ledger
6. **Evaluation** (Part 5) → periodic full eval with reports
7. Inject enhanced weights back into Flower state for next round

---


## 1. Environment Setup

Install the compatible TensorFlow + TFF stack.

> **Important:** After running the install cell, you may need to
> **restart the runtime** (Runtime → Restart runtime) before
> proceeding. Colab will prompt you if needed.


In [5]:
# ── Install Flower + write flwr_adapter.py to Colab ──────────
# TFF 0.86.0 is archived and incompatible with Python ≥3.12 /
# TF ≥2.17.  The project uses a Flower-backed adapter that
# provides the same API surface as TFF.
# ────────────────────────────────────────────────────────────────
import subprocess, sys, textwrap, pathlib

def _pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    label = ' '.join(a for a in args if not a.startswith('-'))
    r = subprocess.run(cmd, capture_output=True, text=True)
    ok = r.returncode == 0
    status = '✅' if ok else '❌'
    print(f'  {status} {label}')
    if not ok:
        for l in (r.stdout + r.stderr).strip().splitlines()[-3:]:
            print(f'     {l}')
    return ok

print(f'Python     : {sys.version.split()[0]}\n')

# ── Step 1: Install Flower ───────────────────────────────────
print('📦 Step 1/2: Installing Flower (flwr) …')
if not _pip('flwr>=1.7'):
    raise RuntimeError('flwr install failed — see error above')

# ── Step 2: Write flwr_adapter.py ────────────────────────────
print('📦 Step 2/2: Writing flwr_adapter.py …')

_FLWR_ADAPTER_CODE = r'''
"""
Flower (flwr) adapter — drop-in replacement for TFF API surface.

Provides the same classes and functions that the project uses from
``tensorflow_federated``, implemented on top of Flower (flwr).
Works on Python 3.12 + TF 2.19.
"""

from __future__ import annotations

import copy, logging
from collections import OrderedDict
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-8s | %(message)s")
logger = logging.getLogger(__name__)

try:
    import flwr
    FLWR_AVAILABLE = True
except ImportError:
    flwr = None
    FLWR_AVAILABLE = False

def _require_flwr():
    if not FLWR_AVAILABLE:
        raise RuntimeError("Flower (flwr) is not installed.\n"
                           "Install it with:  pip install flwr\n")

# ── 1. ModelWeights ──────────────────────────────────────────
@dataclass
class ModelWeights:
    trainable: List[np.ndarray] = field(default_factory=list)
    non_trainable: List[np.ndarray] = field(default_factory=list)

    def assign_weights_to(self, keras_model: tf.keras.Model) -> None:
        for var, val in zip(keras_model.trainable_variables, self.trainable):
            var.assign(val)
        for var, val in zip(keras_model.non_trainable_variables, self.non_trainable):
            var.assign(val)

# ── 2. FlowerVariableModel ───────────────────────────────────
class FlowerVariableModel:
    def __init__(self, keras_model, input_spec, loss, metrics):
        self.keras_model = keras_model
        self.input_spec = input_spec
        self.loss_fn = loss
        self.metrics_list = metrics

    def get_weights(self) -> ModelWeights:
        return ModelWeights(
            trainable=[v.numpy() for v in self.keras_model.trainable_variables],
            non_trainable=[v.numpy() for v in self.keras_model.non_trainable_variables],
        )

def from_keras_model(keras_model, input_spec, loss=None, metrics=None):
    return FlowerVariableModel(
        keras_model=keras_model,
        input_spec=input_spec,
        loss=loss or tf.keras.losses.BinaryCrossentropy(),
        metrics=metrics or [tf.keras.metrics.BinaryAccuracy()],
    )

# ── 3. Output / State ───────────────────────────────────────
@dataclass
class LearningProcessOutput:
    state: Any
    metrics: OrderedDict

@dataclass
class ServerState:
    model_weights: ModelWeights
    round_num: int = 0
    optimizer_state: Optional[Any] = None

# ── 4. FlowerLearningProcess ────────────────────────────────
class FlowerLearningProcess:
    def __init__(self, model_fn, client_optimizer_fn, server_optimizer_fn):
        self._model_fn = model_fn
        self._client_optimizer_fn = client_optimizer_fn
        self._server_optimizer_fn = server_optimizer_fn
        self._ref_var_model = model_fn()

    def initialize(self) -> ServerState:
        weights = self._ref_var_model.get_weights()
        return ServerState(model_weights=weights, round_num=0)

    def next(self, state, federated_data):
        server_weights = state.model_weights
        client_results = []
        for client_ds in federated_data:
            updated_w, n, m = self._client_train(server_weights, client_ds)
            client_results.append((updated_w, n, m))
        total = sum(n for _, n, _ in client_results) or len(client_results)
        avg_trainable = []
        for i in range(len(server_weights.trainable)):
            ws = np.zeros_like(server_weights.trainable[i])
            for cw, n, _ in client_results:
                ws += cw[i] * (n / total)
            avg_trainable.append(ws)
        new_weights = ModelWeights(
            trainable=avg_trainable,
            non_trainable=list(server_weights.non_trainable),
        )
        agg_metrics = self._aggregate_metrics(client_results)
        new_state = ServerState(model_weights=new_weights,
                                round_num=state.round_num + 1)
        return LearningProcessOutput(state=new_state, metrics=agg_metrics)

    def _client_train(self, server_weights, client_ds):
        var_model = self._model_fn()
        km = var_model.keras_model
        server_weights.assign_weights_to(km)
        opt = self._client_optimizer_fn()
        km.compile(optimizer=opt, loss=var_model.loss_fn,
                   metrics=[type(m)() for m in var_model.metrics_list])
        n = 0
        for batch in client_ds:
            n += batch[0].shape[0] if isinstance(batch, (tuple, list)) else batch.shape[0]
        km.fit(client_ds, epochs=1, verbose=0)
        metrics = {m.name: float(m.result()) for m in km.metrics}
        return [v.numpy() for v in km.trainable_variables], n, metrics

    @staticmethod
    def _aggregate_metrics(client_results):
        total = sum(n for _, n, _ in client_results) or 1
        agg = {}
        for _, n, metrics in client_results:
            for k, v in metrics.items():
                agg[k] = agg.get(k, 0.0) + v * (n / total)
        return OrderedDict([
            ("distributor", OrderedDict()),
            ("client_work", OrderedDict([("train", OrderedDict(agg))])),
            ("aggregator", OrderedDict()),
            ("finalizer", OrderedDict()),
        ])

    def get_model_weights(self, state):
        return state.model_weights

    def set_model_weights(self, state, weights):
        return ServerState(model_weights=weights, round_num=state.round_num,
                           optimizer_state=state.optimizer_state)

# ── 5. build_weighted_fed_avg ───────────────────────────────
def build_weighted_fed_avg(model_fn, client_optimizer_fn=None,
                           server_optimizer_fn=None):
    if client_optimizer_fn is None:
        client_optimizer_fn = lambda: tf.keras.optimizers.Adam(1e-4)
    if server_optimizer_fn is None:
        server_optimizer_fn = lambda: tf.keras.optimizers.SGD(1.0)
    return FlowerLearningProcess(model_fn, client_optimizer_fn,
                                 server_optimizer_fn)

# ── 6. Namespace shims ──────────────────────────────────────
class _ModelsNamespace:
    ModelWeights = ModelWeights
    from_keras_model = staticmethod(from_keras_model)
    VariableModel = FlowerVariableModel

class _AlgorithmsNamespace:
    build_weighted_fed_avg = staticmethod(build_weighted_fed_avg)

class _TemplatesNamespace:
    LearningProcess = FlowerLearningProcess

class _LearningNamespace:
    models = _ModelsNamespace
    algorithms = _AlgorithmsNamespace
    templates = _TemplatesNamespace

class _ClientDataShim:
    def __init__(self, client_ids, dataset_fn):
        self._client_ids = client_ids
        self._dataset_fn = dataset_fn

    @property
    def client_ids(self):
        return self._client_ids

    def create_tf_dataset_for_client(self, client_id):
        return self._dataset_fn(client_id)

    @classmethod
    def from_clients_and_tf_fn(cls, client_ids, serializable_dataset_fn):
        return cls(client_ids, serializable_dataset_fn)

class _DatasetsNamespace:
    ClientData = _ClientDataShim

class _SimulationNamespace:
    datasets = _DatasetsNamespace

class FlowerAsTFF:
    learning = _LearningNamespace
    simulation = _SimulationNamespace

    @property
    def __version__(self):
        return f"flwr-adapter (flwr {flwr.__version__})" if FLWR_AVAILABLE else "flwr-adapter (standalone)"

tff_compat = FlowerAsTFF()
'''

pathlib.Path('flwr_adapter.py').write_text(_FLWR_ADAPTER_CODE.strip(), encoding='utf-8')
print('  ✅ flwr_adapter.py written')

# ── Verify ───────────────────────────────────────────────────
print('\n📋 Installed versions:')
for pkg in ['tensorflow', 'flwr', 'numpy']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True)
    ver = next((l.split(':',1)[1].strip() for l in r.stdout.splitlines()
                if l.startswith('Version:')), 'NOT FOUND')
    print(f'  {pkg:30s} {ver}')

print('\n✅  Flower installed & adapter written — proceed to cell 4.')

Python     : 3.12.12

📦 Step 1/2: Installing Flower (flwr) …
  ✅ flwr>=1.7
📦 Step 2/2: Writing flwr_adapter.py …
  ✅ flwr_adapter.py written

📋 Installed versions:
  tensorflow                     2.19.0
  flwr                           1.26.1
  numpy                          2.0.2

✅  Flower installed & adapter written — proceed to cell 4.


In [6]:
# ── Verify environment (Flower adapter + TF) ──────────────────
import sys
import tensorflow as tf
import numpy as np

# Import the Flower-backed TFF adapter
from flwr_adapter import (
    tff_compat as tff,
    FLWR_AVAILABLE,
    ModelWeights,
    FlowerLearningProcess,
    build_weighted_fed_avg,
    from_keras_model,
)

print(f'Python      : {sys.version.split()[0]}')
print(f'TensorFlow  : {tf.__version__}')
print(f'NumPy       : {np.__version__}')
print(f'GPU         : {tf.config.list_physical_devices("GPU")}')
print(f'Flower      : {"✅ available" if FLWR_AVAILABLE else "⚠️ not installed (standalone mode)"}')
print(f'TFF adapter : {tff.__version__}')
print()

# Quick smoke test: verify the adapter API surface
_ = tff.learning.models.ModelWeights
_ = tff.learning.models.from_keras_model
_ = tff.learning.algorithms.build_weighted_fed_avg
_ = tff.simulation.datasets.ClientData

print('✅ Environment ready — Flower adapter provides TFF-compatible API.')

Python      : 3.12.12
TensorFlow  : 2.19.0
NumPy       : 2.0.2
GPU         : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Flower      : ✅ available
TFF adapter : flwr-adapter (flwr 1.26.1)

✅ Environment ready — Flower adapter provides TFF-compatible API.


## 2. Write Module Files

Each cell below uses `%%writefile` to create the corresponding
Python module in the Colab working directory. Run them all sequentially.


### Part 1 — Enhanced Client Selection (multi-criteria scoring)


In [7]:
%%writefile enhanced_client_selection.py
"""
Enhanced Federated Client Selection Strategy
=============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Implements a multi-criteria client selection mechanism for each federated
round, scoring clients on:
  - Local validation performance  (V)
  - Data volume                   (D)
  - Inference latency             (L)  — penalises slow clients
  - Reputation                    (R)  — maintained across rounds
  - Staleness                     (S)  — penalises long-absent clients
"""

from __future__ import annotations

import math
import time
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  DATA STRUCTURES                                                    #
# ====================================================================== #

@dataclass
class ClientMetrics:
    """Raw metric snapshot collected from / about a single client."""
    local_validation_metric: float = 0.0   # e.g. local F1 on holdout split
    data_volume: int = 0                    # number of local samples
    inference_latency: float = 0.0          # seconds for a forward pass batch
    last_selected_round: int = 0            # last round this client participated


@dataclass
class FederatedClient:
    """
    Represents one federated learning participant.

    Parameters
    ----------
    client_id : str
        Unique human-readable identifier.
    local_data : tf.data.Dataset | None
        The client's private dataset (images + labels).
    metrics : ClientMetrics
        Latest raw metrics for this client.
    """
    client_id: str
    local_data: Optional[tf.data.Dataset] = None
    metrics: ClientMetrics = field(default_factory=ClientMetrics)

    def __repr__(self) -> str:
        return (
            f"FederatedClient(id={self.client_id!r}, "
            f"samples={self.metrics.data_volume})"
        )


# ====================================================================== #
#  2.  REPUTATION LEDGER                                                  #
# ====================================================================== #

class ReputationLedger:
    """
    Tracks a per-client reputation score in [0, 1].

    Reputation increases when the client's model update *improves* the
    global validation metric and decreases (or stays flat) otherwise.
    An exponential moving average (EMA) keeps the score smooth.

    Parameters
    ----------
    default_reputation : float
        Initial reputation assigned to every newly registered client.
    ema_alpha : float
        Smoothing factor for the EMA update  (higher → faster adaptation).
    reward : float
        Reputation bump for a *good* update.
    penalty : float
        Reputation decrease for a *bad* or *no-op* update.
    """

    def __init__(
        self,
        default_reputation: float = 0.5,
        ema_alpha: float = 0.3,
        reward: float = 0.1,
        penalty: float = 0.05,
    ) -> None:
        self.default_reputation = default_reputation
        self.ema_alpha = ema_alpha
        self.reward = reward
        self.penalty = penalty
        self._scores: Dict[str, float] = {}

    # ---- public API -------------------------------------------------- #

    def register(self, client_id: str) -> None:
        """Register a client with the default reputation."""
        if client_id not in self._scores:
            self._scores[client_id] = self.default_reputation

    def get(self, client_id: str) -> float:
        """Return the current reputation for *client_id*."""
        return self._scores.get(client_id, self.default_reputation)

    def update(self, client_id: str, update_was_beneficial: bool) -> None:
        """
        Update the reputation of *client_id* after a federated round.

        Parameters
        ----------
        client_id : str
        update_was_beneficial : bool
            ``True`` if the client's local update improved (or at least did
            not degrade) the global model on a held-out validation set.
        """
        old = self.get(client_id)
        delta = self.reward if update_was_beneficial else -self.penalty
        new = old + self.ema_alpha * delta
        self._scores[client_id] = float(np.clip(new, 0.0, 1.0))
        logger.debug(
            "Reputation %s: %.4f → %.4f (beneficial=%s)",
            client_id, old, self._scores[client_id], update_was_beneficial,
        )

    def summary(self) -> Dict[str, float]:
        """Return a copy of the full reputation table."""
        return dict(self._scores)


# ====================================================================== #
#  3.  NORMALISATION & SCORING HELPERS                                    #
# ====================================================================== #

def _min_max_normalise(values: np.ndarray) -> np.ndarray:
    """Min-max normalise an array to [0, 1].  Returns zeros when range = 0."""
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-12:
        return np.zeros_like(values, dtype=np.float64)
    return (values - lo) / (hi - lo)


def _log_scale(values: np.ndarray) -> np.ndarray:
    """Log-scale data volumes before normalising (handles zero gracefully)."""
    return np.log1p(values.astype(np.float64))


def staleness_penalty(last_selected_round: int, current_round: int) -> float:
    """
    Compute a staleness penalty in [0, 1].

    Uses an exponential decay:  ``1 - exp(-gap / 5)``  so that a client
    absent for ~15 rounds is nearly fully penalised.
    """
    gap = max(current_round - last_selected_round, 0)
    return 1.0 - math.exp(-gap / 5.0)


# ====================================================================== #
#  4.  ENHANCED CLIENT SELECTION STRATEGY                                 #
# ====================================================================== #

@dataclass
class SelectionWeights:
    """Tuneable weights for the multi-criteria scoring function."""
    w_v: float = 0.30   # local validation performance
    w_d: float = 0.20   # data volume
    w_l: float = 0.15   # latency  (applied to 1 - L_i)
    w_r: float = 0.25   # reputation
    w_s: float = 0.10   # staleness penalty (subtracted)

    def as_tuple(self) -> Tuple[float, ...]:
        return (self.w_v, self.w_d, self.w_l, self.w_r, self.w_s)


class EnhancedClientSelector:
    """
    Implements the *Enhanced Federated Client Selection Strategy*.

    Per round the selector:
      1. Collects raw metrics from every candidate client.
      2. Normalises them across the pool.
      3. Computes a composite score per client.
      4. Returns the top-K (or above-threshold) clients.

    Parameters
    ----------
    clients : list[FederatedClient]
        The full pool of federated clients.
    reputation_ledger : ReputationLedger
        Shared ledger that persists across rounds.
    weights : SelectionWeights
        Relative importance of each scoring dimension.
    target_k : int
        Number of clients to select each round (top-K mode).
    threshold : float | None
        If set, *all* clients with ``score >= threshold`` are selected
        instead of a fixed K.
    """

    def __init__(
        self,
        clients: List[FederatedClient],
        reputation_ledger: ReputationLedger,
        weights: Optional[SelectionWeights] = None,
        target_k: int = 5,
        threshold: Optional[float] = None,
    ) -> None:
        self.clients = {c.client_id: c for c in clients}
        self.ledger = reputation_ledger
        self.weights = weights or SelectionWeights()
        self.target_k = target_k
        self.threshold = threshold

        # Register every client in the ledger
        for cid in self.clients:
            self.ledger.register(cid)

    # ------------------------------------------------------------------ #
    #  Core selection logic                                               #
    # ------------------------------------------------------------------ #

    def score_clients(
        self, current_round: int
    ) -> List[Tuple[str, float]]:
        """
        Compute and return ``(client_id, score)`` for every client,
        sorted descending by score.
        """
        ids = list(self.clients.keys())
        n = len(ids)

        # --- gather raw vectors ---------------------------------------- #
        raw_v = np.array([self.clients[i].metrics.local_validation_metric for i in ids])
        raw_d = np.array([self.clients[i].metrics.data_volume for i in ids])
        raw_l = np.array([self.clients[i].metrics.inference_latency for i in ids])

        # --- normalise ------------------------------------------------- #
        V = _min_max_normalise(raw_v)                    # higher is better
        D = _min_max_normalise(_log_scale(raw_d))        # log-scaled, higher is better
        L = _min_max_normalise(raw_l)                    # 1 = slowest (penalty)
        R = np.array([self.ledger.get(i) for i in ids])  # already in [0,1]
        S = np.array([
            staleness_penalty(self.clients[i].metrics.last_selected_round, current_round)
            for i in ids
        ])

        # --- composite score ------------------------------------------- #
        w = self.weights
        scores = (
            w.w_v * V
            + w.w_d * D
            + w.w_l * (1.0 - L)
            + w.w_r * R
            - w.w_s * S
        )

        ranked = sorted(zip(ids, scores), key=lambda x: x[1], reverse=True)
        return ranked

    def select(
        self, current_round: int
    ) -> List[FederatedClient]:
        """
        Select clients for the current federated round.

        Returns
        -------
        list[FederatedClient]
            The chosen participants (top-K or above-threshold).
        """
        ranked = self.score_clients(current_round)

        if self.threshold is not None:
            selected_ids = [cid for cid, sc in ranked if sc >= self.threshold]
            # Guarantee at least 1 client even when none meets threshold
            if not selected_ids:
                selected_ids = [ranked[0][0]]
                logger.warning(
                    "Round %d — no client met threshold %.3f; "
                    "falling back to best client %s (score %.4f).",
                    current_round, self.threshold,
                    ranked[0][0], ranked[0][1],
                )
        else:
            k = min(self.target_k, len(ranked))
            selected_ids = [cid for cid, _ in ranked[:k]]

        logger.info(
            "Round %d — selected %d / %d clients: %s",
            current_round, len(selected_ids), len(ranked), selected_ids,
        )
        return [self.clients[cid] for cid in selected_ids]


# ====================================================================== #
#  5.  FEDERATED ROUND RUNNER  (skeleton compatible with TFF)             #
# ====================================================================== #

class FederatedRoundRunner:
    """
    Orchestrates the full Enhanced Federated Learning cycle.

    This class ties together:
      * The global model (loaded from the .h5 file)
      * The enhanced client selector
      * Local training on selected clients
      * Federated averaging of model updates
      * Reputation ledger updates after each round

    Parameters
    ----------
    global_model_path : str
        Path to the initial global Keras model (``*.h5``).
    clients : list[FederatedClient]
        All available federated clients.
    selector : EnhancedClientSelector
        The client selection strategy instance.
    local_epochs : int
        Number of local training epochs per client per round.
    local_batch_size : int
        Batch size for local training.
    learning_rate : float
        Learning rate for local SGD.
    """

    def __init__(
        self,
        global_model_path: str,
        clients: List[FederatedClient],
        selector: EnhancedClientSelector,
        local_epochs: int = 1,
        local_batch_size: int = 32,
        learning_rate: float = 1e-4,
    ) -> None:
        # Register EfficientNet preprocessing so Keras can deserialize the .h5
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        _custom = {"preprocess_input": _effnet_preprocess}
        self.global_model: tf.keras.Model = tf.keras.models.load_model(
            global_model_path, custom_objects=_custom, compile=False,
        )
        self.global_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        self.clients = clients
        self.selector = selector
        self.local_epochs = local_epochs
        self.local_batch_size = local_batch_size
        self.learning_rate = learning_rate

    # ------------------------------------------------------------------ #
    #  Federated Averaging helpers                                        #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _fedavg(
        global_weights: List[np.ndarray],
        client_weights_list: List[List[np.ndarray]],
        sample_counts: List[int],
    ) -> List[np.ndarray]:
        """
        Weighted Federated Averaging (FedAvg).

        Each client's contribution is proportional to its local sample count.
        """
        total = sum(sample_counts)
        averaged = [
            np.zeros_like(w) for w in global_weights
        ]
        for cw, n in zip(client_weights_list, sample_counts):
            for idx, w in enumerate(cw):
                averaged[idx] += w * (n / total)
        return averaged

    # ------------------------------------------------------------------ #
    #  Local training on a single client                                  #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int, float]:
        """
        Perform local training on *client* starting from *global_weights*.

        Returns
        -------
        updated_weights : list[np.ndarray]
        num_samples : int
        local_val_metric : float   (e.g. accuracy on local holdout)
        """
        # Build a fresh copy with global weights
        local_model: tf.keras.Model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(self.learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            logger.warning("Client %s has no local data — skipping.", client.client_id)
            return global_weights, 0, 0.0

        dataset = client.local_data.batch(self.local_batch_size)

        # Measure inference latency (single batch)
        t0 = time.perf_counter()
        for batch in dataset.take(1):
            local_model.predict(batch[0], verbose=0)
        client.metrics.inference_latency = time.perf_counter() - t0

        # Local training
        local_model.fit(dataset, epochs=self.local_epochs, verbose=0)

        # Evaluate on same data as a proxy (in production: use a held-out split)
        results = local_model.evaluate(dataset, verbose=0, return_dict=True)
        local_val = results.get("accuracy", 0.0)

        num_samples = client.metrics.data_volume

        return local_model.get_weights(), num_samples, local_val

    # ------------------------------------------------------------------ #
    #  Validate an individual client update against the global model      #
    # ------------------------------------------------------------------ #

    def _validate_update(
        self,
        updated_weights: List[np.ndarray],
        validation_data: Optional[tf.data.Dataset],
    ) -> Optional[float]:
        """
        Evaluate *updated_weights* on a global validation set.

        Returns the accuracy (or ``None`` if no validation data is provided).
        """
        if validation_data is None:
            return None
        temp_model: tf.keras.Model = tf.keras.models.clone_model(self.global_model)
        temp_model.build(self.global_model.input_shape)
        temp_model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        temp_model.set_weights(updated_weights)
        results = temp_model.evaluate(
            validation_data.batch(self.local_batch_size), verbose=0, return_dict=True,
        )
        return results.get("accuracy", 0.0)

    # ------------------------------------------------------------------ #
    #  Main loop                                                          #
    # ------------------------------------------------------------------ #

    def run(
        self,
        num_rounds: int = 10,
        global_val_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Execute the full federated learning cycle.

        Parameters
        ----------
        num_rounds : int
            Total communication rounds ``T``.
        global_val_data : tf.data.Dataset | None
            A small global validation set used to judge update quality
            (for the reputation ledger).

        Returns
        -------
        history : dict
            ``{"round": [...], "global_acc": [...], "selected_clients": [...]}``
        """
        history: Dict[str, list] = {
            "round": [],
            "global_accuracy": [],
            "selected_clients": [],
        }

        # Compute baseline accuracy before federated training
        if global_val_data is not None:
            baseline = self.global_model.evaluate(
                global_val_data.batch(self.local_batch_size),
                verbose=0,
                return_dict=True,
            )
            logger.info("Baseline global accuracy: %.4f", baseline.get("accuracy", 0.0))

        for t in range(1, num_rounds + 1):
            logger.info("=" * 60)
            logger.info("ROUND %d / %d", t, num_rounds)
            logger.info("=" * 60)

            # ---- 1. Select clients ----------------------------------- #
            selected = self.selector.select(current_round=t)

            global_weights = self.global_model.get_weights()
            client_updates: List[List[np.ndarray]] = []
            sample_counts: List[int] = []

            # ---- 2. Local training ----------------------------------- #
            for client in selected:
                updated_w, n_samples, local_val = self._local_train(
                    client, global_weights
                )
                client.metrics.local_validation_metric = local_val
                client.metrics.last_selected_round = t

                client_updates.append(updated_w)
                sample_counts.append(max(n_samples, 1))  # avoid div-by-zero

                # ---- 3. Reputation update ----------------------------- #
                if global_val_data is not None:
                    update_acc = self._validate_update(updated_w, global_val_data)
                    current_acc = self._validate_update(global_weights, global_val_data)
                    beneficial = (
                        update_acc is not None
                        and current_acc is not None
                        and update_acc >= current_acc - 1e-4
                    )
                else:
                    # Optimistic: assume beneficial when we cannot verify
                    beneficial = True

                self.selector.ledger.update(client.client_id, beneficial)

            # ---- 4. Federated averaging ------------------------------ #
            if client_updates:
                new_global = self._fedavg(global_weights, client_updates, sample_counts)
                self.global_model.set_weights(new_global)

            # ---- 5. Global evaluation -------------------------------- #
            round_acc = None
            if global_val_data is not None:
                result = self.global_model.evaluate(
                    global_val_data.batch(self.local_batch_size),
                    verbose=0,
                    return_dict=True,
                )
                round_acc = result.get("accuracy", 0.0)
                logger.info("Round %d global accuracy: %.4f", t, round_acc)

            history["round"].append(t)
            history["global_accuracy"].append(round_acc)
            history["selected_clients"].append([c.client_id for c in selected])

        logger.info("Federated training complete — %d rounds.", num_rounds)
        return history


# ====================================================================== #
#  6.  CONVENIENCE FACTORY                                                #
# ====================================================================== #

def build_default_pipeline(
    model_path: str = "effnet_ffpp_small_data.h5",
    num_clients: int = 10,
    target_k: int = 5,
    threshold: Optional[float] = None,
    weights: Optional[SelectionWeights] = None,
) -> Tuple[FederatedRoundRunner, List[FederatedClient]]:
    """
    Quick-start helper that wires up all components.

    In practice you would replace the synthetic client list with real
    data partitions.

    Returns
    -------
    runner : FederatedRoundRunner
    clients : list[FederatedClient]
    """
    # --- Create placeholder clients (replace with real data loaders) --- #
    clients: List[FederatedClient] = []
    for i in range(num_clients):
        c = FederatedClient(
            client_id=f"client_{i:02d}",
            local_data=None,           # <-- plug real tf.data.Dataset here
            metrics=ClientMetrics(
                local_validation_metric=np.random.uniform(0.5, 0.95),
                data_volume=int(np.random.randint(200, 5000)),
                inference_latency=np.random.uniform(0.01, 0.5),
                last_selected_round=0,
            ),
        )
        clients.append(c)

    ledger = ReputationLedger()
    selector = EnhancedClientSelector(
        clients=clients,
        reputation_ledger=ledger,
        weights=weights or SelectionWeights(),
        target_k=target_k,
        threshold=threshold,
    )
    runner = FederatedRoundRunner(
        global_model_path=model_path,
        clients=clients,
        selector=selector,
    )
    return runner, clients


# ====================================================================== #
#  7.  MAIN — demo / smoke test                                           #
# ====================================================================== #

if __name__ == "__main__":
    # -------------------------------------------------------------- #
    #  Standalone demo:  runs selection scoring with synthetic clients #
    #  (no real data or model loading — safe to execute anywhere).    #
    # -------------------------------------------------------------- #

    NUM_CLIENTS = 10
    TARGET_K = 4
    NUM_ROUNDS = 5

    # Create synthetic client pool
    np.random.seed(42)
    demo_clients: List[FederatedClient] = []
    for idx in range(NUM_CLIENTS):
        demo_clients.append(
            FederatedClient(
                client_id=f"client_{idx:02d}",
                local_data=None,
                metrics=ClientMetrics(
                    local_validation_metric=np.random.uniform(0.4, 0.95),
                    data_volume=int(np.random.randint(100, 8000)),
                    inference_latency=np.random.uniform(0.01, 1.0),
                    last_selected_round=0,
                ),
            )
        )

    ledger = ReputationLedger()
    selector = EnhancedClientSelector(
        clients=demo_clients,
        reputation_ledger=ledger,
        weights=SelectionWeights(w_v=0.30, w_d=0.20, w_l=0.15, w_r=0.25, w_s=0.10),
        target_k=TARGET_K,
    )

    print("\n===  Enhanced Client Selection — Demo  ===\n")
    for rnd in range(1, NUM_ROUNDS + 1):
        ranked = selector.score_clients(current_round=rnd)
        selected = selector.select(current_round=rnd)

        print(f"\n--- Round {rnd} ---")
        print(f"{'Client':<14} {'Score':>8}")
        print("-" * 24)
        for cid, score in ranked:
            marker = " ✓" if cid in {c.client_id for c in selected} else ""
            print(f"{cid:<14} {score:>8.4f}{marker}")

        # Simulate reputation changes (random for demo)
        for c in selected:
            beneficial = np.random.random() > 0.3   # 70 % chance of good update
            ledger.update(c.client_id, bool(beneficial))
            c.metrics.last_selected_round = rnd

    print("\n--- Final Reputation Ledger ---")
    for cid, rep in sorted(ledger.summary().items()):
        print(f"  {cid}: {rep:.4f}")


Overwriting enhanced_client_selection.py


### Part 2 — Update Validation & Contribution Weighing


In [8]:
%%writefile update_validation.py
"""
Update Validation and Contribution Weighing
============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

After each federated round, every client update is individually validated
and assigned a **contribution weight** before aggregation.  The pipeline:

  1. Norm check — flag / clip suspiciously large updates.
  2. Server-side validation gain — apply the update to a temp copy of the
     global model and measure the score delta on a held-out server set.
  3. Similarity check — cosine similarity with the recent global update
     history (catches free-riders that echo old gradients).
  4. Multi-criteria raw contribution score.
  5. Weighted aggregation (contribution-weighted FedAvg).
  6. Reputation ledger feedback from observed gains.

Imports the shared data-structures from ``enhanced_client_selection.py``.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- shared types from Part 1 ---------------------------------- #
from enhanced_client_selection import (
    FederatedClient,
    ClientMetrics,
    ReputationLedger,
    _min_max_normalise,
    _log_scale,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class ContributionWeights:
    """
    Tuneable weights for the contribution scoring formula.

    ``raw = α·G_i + β·sim_i + γ·norm(D_i) + δ·R_i``
    """
    alpha: float = 0.40   # server-side validation gain
    beta:  float = 0.15   # cosine similarity to global update history
    gamma: float = 0.20   # normalised data volume
    delta: float = 0.25   # reputation

    def as_tuple(self) -> Tuple[float, ...]:
        return (self.alpha, self.beta, self.gamma, self.delta)


@dataclass
class ClippingConfig:
    """Parameters for the norm-based update clipping / rejection."""
    clip_threshold: float = 10.0     # max allowed L2 norm of a flattened update
    clip_value: Optional[float] = None  # if set, clip *to* this norm instead of rejecting


# ====================================================================== #
#  2.  HELPER UTILITIES                                                   #
# ====================================================================== #

def flatten_weights(weights: List[np.ndarray]) -> np.ndarray:
    """Concatenate a list of weight arrays into a single 1-D vector."""
    return np.concatenate([w.ravel() for w in weights])


def unflatten_weights(
    flat: np.ndarray,
    shapes: List[Tuple[int, ...]],
) -> List[np.ndarray]:
    """Inverse of ``flatten_weights``: split a 1-D vector back into arrays."""
    arrays: List[np.ndarray] = []
    offset = 0
    for shape in shapes:
        size = int(np.prod(shape))
        arrays.append(flat[offset : offset + size].reshape(shape))
        offset += size
    return arrays


def compute_update_delta(
    global_weights: List[np.ndarray],
    updated_weights: List[np.ndarray],
) -> List[np.ndarray]:
    """Return the element-wise difference  ``updated − global``."""
    return [u - g for u, g in zip(updated_weights, global_weights)]


def apply_update(
    base_weights: List[np.ndarray],
    delta: List[np.ndarray],
    scale: float = 1.0,
) -> List[np.ndarray]:
    """Return ``base + scale * delta`` (per-layer)."""
    return [b + scale * d for b, d in zip(base_weights, delta)]


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two 1-D vectors.

    Returns 0.0 when either vector has near-zero norm (avoids NaN).
    """
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a < 1e-12 or norm_b < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


def _normalise_scalar_to_01(
    values: np.ndarray,
) -> np.ndarray:
    """Scale an array into [0, 1] via min-max.  Alias for readability."""
    return _min_max_normalise(values)


# ====================================================================== #
#  3.  GLOBAL UPDATE HISTORY                                              #
# ====================================================================== #

class GlobalUpdateHistory:
    """
    Maintains a rolling window of the last *N* aggregated global updates
    (as flattened vectors) so the validator can compute cosine similarity
    between a client update and "what the model has been learning lately".

    Parameters
    ----------
    max_history : int
        Maximum number of past global deltas to keep.
    """

    def __init__(self, max_history: int = 10) -> None:
        self.max_history = max_history
        self._history: List[np.ndarray] = []   # each entry is a 1-D vector

    def push(self, global_delta_flat: np.ndarray) -> None:
        """Append the latest aggregated global delta."""
        self._history.append(global_delta_flat.copy())
        if len(self._history) > self.max_history:
            self._history.pop(0)

    @property
    def mean_direction(self) -> Optional[np.ndarray]:
        """
        Return the mean direction of stored history.

        This single vector captures the *average trend* the global model
        has been moving in.  Returns ``None`` before the first round.
        """
        if not self._history:
            return None
        stacked = np.stack(self._history, axis=0)
        return stacked.mean(axis=0)

    @property
    def size(self) -> int:
        return len(self._history)


# ====================================================================== #
#  4.  UPDATE VALIDATOR & CONTRIBUTION SCORER                             #
# ====================================================================== #

@dataclass
class ClientUpdateRecord:
    """Result of validating a single client's update."""
    client_id: str
    delta: List[np.ndarray]          # raw weight delta  (updated − global)
    norm: float = 0.0                # L2 norm of flattened delta
    is_suspicious: bool = False      # flagged by norm check
    validation_gain: float = 0.0     # G_i = new_score − baseline_score
    similarity: float = 0.0         # cosine similarity with history
    raw_contribution: float = 0.0    # before normalisation
    contribution_weight: float = 0.0 # final c_i in [0, 1]
    rejected: bool = False           # update completely rejected


class UpdateValidator:
    """
    Validates client updates and computes contribution weights.

    This is the **core class** of the second part of the federated cycle.

    Parameters
    ----------
    global_model : tf.keras.Model
        The current global model (used to create temp copies for eval).
    reputation_ledger : ReputationLedger
        Shared reputation tracker (read for R_i, written after scoring).
    weights : ContributionWeights
        α, β, γ, δ for the composite score.
    clipping : ClippingConfig
        Norm-clipping parameters.
    harmful_threshold : float
        ε — if ``G_i < −ε`` the update is rejected outright.
    batch_size : int
        Batch size when evaluating on the server validation set.
    eval_metric : str
        Name of the Keras metric to use as *score* (e.g. ``"accuracy"``
        or ``"f1_score"``).
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        reputation_ledger: ReputationLedger,
        weights: Optional[ContributionWeights] = None,
        clipping: Optional[ClippingConfig] = None,
        harmful_threshold: float = 0.02,
        batch_size: int = 32,
        eval_metric: str = "accuracy",
    ) -> None:
        self.global_model = global_model
        self.ledger = reputation_ledger
        self.weights = weights or ContributionWeights()
        self.clipping = clipping or ClippingConfig()
        self.harmful_threshold = harmful_threshold
        self.batch_size = batch_size
        self.eval_metric = eval_metric
        self.update_history = GlobalUpdateHistory()

    # ------------------------------------------------------------------ #
    #  Evaluation helper                                                  #
    # ------------------------------------------------------------------ #

    def _evaluate(
        self,
        model_weights: List[np.ndarray],
        val_data: tf.data.Dataset,
    ) -> float:
        """
        Evaluate *model_weights* on *val_data* by creating a temporary
        model clone, setting the weights, and running ``evaluate()``.

        Returns the scalar value of ``self.eval_metric``.
        """
        temp: tf.keras.Model = tf.keras.models.clone_model(self.global_model)
        temp.build(self.global_model.input_shape)
        temp.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        temp.set_weights(model_weights)
        results = temp.evaluate(
            val_data.batch(self.batch_size), verbose=0, return_dict=True,
        )
        return float(results.get(self.eval_metric, 0.0))

    # ------------------------------------------------------------------ #
    #  Norm check                                                         #
    # ------------------------------------------------------------------ #

    def _norm_check(
        self,
        delta_flat: np.ndarray,
    ) -> Tuple[bool, np.ndarray]:
        """
        Check if the update norm exceeds the clip threshold.

        Returns
        -------
        is_suspicious : bool
        clipped_flat : np.ndarray
            The (possibly clipped) flat delta.
        """
        norm = float(np.linalg.norm(delta_flat))
        if norm <= self.clipping.clip_threshold:
            return False, delta_flat

        logger.warning(
            "Update norm %.4f exceeds threshold %.4f",
            norm, self.clipping.clip_threshold,
        )
        if self.clipping.clip_value is not None:
            # Clip to allowed magnitude instead of outright rejection
            scale = self.clipping.clip_value / (norm + 1e-12)
            return True, delta_flat * scale
        # No clip_value ⇒ hard rejection (caller sets weight = 0)
        return True, delta_flat

    # ------------------------------------------------------------------ #
    #  Main validation pipeline                                           #
    # ------------------------------------------------------------------ #

    def validate_updates(
        self,
        client_updates: Dict[str, List[np.ndarray]],
        data_volumes: Dict[str, int],
        server_val_data: tf.data.Dataset,
    ) -> List[ClientUpdateRecord]:
        """
        Validate every client update and assign contribution weights.

        Parameters
        ----------
        client_updates : dict[str, list[np.ndarray]]
            Mapping ``client_id → updated model weights`` (full weights,
            **not** deltas — deltas are computed internally).
        data_volumes : dict[str, int]
            Mapping ``client_id → number of local training samples``.
        server_val_data : tf.data.Dataset
            The server-side held-out validation set.

        Returns
        -------
        records : list[ClientUpdateRecord]
            One record per client, with ``contribution_weight`` set.
        """
        global_weights = self.global_model.get_weights()
        shapes = [w.shape for w in global_weights]
        global_flat = flatten_weights(global_weights)

        # ---- 0. Baseline score on server val set ---------------------- #
        baseline_score = self._evaluate(global_weights, server_val_data)
        logger.info("Baseline server score (%s): %.4f", self.eval_metric, baseline_score)

        records: List[ClientUpdateRecord] = []
        gains: List[float] = []       # for min-max normalisation later
        sims: List[float] = []
        raw_data_vols: List[int] = []
        reps: List[float] = []

        # ---- Per-client loop ----------------------------------------- #
        for cid, updated_weights in client_updates.items():
            delta = compute_update_delta(global_weights, updated_weights)
            delta_flat = flatten_weights(delta)
            norm = float(np.linalg.norm(delta_flat))

            rec = ClientUpdateRecord(client_id=cid, delta=delta, norm=norm)

            # 1.  Norm check
            is_suspicious, clipped_flat = self._norm_check(delta_flat)
            rec.is_suspicious = is_suspicious

            if is_suspicious and self.clipping.clip_value is None:
                # Hard rejection — set weight = 0, skip evaluation
                rec.rejected = True
                rec.contribution_weight = 0.0
                records.append(rec)
                gains.append(0.0)
                sims.append(0.0)
                raw_data_vols.append(data_volumes.get(cid, 0))
                reps.append(self.ledger.get(cid))
                logger.info(
                    "Client %s REJECTED (norm %.4f > %.4f, no clip_value).",
                    cid, norm, self.clipping.clip_threshold,
                )
                continue

            # Possibly overwrite delta with clipped version
            if is_suspicious:
                delta = unflatten_weights(clipped_flat, shapes)
                rec.delta = delta

            # 2.  Server-side validation gain
            temp_weights = apply_update(global_weights, delta, scale=1.0)
            new_score = self._evaluate(temp_weights, server_val_data)
            G_i = new_score - baseline_score
            rec.validation_gain = G_i

            # 3.  Similarity check
            hist_dir = self.update_history.mean_direction
            if hist_dir is not None:
                sim_i = cosine_similarity(flatten_weights(delta), hist_dir)
            else:
                sim_i = 0.5   # neutral when no history yet
            rec.similarity = sim_i

            gains.append(G_i)
            sims.append(sim_i)
            raw_data_vols.append(data_volumes.get(cid, 0))
            reps.append(self.ledger.get(cid))
            records.append(rec)

        # ---- 4. Combine into normalised contribution weights ---------- #
        n = len(records)
        if n == 0:
            return records

        arr_G = np.array(gains, dtype=np.float64)
        arr_sim = np.array(sims, dtype=np.float64)
        arr_D = _normalise_scalar_to_01(_log_scale(np.array(raw_data_vols, dtype=np.float64)))
        arr_R = np.array(reps, dtype=np.float64)          # already [0, 1]

        w = self.weights
        raw_scores = (
            w.alpha * arr_G
            + w.beta  * arr_sim
            + w.gamma * arr_D
            + w.delta * arr_R
        )

        # Normalise raw scores into [0, 1]
        c = _normalise_scalar_to_01(raw_scores)

        # Reject strongly harmful updates  (G_i < −ε)
        for idx, rec in enumerate(records):
            if rec.rejected:
                c[idx] = 0.0
                continue
            rec.raw_contribution = float(raw_scores[idx])
            rec.contribution_weight = float(c[idx])

            if rec.validation_gain < -self.harmful_threshold:
                rec.contribution_weight = 0.0
                rec.rejected = True
                logger.info(
                    "Client %s rejected — G_i=%.4f < −ε (%.4f).",
                    rec.client_id, rec.validation_gain, self.harmful_threshold,
                )

        return records

    # ------------------------------------------------------------------ #
    #  5. Weighted aggregation                                            #
    # ------------------------------------------------------------------ #

    def aggregate_weighted(
        self,
        records: List[ClientUpdateRecord],
        global_weights: Optional[List[np.ndarray]] = None,
    ) -> List[np.ndarray]:
        """
        Contribution-weighted aggregation of client deltas.

        ``new_global = global + Σ_i  (c_i / Σ c_j) · delta_i``

        Parameters
        ----------
        records : list[ClientUpdateRecord]
            Output of ``validate_updates``.
        global_weights : list[np.ndarray] | None
            If ``None``, reads from ``self.global_model``.

        Returns
        -------
        new_global_weights : list[np.ndarray]
        """
        if global_weights is None:
            global_weights = self.global_model.get_weights()

        # Filter to accepted updates with positive weight
        active = [(r.delta, r.contribution_weight) for r in records
                  if not r.rejected and r.contribution_weight > 0]

        if not active:
            logger.warning("No valid updates this round — global model unchanged.")
            return global_weights

        total_c = sum(c for _, c in active)
        aggregated_delta = [np.zeros_like(w) for w in global_weights]

        for delta, c_i in active:
            weight = c_i / total_c
            for idx, d in enumerate(delta):
                aggregated_delta[idx] += weight * d

        new_weights = apply_update(global_weights, aggregated_delta)

        # Push this aggregated delta into the history for future similarity
        self.update_history.push(flatten_weights(aggregated_delta))

        return new_weights

    # ------------------------------------------------------------------ #
    #  6. Reputation feedback                                             #
    # ------------------------------------------------------------------ #

    def update_reputations(
        self,
        records: List[ClientUpdateRecord],
    ) -> None:
        """
        Feed observed validation gains and contribution weights back into
        the reputation ledger.

        - Clients with ``G_i > 0`` and ``c_i > 0`` are rewarded.
        - Clients with ``G_i < 0`` or no contribution are penalised.
        - Rejected clients receive a stronger penalty.
        """
        for rec in records:
            if rec.rejected:
                # Stronger penalty for rejected / harmful updates
                self.ledger.update(rec.client_id, update_was_beneficial=False)
                self.ledger.update(rec.client_id, update_was_beneficial=False)
                logger.debug("Reputation double-penalty for %s (rejected).", rec.client_id)
            elif rec.validation_gain > 0 and rec.contribution_weight > 0:
                self.ledger.update(rec.client_id, update_was_beneficial=True)
            else:
                self.ledger.update(rec.client_id, update_was_beneficial=False)


# ====================================================================== #
#  5.  FULL ROUND RUNNER  (wraps Part 1 selection + Part 2 validation)    #
# ====================================================================== #

class ValidatedFederatedRound:
    """
    End-to-end runner for one federated round that integrates:

    * **Part 1** — Enhanced client selection
    * **Part 2** — Update validation & contribution-weighted aggregation

    Parameters
    ----------
    global_model : tf.keras.Model
        The current global model (mutated in-place each round).
    clients : list[FederatedClient]
        Full client pool.
    selector
        An ``EnhancedClientSelector`` instance (Part 1).
    validator : UpdateValidator
        The update-validation & scoring engine (Part 2).
    local_epochs : int
        Local training epochs per client.
    local_batch_size : int
        Batch size for local training.
    learning_rate : float
        Client-side optimiser learning rate.
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        clients: List[FederatedClient],
        selector,                          # EnhancedClientSelector
        validator: UpdateValidator,
        local_epochs: int = 1,
        local_batch_size: int = 32,
        learning_rate: float = 1e-4,
    ) -> None:
        self.global_model = global_model
        self.clients = {c.client_id: c for c in clients}
        self.selector = selector
        self.validator = validator
        self.local_epochs = local_epochs
        self.local_batch_size = local_batch_size
        self.learning_rate = learning_rate

    # ------------------------------------------------------------------ #
    #  Local training helper                                              #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int]:
        """
        Run local training on *client* starting from *global_weights*.

        Returns ``(updated_weights, num_samples)``.
        """
        local_model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(self.learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            logger.warning("Client %s has no data — returning global weights.", client.client_id)
            return global_weights, 0

        dataset = client.local_data.batch(self.local_batch_size)
        local_model.fit(dataset, epochs=self.local_epochs, verbose=0)
        return local_model.get_weights(), client.metrics.data_volume

    # ------------------------------------------------------------------ #
    #  Single round                                                       #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        current_round: int,
        server_val_data: tf.data.Dataset,
    ) -> Dict:
        """
        Execute one complete federated round.

        1. Select clients  (Part 1).
        2. Distribute global weights & run local training.
        3. Validate updates & compute contribution weights (Part 2).
        4. Weighted aggregation.
        5. Update reputation ledger.

        Returns
        -------
        round_info : dict
            Summary including selected clients, records, and new accuracy.
        """
        # 1. Client selection
        selected = self.selector.select(current_round=current_round)
        global_weights = self.global_model.get_weights()

        # 2. Local training
        client_updates: Dict[str, List[np.ndarray]] = {}
        data_volumes: Dict[str, int] = {}
        for client in selected:
            updated_w, n_samples = self._local_train(client, global_weights)
            client_updates[client.client_id] = updated_w
            data_volumes[client.client_id] = max(n_samples, 1)
            client.metrics.last_selected_round = current_round

        # 3. Validate updates & contribution scoring
        records = self.validator.validate_updates(
            client_updates=client_updates,
            data_volumes=data_volumes,
            server_val_data=server_val_data,
        )

        # 4. Weighted aggregation
        new_weights = self.validator.aggregate_weighted(records, global_weights)
        self.global_model.set_weights(new_weights)

        # Also update the validator's reference model
        self.validator.global_model.set_weights(new_weights)

        # 5. Reputation feedback
        self.validator.update_reputations(records)

        # Evaluate new global model
        result = self.global_model.evaluate(
            server_val_data.batch(self.local_batch_size),
            verbose=0,
            return_dict=True,
        )
        acc = result.get("accuracy", 0.0)
        logger.info("Round %d — post-aggregation accuracy: %.4f", current_round, acc)

        return {
            "round": current_round,
            "selected": [c.client_id for c in selected],
            "records": records,
            "global_accuracy": acc,
        }

    # ------------------------------------------------------------------ #
    #  Multi-round loop                                                   #
    # ------------------------------------------------------------------ #

    def run(
        self,
        num_rounds: int,
        server_val_data: tf.data.Dataset,
    ) -> Dict[str, list]:
        """
        Run *num_rounds* federated rounds end-to-end.

        Returns a history dict similar to ``FederatedRoundRunner.run()``.
        """
        history: Dict[str, list] = {
            "round": [],
            "global_accuracy": [],
            "selected_clients": [],
            "contribution_weights": [],
        }

        for t in range(1, num_rounds + 1):
            logger.info("=" * 60)
            logger.info("ROUND %d / %d", t, num_rounds)
            logger.info("=" * 60)

            info = self.execute_round(t, server_val_data)
            history["round"].append(t)
            history["global_accuracy"].append(info["global_accuracy"])
            history["selected_clients"].append(info["selected"])
            history["contribution_weights"].append(
                {r.client_id: r.contribution_weight for r in info["records"]}
            )

        logger.info("Federated training complete — %d rounds.", num_rounds)
        return history


# ====================================================================== #
#  DEMO / SMOKE-TEST  (no model or data — synthetic only)                 #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Update Validation & Contribution Weighing — Demo  ===\n")

    # ---- synthetic setup --------------------------------------------- #
    np.random.seed(42)

    NUM_CLIENTS = 8
    NUM_LAYERS = 3                 # pretend model has 3 weight tensors
    LAYER_SHAPE = (4, 4)          # small for demo

    # Fake "global weights"
    global_weights = [np.random.randn(*LAYER_SHAPE).astype(np.float32)
                      for _ in range(NUM_LAYERS)]

    # Build client updates (simulate deltas of varying quality)
    client_updates: Dict[str, List[np.ndarray]] = {}
    data_volumes: Dict[str, int] = {}

    for i in range(NUM_CLIENTS):
        cid = f"client_{i:02d}"
        # Varying quality: some improve, some degrade, one is huge (suspicious)
        if i == 5:
            # suspiciously large update
            noise_scale = 50.0
        elif i == 7:
            # harmful update (large negative impact simulated later)
            noise_scale = 2.0
        else:
            noise_scale = 0.3

        delta = [np.random.randn(*LAYER_SHAPE).astype(np.float32) * noise_scale
                 for _ in range(NUM_LAYERS)]
        client_updates[cid] = [g + d for g, d in zip(global_weights, delta)]
        data_volumes[cid] = int(np.random.randint(200, 5000))

    # ---- Reputation ledger ------------------------------------------- #
    ledger = ReputationLedger()
    for cid in client_updates:
        ledger.register(cid)

    # ---- Build a minimal "model" for the demo (no real inference) ---- #
    # We can't call _evaluate without a real Keras model, so we'll
    # exercise everything *except* the TF evaluation path by subclassing.

    class _DemoValidator(UpdateValidator):
        """Override _evaluate to return a synthetic score for the demo."""

        def _evaluate(self, model_weights, val_data):
            # Score = negative of mean absolute weight magnitude
            # (smaller weights ⇒ higher "score", just for illustration)
            flat = flatten_weights(model_weights)
            return float(1.0 / (1.0 + np.mean(np.abs(flat))))

    validator = _DemoValidator(
        global_model=None,      # not used in our overridden _evaluate
        reputation_ledger=ledger,
        weights=ContributionWeights(alpha=0.40, beta=0.15, gamma=0.20, delta=0.25),
        clipping=ClippingConfig(clip_threshold=10.0, clip_value=5.0),
        harmful_threshold=0.02,
    )

    # Monkey-patch get_weights so aggregate_weighted works without a real model
    class _FakeModel:
        def get_weights(self):
            return global_weights
    validator.global_model = _FakeModel()

    # ---- Create a fake tf.data.Dataset (unused by our mock) ---------- #
    fake_val = tf.data.Dataset.from_tensor_slices(
        (np.zeros((2, 4)), np.zeros((2,)))
    )

    # ---- Run validation pipeline ------------------------------------- #
    records = validator.validate_updates(
        client_updates=client_updates,
        data_volumes=data_volumes,
        server_val_data=fake_val,
    )

    print(f"{'Client':<14} {'Norm':>8} {'G_i':>8} {'Sim':>8} "
          f"{'c_i':>8} {'Status':<12}")
    print("-" * 62)
    for rec in records:
        status = "REJECTED" if rec.rejected else (
            "SUSPICIOUS" if rec.is_suspicious else "ok"
        )
        print(f"{rec.client_id:<14} {rec.norm:>8.3f} {rec.validation_gain:>8.4f} "
              f"{rec.similarity:>8.4f} {rec.contribution_weight:>8.4f} {status:<12}")

    # ---- Weighted aggregation ---------------------------------------- #
    new_weights = validator.aggregate_weighted(records, global_weights)
    print(f"\nAggregated delta norm: "
          f"{np.linalg.norm(flatten_weights(compute_update_delta(global_weights, new_weights))):.4f}")

    # ---- Reputation update ------------------------------------------- #
    validator.update_reputations(records)
    print("\n--- Reputation Ledger After Round ---")
    for cid, rep in sorted(ledger.summary().items()):
        print(f"  {cid}: {rep:.4f}")


Overwriting update_validation.py


### Part 3 — Server-Side Knowledge Distillation


In [9]:
%%writefile knowledge_distillation.py
"""
Server-Side Knowledge Distillation
===================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

After federated aggregation (Parts 1 & 2), the server refines the global
model by distilling knowledge from the contribution-weighted ensemble of
client models into the global student model, using unlabelled proxy data
from FF++ c23.

Pipeline
--------
1.  **Build teacher logits** — weighted average of per-client logits on
    every proxy input, where the weights are the contribution scores
    ``{c_i}`` from Part 2.
2.  **Distillation loop** — minimise ``λ · T² · KL(p_teach ‖ p_stud)``
    (soft targets) plus an optional ``(1−λ) · CE`` supervised term when
    labelled data is available.
3.  Return the distilled global model.

Imports shared types from ``enhanced_client_selection.py`` and helpers
from ``update_validation.py``.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- shared types from Parts 1 & 2 ----------------------------- #
from enhanced_client_selection import FederatedClient, ReputationLedger

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class DistillationConfig:
    """
    Hyper-parameters for server-side knowledge distillation.

    Parameters
    ----------
    temperature : float
        Softmax temperature ``T`` — higher values produce softer
        probability distributions, transferring more "dark knowledge".
    lam : float
        Interpolation weight ``λ`` between the distillation loss and the
        optional supervised cross-entropy loss:
        ``L_total = λ · L_KD  +  (1 − λ) · L_sup``
    epochs : int
        Number of distillation training epochs.
    batch_size : int
        Batch size for iterating over proxy / supervised data.
    learning_rate : float
        Learning rate for the distillation optimiser.
    """
    temperature: float = 3.0
    lam: float = 0.7
    epochs: int = 5
    batch_size: int = 32
    learning_rate: float = 1e-4


# ====================================================================== #
#  2.  TEACHER LOGIT BUILDER                                              #
# ====================================================================== #

class TeacherEnsemble:
    """
    Builds a *virtual* teacher by computing the contribution-weighted
    average of per-client logits for every proxy input.

    The teacher is never materialised as a single model; instead its
    output logits are pre-computed and cached so the distillation loop
    can iterate over them efficiently.

    Parameters
    ----------
    global_model : tf.keras.Model
        Used as the architecture template for loading client weights.
    client_weights : dict[str, list[np.ndarray]]
        ``{client_id: model_weights}`` for every participating client.
    contribution_weights : dict[str, float]
        ``{client_id: c_i}`` from Part 2 — only clients with ``c_i > 0``
        are included.
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        client_weights: Dict[str, List[np.ndarray]],
        contribution_weights: Dict[str, float],
    ) -> None:
        self.global_model = global_model
        # Filter to clients that actually contribute
        self.client_weights = {
            cid: w for cid, w in client_weights.items()
            if contribution_weights.get(cid, 0.0) > 0
        }
        self.contribution_weights = {
            cid: contribution_weights[cid]
            for cid in self.client_weights
        }
        total_c = sum(self.contribution_weights.values())
        # Normalise so weights sum to 1
        self._norm_weights = {
            cid: c / total_c for cid, c in self.contribution_weights.items()
        }
        logger.info(
            "TeacherEnsemble: %d client(s), normalised weights: %s",
            len(self._norm_weights),
            {k: round(v, 4) for k, v in self._norm_weights.items()},
        )

    # ------------------------------------------------------------------ #
    #  Logit-model builder (creates a model that outputs pre-softmax)     #
    # ------------------------------------------------------------------ #

    def _build_logit_model(
        self,
        weights: List[np.ndarray],
    ) -> tf.keras.Model:
        """
        Clone the global model, load *weights*, and strip the final
        activation so the output is raw **logits**.

        Strategy: rebuild the architecture layer-by-layer. For the last
        Dense layer, replace its activation with ``linear``.  This avoids
        fragile graph surgery and works across Sequential / Functional
        models in all TF 2.x versions.
        """
        # --- Rebuild architecture with linear final activation --------- #
        logit_model = self._rebuild_with_linear_output(self.global_model)
        logit_model.set_weights(weights)
        return logit_model

    @staticmethod
    def _rebuild_with_linear_output(
        ref_model: tf.keras.Model,
    ) -> tf.keras.Model:
        """
        Rebuild *ref_model* identically, except the last ``Dense`` layer
        uses ``activation='linear'`` so the output is raw logits.

        Falls back to the original architecture when no Dense layer is
        found (e.g. the model already produces logits).
        """
        # Walk layers to find the last Dense
        last_dense_idx: Optional[int] = None
        for idx, layer in enumerate(ref_model.layers):
            if isinstance(layer, tf.keras.layers.Dense):
                last_dense_idx = idx

        # Identify which layers are *not* InputLayer
        non_input_layers = [
            (idx, layer) for idx, layer in enumerate(ref_model.layers)
            if not isinstance(layer, tf.keras.layers.InputLayer)
        ]

        # Build input
        input_shape = ref_model.input_shape
        if isinstance(input_shape, tuple):
            # Strip the batch dimension
            input_shape = input_shape[1:]
        x = inp = tf.keras.layers.Input(shape=input_shape)

        for idx, layer in non_input_layers:
            if idx == last_dense_idx:
                # Replace activation with linear
                cfg = layer.get_config()
                cfg["name"] = cfg["name"] + "_logits"
                cfg["activation"] = "linear"
                new_layer = tf.keras.layers.Dense.from_config(cfg)
                x = new_layer(x)
            else:
                # Re-use the same layer class & config
                cloned = layer.__class__.from_config(layer.get_config())
                x = cloned(x)

        logit_model = tf.keras.Model(inputs=inp, outputs=x)

        # Transfer weights layer by layer (names may differ for the
        # modified Dense, so match by position among non-input layers).
        ref_non_input = [l for l in ref_model.layers
                         if not isinstance(l, tf.keras.layers.InputLayer)]
        logit_non_input = [l for l in logit_model.layers
                           if not isinstance(l, tf.keras.layers.InputLayer)]
        for src, dst in zip(ref_non_input, logit_non_input):
            ws = src.get_weights()
            if ws:
                dst.set_weights(ws)

        return logit_model

    # ------------------------------------------------------------------ #
    #  Compute teacher logits for a batch                                 #
    # ------------------------------------------------------------------ #

    def compute_teacher_logits_batch(
        self,
        x_batch: tf.Tensor,
    ) -> tf.Tensor:
        """
        Return the contribution-weighted average teacher logits for
        *x_batch*.

        ``z_teach(x) = Σ_i  w_i · logits(M_i, x)``
        """
        weighted_logits = None
        for cid, weights in self.client_weights.items():
            logit_model = self._build_logit_model(weights)
            logits = logit_model(x_batch, training=False)     # (B, num_classes) or (B, 1)
            scaled = tf.cast(logits, tf.float32) * self._norm_weights[cid]
            if weighted_logits is None:
                weighted_logits = scaled
            else:
                weighted_logits = weighted_logits + scaled
        return weighted_logits                                 # type: ignore[return-value]

    # ------------------------------------------------------------------ #
    #  Pre-compute teacher logits for entire proxy dataset                #
    # ------------------------------------------------------------------ #

    def precompute_teacher_logits(
        self,
        proxy_data: tf.data.Dataset,
        batch_size: int = 32,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Pre-compute teacher logits for every sample in *proxy_data*.

        Parameters
        ----------
        proxy_data : tf.data.Dataset
            Must yield ``(images,)`` or ``(images, _)`` — labels are
            ignored.
        batch_size : int

        Returns
        -------
        all_inputs : np.ndarray   — shape ``(N, ...)``
        all_logits : np.ndarray   — shape ``(N, num_classes)`` or ``(N, 1)``
        """
        all_inputs: List[np.ndarray] = []
        all_logits: List[np.ndarray] = []

        batched = proxy_data.batch(batch_size)
        for batch in batched:
            if isinstance(batch, (list, tuple)):
                x_batch = batch[0]
            else:
                x_batch = batch

            teacher_logits = self.compute_teacher_logits_batch(x_batch)
            all_inputs.append(x_batch.numpy())
            all_logits.append(teacher_logits.numpy())

        return np.concatenate(all_inputs), np.concatenate(all_logits)


# ====================================================================== #
#  3.  DISTILLATION LOSSES                                                #
# ====================================================================== #

def distillation_loss(
    teacher_logits: tf.Tensor,
    student_logits: tf.Tensor,
    temperature: float,
) -> tf.Tensor:
    """
    Compute the knowledge-distillation loss:

        ``L_KD = T² · KL( softmax(z_teach / T)  ‖  softmax(z_stud / T) )``

    Works for both multi-class (last dim > 1) and binary (last dim = 1).
    """
    # Determine if binary or multi-class
    num_outputs = teacher_logits.shape[-1]

    if num_outputs == 1:
        # Binary case: use sigmoid + binary KL
        p_teach = tf.sigmoid(teacher_logits / temperature)
        p_stud  = tf.sigmoid(student_logits / temperature)
        # Binary KL divergence  KL(p || q) = p·log(p/q) + (1-p)·log((1-p)/(1-q))
        eps = 1e-7
        p_teach = tf.clip_by_value(p_teach, eps, 1.0 - eps)
        p_stud  = tf.clip_by_value(p_stud,  eps, 1.0 - eps)
        kl = (
            p_teach * tf.math.log(p_teach / p_stud)
            + (1.0 - p_teach) * tf.math.log((1.0 - p_teach) / (1.0 - p_stud))
        )
        return temperature ** 2 * tf.reduce_mean(kl)
    else:
        # Multi-class case: standard KL on softmax outputs
        p_teach = tf.nn.softmax(teacher_logits / temperature)
        log_p_stud = tf.nn.log_softmax(student_logits / temperature)
        # KL(p_teach || p_stud) = Σ p_teach · (log p_teach − log p_stud)
        kl = tf.reduce_sum(
            p_teach * (tf.math.log(p_teach + 1e-12) - log_p_stud),
            axis=-1,
        )
        return temperature ** 2 * tf.reduce_mean(kl)


def supervised_loss(
    student_logits: tf.Tensor,
    labels: tf.Tensor,
) -> tf.Tensor:
    """
    Standard supervised cross-entropy loss.

    Handles binary (1-unit output, BCE) and multi-class (softmax CE)
    automatically.
    """
    num_outputs = student_logits.shape[-1]
    if num_outputs == 1:
        # Ensure labels have the same rank as logits: (B,) → (B, 1)
        labels = tf.cast(labels, tf.float32)
        if len(labels.shape) < len(student_logits.shape):
            labels = tf.expand_dims(labels, -1)
        return tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(
                labels, student_logits, from_logits=True,
            )
        )
    else:
        return tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels, student_logits, from_logits=True,
            )
        )


# ====================================================================== #
#  4.  KNOWLEDGE DISTILLATION ENGINE                                      #
# ====================================================================== #

class KnowledgeDistiller:
    """
    Performs server-side knowledge distillation from a weighted ensemble
    of client models (teacher) into the aggregated global model (student).

    Parameters
    ----------
    global_model : tf.keras.Model
        The global/student model — **modified in-place**.
    teacher : TeacherEnsemble
        The virtual teacher (built from client logits).
    config : DistillationConfig
        Temperature, λ, epochs, batch size, learning rate.
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        teacher: TeacherEnsemble,
        config: Optional[DistillationConfig] = None,
    ) -> None:
        self.global_model = global_model
        self.teacher = teacher
        self.config = config or DistillationConfig()
        self.optimizer = tf.keras.optimizers.Adam(self.config.learning_rate)

    # ------------------------------------------------------------------ #
    #  Build a "logit model" view of the global student                   #
    # ------------------------------------------------------------------ #

    def _build_student_logit_model(self) -> tf.keras.Model:
        """
        Return a version of the global model that outputs raw logits.

        Uses the same stripping logic as ``TeacherEnsemble``.
        """
        return self.teacher._build_logit_model(self.global_model.get_weights())

    # ------------------------------------------------------------------ #
    #  Single training step                                               #
    # ------------------------------------------------------------------ #

    @tf.function
    def _train_step_kd_only(
        self,
        x_batch: tf.Tensor,
        teacher_logits: tf.Tensor,
        student_model: tf.keras.Model,
        temperature: float,
    ) -> tf.Tensor:
        """Pure distillation step (no supervised term)."""
        with tf.GradientTape() as tape:
            student_logits = student_model(x_batch, training=True)
            loss = distillation_loss(teacher_logits, student_logits, temperature)
        grads = tape.gradient(loss, student_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, student_model.trainable_variables)
        )
        return loss

    @tf.function
    def _train_step_combined(
        self,
        x_proxy: tf.Tensor,
        teacher_logits: tf.Tensor,
        x_sup: tf.Tensor,
        y_sup: tf.Tensor,
        student_model: tf.keras.Model,
        temperature: float,
        lam: float,
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """Combined distillation + supervised step."""
        with tf.GradientTape() as tape:
            # KD part
            stud_logits_proxy = student_model(x_proxy, training=True)
            l_kd = distillation_loss(teacher_logits, stud_logits_proxy, temperature)

            # Supervised part
            stud_logits_sup = student_model(x_sup, training=True)
            l_sup = supervised_loss(stud_logits_sup, y_sup)

            l_total = lam * l_kd + (1.0 - lam) * l_sup

        grads = tape.gradient(l_total, student_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, student_model.trainable_variables)
        )
        return l_total, l_kd, l_sup

    # ------------------------------------------------------------------ #
    #  Full distillation loop                                             #
    # ------------------------------------------------------------------ #

    def distill(
        self,
        proxy_data: tf.data.Dataset,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, List[float]]:
        """
        Run the full distillation loop.

        Parameters
        ----------
        proxy_data : tf.data.Dataset
            Unlabelled proxy inputs from FF++ c23.
            Yields ``(images,)`` or ``(images, _)`` — labels ignored.
        supervised_data : tf.data.Dataset | None
            Optional labelled data yielding ``(images, labels)``.
            When provided, the combined loss
            ``λ · L_KD + (1−λ) · L_sup`` is used.

        Returns
        -------
        history : dict
            ``{"epoch": [...], "loss_total": [...], "loss_kd": [...],
               "loss_sup": [...]}``
        """
        cfg = self.config
        T = cfg.temperature
        lam = cfg.lam

        history: Dict[str, List[float]] = {
            "epoch": [],
            "loss_total": [],
            "loss_kd": [],
            "loss_sup": [],
        }

        # --- Pre-compute teacher logits ------------------------------- #
        logger.info("Pre-computing teacher logits (T=%.1f) …", T)
        proxy_inputs, teacher_logits_all = self.teacher.precompute_teacher_logits(
            proxy_data, batch_size=cfg.batch_size,
        )
        logger.info(
            "Teacher logits ready — %d samples, shape %s",
            len(proxy_inputs), teacher_logits_all.shape,
        )

        # Wrap into a tf.data.Dataset for batching
        proxy_ds = (
            tf.data.Dataset.from_tensor_slices((proxy_inputs, teacher_logits_all))
            .shuffle(buffer_size=len(proxy_inputs))
            .batch(cfg.batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        # Prepare supervised data iterator (if available)
        if supervised_data is not None:
            sup_ds = (
                supervised_data
                .shuffle(buffer_size=1000)
                .batch(cfg.batch_size)
                .repeat()                       # cycle forever — we zip with proxy
                .prefetch(tf.data.AUTOTUNE)
            )
            sup_iter = iter(sup_ds)
        else:
            sup_iter = None

        # --- Build student logit model -------------------------------- #
        # We train a clone that shares the same architecture & initial
        # weights, then copy weights back to self.global_model at the end.
        student = self._build_student_logit_model()

        # --- Distillation epochs -------------------------------------- #
        for epoch in range(1, cfg.epochs + 1):
            epoch_loss_total = []
            epoch_loss_kd = []
            epoch_loss_sup = []

            for batch in proxy_ds:
                x_proxy_batch, teach_logits_batch = batch

                if sup_iter is not None:
                    # Combined loss
                    try:
                        x_sup_batch, y_sup_batch = next(sup_iter)
                    except StopIteration:
                        sup_iter = iter(sup_ds)
                        x_sup_batch, y_sup_batch = next(sup_iter)

                    l_total, l_kd, l_sup = self._train_step_combined(
                        x_proxy_batch, teach_logits_batch,
                        x_sup_batch, y_sup_batch,
                        student, T, lam,
                    )
                    epoch_loss_total.append(float(l_total))
                    epoch_loss_kd.append(float(l_kd))
                    epoch_loss_sup.append(float(l_sup))
                else:
                    # KD only
                    l_kd = self._train_step_kd_only(
                        x_proxy_batch, teach_logits_batch,
                        student, T,
                    )
                    epoch_loss_total.append(float(l_kd))
                    epoch_loss_kd.append(float(l_kd))
                    epoch_loss_sup.append(0.0)

            mean_total = float(np.mean(epoch_loss_total))
            mean_kd    = float(np.mean(epoch_loss_kd))
            mean_sup   = float(np.mean(epoch_loss_sup))

            history["epoch"].append(epoch)
            history["loss_total"].append(mean_total)
            history["loss_kd"].append(mean_kd)
            history["loss_sup"].append(mean_sup)

            logger.info(
                "Distillation epoch %d/%d — L_total=%.5f  L_KD=%.5f  L_sup=%.5f",
                epoch, cfg.epochs, mean_total, mean_kd, mean_sup,
            )

        # --- Copy distilled weights back to global model -------------- #
        self.global_model.set_weights(student.get_weights())
        logger.info("Distilled weights applied to global model.")

        return history


# ====================================================================== #
#  5.  CONVENIENCE: one-call distillation after a federated round         #
# ====================================================================== #

def run_distillation_round(
    global_model: tf.keras.Model,
    client_weights: Dict[str, List[np.ndarray]],
    contribution_weights: Dict[str, float],
    proxy_data: tf.data.Dataset,
    supervised_data: Optional[tf.data.Dataset] = None,
    config: Optional[DistillationConfig] = None,
) -> Dict[str, List[float]]:
    """
    One-liner helper that creates the teacher ensemble, distiller,
    and runs the distillation loop.

    Parameters
    ----------
    global_model : tf.keras.Model
        Aggregated global model (modified in-place).
    client_weights : dict[str, list[np.ndarray]]
        Per-client model weights.
    contribution_weights : dict[str, float]
        Per-client contribution scores ``c_i`` from Part 2.
    proxy_data : tf.data.Dataset
        Unlabelled proxy inputs (FF++ c23).
    supervised_data : tf.data.Dataset | None
        Optional labelled data for the combined loss.
    config : DistillationConfig | None
        Hyper-parameters (defaults are sensible).

    Returns
    -------
    history : dict
    """
    config = config or DistillationConfig()

    teacher = TeacherEnsemble(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
    )
    distiller = KnowledgeDistiller(
        global_model=global_model,
        teacher=teacher,
        config=config,
    )
    return distiller.distill(proxy_data, supervised_data)


# ====================================================================== #
#  DEMO / SMOKE-TEST  (synthetic data — no real model needed)             #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Server-Side Knowledge Distillation — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model to act as global / student ------------ #
    INPUT_DIM = 16
    NUM_CLASSES = 1          # binary (DeepFake vs Real)
    NUM_PROXY = 200
    NUM_SUP = 60
    NUM_CLIENTS = 4

    global_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="sigmoid"),
    ])
    global_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Global model params: {global_model.count_params()}")

    # ---- 2. Synthetic client models (slight perturbations) ----------- #
    base_weights = global_model.get_weights()
    client_weights: Dict[str, List[np.ndarray]] = {}
    contribution_weights: Dict[str, float] = {}

    for i in range(NUM_CLIENTS):
        cid = f"client_{i:02d}"
        noise = [np.random.randn(*w.shape).astype(np.float32) * 0.05
                 for w in base_weights]
        client_weights[cid] = [w + n for w, n in zip(base_weights, noise)]
        contribution_weights[cid] = np.random.uniform(0.3, 1.0)

    print(f"Contribution weights: {contribution_weights}")

    # ---- 3. Synthetic proxy & supervised data ------------------------ #
    proxy_x = np.random.randn(NUM_PROXY, INPUT_DIM).astype(np.float32)
    proxy_data = tf.data.Dataset.from_tensor_slices(proxy_x)

    sup_x = np.random.randn(NUM_SUP, INPUT_DIM).astype(np.float32)
    sup_y = np.random.randint(0, 2, size=(NUM_SUP,)).astype(np.float32)
    supervised_data = tf.data.Dataset.from_tensor_slices((sup_x, sup_y))

    # ---- 4. Run distillation (KD only) ------------------------------- #
    print("\n--- KD-only distillation ---")
    history_kd = run_distillation_round(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
        proxy_data=proxy_data,
        supervised_data=None,              # no supervised term
        config=DistillationConfig(
            temperature=3.0,
            lam=0.7,
            epochs=3,
            batch_size=32,
            learning_rate=1e-3,
        ),
    )
    for ep, lt, lk in zip(history_kd["epoch"], history_kd["loss_total"],
                          history_kd["loss_kd"]):
        print(f"  Epoch {ep}: L_total={lt:.5f}  L_KD={lk:.5f}")

    # ---- 5. Run distillation (combined KD + supervised) -------------- #
    print("\n--- Combined KD + supervised distillation ---")
    # Reset model to base weights for a fair comparison
    global_model.set_weights(base_weights)

    history_comb = run_distillation_round(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
        proxy_data=proxy_data,
        supervised_data=supervised_data,
        config=DistillationConfig(
            temperature=3.0,
            lam=0.7,
            epochs=3,
            batch_size=32,
            learning_rate=1e-3,
        ),
    )
    for ep, lt, lk, ls in zip(
        history_comb["epoch"], history_comb["loss_total"],
        history_comb["loss_kd"], history_comb["loss_sup"],
    ):
        print(f"  Epoch {ep}: L_total={lt:.5f}  L_KD={lk:.5f}  L_sup={ls:.5f}")

    print("\nDone.")


Writing knowledge_distillation.py


### Part 4 — Client Reputation & Persistent Ledger


In [10]:
%%writefile client_reputation_ledger.py
"""
Client Reputation and Client Ledger
====================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Extends the basic ``ReputationLedger`` from Part 1 with:

* **Validation-gain-based reputation update** — reputation increases
  proportionally to the measured validation gain ``G_i`` when it exceeds
  a threshold ``θ``, and holds steady (or decays) otherwise.
* **Persistent ledger** with full round-by-round history, JSON
  serialisation, and audit trail.
* **Decay & floor mechanics** — inactive or consistently harmful clients
  slowly lose reputation; a configurable floor prevents permanent
  exclusion.
* **Integration helpers** that plug directly into Parts 1–3.

Pseudo-code from the thesis proposal is faithfully reproduced.
"""

from __future__ import annotations

import json
import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

# ---------- shared types from Part 1 ---------------------------------- #
from enhanced_client_selection import ReputationLedger

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class ReputationConfig:
    """
    Hyper-parameters for the gain-based reputation update rule.

    Parameters
    ----------
    theta : float
        Validation-gain threshold ``θ``.  An update is considered *valid*
        only when ``G_i > θ``.
    gamma : float
        Reputation update factor ``γ``.  Controls how strongly a positive
        gain boosts reputation.
    decay_rate : float
        Per-round multiplicative decay applied to clients that did **not**
        participate (models staleness into reputation).  Set to ``1.0`` to
        disable decay.
    floor : float
        Minimum reputation — prevents permanent exclusion so every client
        retains a chance to be re-selected.
    ceiling : float
        Maximum reputation (typically ``1.0``).
    initial_reputation : float
        Reputation assigned to newly registered clients.
    penalty_factor : float
        Fraction of ``|G_i|`` subtracted from reputation when the update
        is actively harmful (``G_i < -θ``).  Set to ``0.0`` for the
        "no-change-on-invalid" variant from the thesis pseudo-code.
    """
    theta: float = 0.0               # validation-gain threshold
    gamma: float = 0.10              # reputation update factor
    decay_rate: float = 0.995        # per-round decay for non-participants
    floor: float = 0.05              # minimum reputation
    ceiling: float = 1.0             # maximum reputation
    initial_reputation: float = 0.5  # starting score
    penalty_factor: float = 0.0      # penalty multiplier for harmful updates


# ====================================================================== #
#  2.  LEDGER ENTRY — per-client record                                   #
# ====================================================================== #

@dataclass
class ClientLedgerEntry:
    """
    Full reputation record for a single client, including audit history.
    """
    client_id: str
    reputation: float = 0.5
    total_rounds_participated: int = 0
    total_valid_updates: int = 0
    total_invalid_updates: int = 0
    cumulative_gain: float = 0.0
    last_participated_round: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

    # ---- serialisation ----------------------------------------------- #

    def to_dict(self) -> Dict[str, Any]:
        return {
            "client_id": self.client_id,
            "reputation": self.reputation,
            "total_rounds_participated": self.total_rounds_participated,
            "total_valid_updates": self.total_valid_updates,
            "total_invalid_updates": self.total_invalid_updates,
            "cumulative_gain": self.cumulative_gain,
            "last_participated_round": self.last_participated_round,
            "history": self.history,
        }

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "ClientLedgerEntry":
        return cls(**d)


# ====================================================================== #
#  3.  CLIENT REPUTATION LEDGER (extended)                                #
# ====================================================================== #

class ClientReputationLedger:
    """
    Persistent, auditable reputation ledger for all federated clients.

    Implements the thesis pseudo-code:

    .. code-block:: text

        For each client i:
            G_i = validation_gain(u_i)
            if G_i > θ:  is_valid, delta_i = True,  G_i
            else:         is_valid, delta_i = False, 0
            if is_valid:  R_i = min(1, R_i + γ · delta_i)
            else:         R_i = R_i          # no change
            store(i, R_i)

    Additionally supports:

    * Optional **penalty** for harmful updates (``G_i < -θ``).
    * **Decay** for non-participating clients each round.
    * Full **history** / audit trail per client.
    * **JSON persistence** (save / load).
    * Backward-compatible bridge to the basic ``ReputationLedger`` from
      Part 1 via :meth:`as_basic_ledger`.

    Parameters
    ----------
    config : ReputationConfig
        All tuneable knobs.
    """

    def __init__(self, config: Optional[ReputationConfig] = None) -> None:
        self.config = config or ReputationConfig()
        self._entries: Dict[str, ClientLedgerEntry] = {}

    # ------------------------------------------------------------------ #
    #  Registration                                                       #
    # ------------------------------------------------------------------ #

    def register(self, client_id: str) -> None:
        """Register a new client with the default initial reputation."""
        if client_id not in self._entries:
            self._entries[client_id] = ClientLedgerEntry(
                client_id=client_id,
                reputation=self.config.initial_reputation,
            )
            logger.debug("Registered client %s (R=%.4f).",
                         client_id, self.config.initial_reputation)

    def register_many(self, client_ids: List[str]) -> None:
        for cid in client_ids:
            self.register(cid)

    # ------------------------------------------------------------------ #
    #  Read                                                                #
    # ------------------------------------------------------------------ #

    def get(self, client_id: str) -> float:
        """Return the current reputation for *client_id*."""
        entry = self._entries.get(client_id)
        if entry is None:
            return self.config.initial_reputation
        return entry.reputation

    def get_entry(self, client_id: str) -> Optional[ClientLedgerEntry]:
        """Return the full ledger entry (or ``None``)."""
        return self._entries.get(client_id)

    def all_reputations(self) -> Dict[str, float]:
        """Return ``{client_id: reputation}`` for every registered client."""
        return {cid: e.reputation for cid, e in self._entries.items()}

    # ------------------------------------------------------------------ #
    #  Core update — implements the thesis pseudo-code                    #
    # ------------------------------------------------------------------ #

    def update_reputation(
        self,
        client_id: str,
        validation_gain: float,
        current_round: int,
        contribution_weight: Optional[float] = None,
    ) -> float:
        """
        Update a single client's reputation based on validation gain.

        Follows the pseudo-code:

        .. code-block:: text

            if G_i > θ:  R_i = min(ceiling, R_i + γ · G_i)
            else:        R_i = R_i                       # no change
            (optional)   if G_i < -θ:  R_i = max(floor, R_i - penalty · |G_i|)

        Parameters
        ----------
        client_id : str
        validation_gain : float
            ``G_i`` — the measured improvement on the server validation set.
        current_round : int
            Current federated round number (for the audit trail).
        contribution_weight : float | None
            ``c_i`` from Part 2 (stored in the audit trail, not used in
            the reputation formula itself).

        Returns
        -------
        new_reputation : float
        """
        self.register(client_id)                 # idempotent
        entry = self._entries[client_id]
        cfg = self.config
        old_rep = entry.reputation

        # --- 1. Determine validity ------------------------------------ #
        is_valid = validation_gain > cfg.theta
        delta = validation_gain if is_valid else 0.0

        # --- 2. Compute new reputation -------------------------------- #
        if is_valid:
            new_rep = min(cfg.ceiling, old_rep + cfg.gamma * delta)
        else:
            new_rep = old_rep                    # no change on invalid

        # --- 2b. Optional penalty for actively harmful updates -------- #
        if cfg.penalty_factor > 0 and validation_gain < -cfg.theta:
            penalty = cfg.penalty_factor * abs(validation_gain)
            new_rep = max(cfg.floor, new_rep - penalty)

        # --- 3. Clamp to [floor, ceiling] ----------------------------- #
        new_rep = float(np.clip(new_rep, cfg.floor, cfg.ceiling))

        # --- 4. Store ------------------------------------------------- #
        entry.reputation = new_rep
        entry.total_rounds_participated += 1
        entry.last_participated_round = current_round
        entry.cumulative_gain += validation_gain
        if is_valid:
            entry.total_valid_updates += 1
        else:
            entry.total_invalid_updates += 1

        # Audit trail
        entry.history.append({
            "round": current_round,
            "G_i": round(validation_gain, 6),
            "is_valid": is_valid,
            "delta": round(delta, 6),
            "R_old": round(old_rep, 6),
            "R_new": round(new_rep, 6),
            "c_i": round(contribution_weight, 6) if contribution_weight is not None else None,
            "ts": time.time(),
        })

        logger.debug(
            "Client %s: G_i=%.4f, valid=%s, R %.4f → %.4f",
            client_id, validation_gain, is_valid, old_rep, new_rep,
        )
        return new_rep

    # ------------------------------------------------------------------ #
    #  Batch update — process all clients in a round                      #
    # ------------------------------------------------------------------ #

    def update_round(
        self,
        gains: Dict[str, float],
        current_round: int,
        contribution_weights: Optional[Dict[str, float]] = None,
        participant_ids: Optional[List[str]] = None,
    ) -> Dict[str, float]:
        """
        Update reputations for an entire federated round.

        Parameters
        ----------
        gains : dict[str, float]
            ``{client_id: G_i}`` for each participating client.
        current_round : int
        contribution_weights : dict[str, float] | None
            ``{client_id: c_i}`` from Part 2 (for the audit trail).
        participant_ids : list[str] | None
            Explicit list of participants.  If ``None``, derived from
            ``gains.keys()``.

        Returns
        -------
        updated : dict[str, float]
            ``{client_id: new_reputation}`` for every registered client
            (including decayed non-participants).
        """
        participants = set(participant_ids or gains.keys())
        cw = contribution_weights or {}

        # Update participants
        for cid in participants:
            g = gains.get(cid, 0.0)
            c = cw.get(cid)
            self.update_reputation(cid, g, current_round, c)

        # Decay non-participants
        self._decay_non_participants(participants, current_round)

        return self.all_reputations()

    # ------------------------------------------------------------------ #
    #  Decay for idle clients                                             #
    # ------------------------------------------------------------------ #

    def _decay_non_participants(
        self,
        participants: set,
        current_round: int,
    ) -> None:
        """
        Apply multiplicative decay to clients that did **not** participate
        this round.  Keeps reputation ≥ floor.
        """
        cfg = self.config
        if cfg.decay_rate >= 1.0:
            return                               # decay disabled

        for cid, entry in self._entries.items():
            if cid in participants:
                continue
            old = entry.reputation
            new = max(cfg.floor, old * cfg.decay_rate)
            if abs(new - old) > 1e-9:
                entry.reputation = new
                entry.history.append({
                    "round": current_round,
                    "event": "decay",
                    "R_old": round(old, 6),
                    "R_new": round(new, 6),
                    "ts": time.time(),
                })
                logger.debug(
                    "Client %s decayed: R %.4f → %.4f (non-participant).",
                    cid, old, new,
                )

    # ------------------------------------------------------------------ #
    #  Querying & analytics                                               #
    # ------------------------------------------------------------------ #

    def ranked(self, descending: bool = True) -> List[Tuple[str, float]]:
        """Return ``(client_id, reputation)`` sorted by reputation."""
        items = list(self.all_reputations().items())
        items.sort(key=lambda x: x[1], reverse=descending)
        return items

    def statistics(self) -> Dict[str, Any]:
        """Aggregate statistics across all registered clients."""
        reps = [e.reputation for e in self._entries.values()]
        if not reps:
            return {}
        arr = np.array(reps)
        return {
            "num_clients": len(reps),
            "mean_reputation": float(arr.mean()),
            "std_reputation": float(arr.std()),
            "min_reputation": float(arr.min()),
            "max_reputation": float(arr.max()),
            "median_reputation": float(np.median(arr)),
        }

    def client_summary(self, client_id: str) -> Optional[Dict[str, Any]]:
        """Return a human-readable summary dict for one client."""
        entry = self._entries.get(client_id)
        if entry is None:
            return None
        return {
            "client_id": entry.client_id,
            "reputation": entry.reputation,
            "rounds_participated": entry.total_rounds_participated,
            "valid_updates": entry.total_valid_updates,
            "invalid_updates": entry.total_invalid_updates,
            "cumulative_gain": entry.cumulative_gain,
            "last_participated_round": entry.last_participated_round,
            "history_length": len(entry.history),
        }

    # ------------------------------------------------------------------ #
    #  Bridge to Part 1 basic ReputationLedger                            #
    # ------------------------------------------------------------------ #

    def as_basic_ledger(self) -> ReputationLedger:
        """
        Export current reputations into a new ``ReputationLedger``
        (Part 1 format) so the client selector can consume them directly.
        """
        basic = ReputationLedger(
            default_reputation=self.config.initial_reputation,
        )
        for cid, entry in self._entries.items():
            basic.register(cid)
            # Overwrite internal score to match our ledger
            basic._scores[cid] = entry.reputation
        return basic

    def sync_from_basic_ledger(self, basic: ReputationLedger) -> None:
        """
        Import scores from an existing ``ReputationLedger`` (Part 1)
        into this extended ledger — useful when migrating mid-experiment.
        """
        for cid, score in basic.summary().items():
            self.register(cid)
            self._entries[cid].reputation = score

    # ------------------------------------------------------------------ #
    #  Persistence  (JSON)                                                #
    # ------------------------------------------------------------------ #

    def save(self, path: str | Path) -> None:
        """Serialise the full ledger (including history) to a JSON file."""
        data = {
            "config": {
                "theta": self.config.theta,
                "gamma": self.config.gamma,
                "decay_rate": self.config.decay_rate,
                "floor": self.config.floor,
                "ceiling": self.config.ceiling,
                "initial_reputation": self.config.initial_reputation,
                "penalty_factor": self.config.penalty_factor,
            },
            "entries": {cid: e.to_dict() for cid, e in self._entries.items()},
        }
        Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")
        logger.info("Ledger saved to %s (%d clients).", path, len(self._entries))

    @classmethod
    def load(cls, path: str | Path) -> "ClientReputationLedger":
        """Load a ledger from a previously saved JSON file."""
        raw = json.loads(Path(path).read_text(encoding="utf-8"))
        cfg = ReputationConfig(**raw["config"])
        ledger = cls(config=cfg)
        for cid, edict in raw["entries"].items():
            ledger._entries[cid] = ClientLedgerEntry.from_dict(edict)
        logger.info("Ledger loaded from %s (%d clients).", path, len(ledger._entries))
        return ledger


# ====================================================================== #
#  4.  INTEGRATION HELPER — plug into Parts 2 & 3                        #
# ====================================================================== #

def update_ledger_from_records(
    ledger: ClientReputationLedger,
    records,                        # List[ClientUpdateRecord] from Part 2
    current_round: int,
) -> Dict[str, float]:
    """
    Convenience function: extract ``G_i`` and ``c_i`` from Part 2's
    ``ClientUpdateRecord`` list and feed them into the extended ledger.

    Parameters
    ----------
    ledger : ClientReputationLedger
    records : list[ClientUpdateRecord]
        Output of ``UpdateValidator.validate_updates()``.
    current_round : int

    Returns
    -------
    updated : dict[str, float]
    """
    gains: Dict[str, float] = {}
    cws: Dict[str, float] = {}
    for rec in records:
        gains[rec.client_id] = rec.validation_gain
        cws[rec.client_id] = rec.contribution_weight
    return ledger.update_round(gains, current_round, contribution_weights=cws)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Client Reputation & Ledger — Demo  ===\n")

    np.random.seed(42)

    NUM_CLIENTS = 8
    NUM_ROUNDS = 10

    # ---- Configuration ----------------------------------------------- #
    cfg = ReputationConfig(
        theta=0.0,                    # any positive gain counts
        gamma=0.10,                   # reputation update factor
        decay_rate=0.99,              # 1 % decay per idle round
        floor=0.05,                   # minimum reputation
        ceiling=1.0,
        initial_reputation=0.50,
        penalty_factor=0.05,          # small penalty for harmful updates
    )

    ledger = ClientReputationLedger(config=cfg)
    client_ids = [f"client_{i:02d}" for i in range(NUM_CLIENTS)]
    ledger.register_many(client_ids)

    print(f"Config: θ={cfg.theta}, γ={cfg.gamma}, decay={cfg.decay_rate}, "
          f"floor={cfg.floor}, penalty={cfg.penalty_factor}")
    print(f"Registered {NUM_CLIENTS} clients, initial R={cfg.initial_reputation}\n")

    # ---- Simulate rounds --------------------------------------------- #
    for rnd in range(1, NUM_ROUNDS + 1):
        # Each round, randomly select ~60 % of clients
        num_selected = max(2, int(0.6 * NUM_CLIENTS))
        participants = list(np.random.choice(client_ids, size=num_selected, replace=False))

        # Simulate validation gains — mostly small positive, some negative
        gains: Dict[str, float] = {}
        for cid in participants:
            g = np.random.normal(loc=0.02, scale=0.04)   # mean 2 %, σ 4 %
            gains[cid] = float(g)

        # Simulate contribution weights (from Part 2)
        cws = {cid: float(np.random.uniform(0.3, 1.0)) for cid in participants}

        updated = ledger.update_round(
            gains=gains,
            current_round=rnd,
            contribution_weights=cws,
            participant_ids=participants,
        )

        # Print round summary
        print(f"--- Round {rnd} ---  participants: {participants}")
        print(f"  Gains:  { {k: f'{v:+.4f}' for k, v in gains.items()} }")
        ranked = ledger.ranked()
        print(f"  Reputations: { {k: f'{v:.4f}' for k, v in ranked} }\n")

    # ---- Final statistics -------------------------------------------- #
    print("=" * 50)
    print("Final Ledger Statistics:")
    stats = ledger.statistics()
    for k, v in stats.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    print("\nPer-client summaries:")
    for cid in client_ids:
        s = ledger.client_summary(cid)
        print(f"  {cid}: R={s['reputation']:.4f}, "
              f"participated={s['rounds_participated']}, "
              f"valid={s['valid_updates']}, invalid={s['invalid_updates']}, "
              f"cumG={s['cumulative_gain']:+.4f}")

    # ---- Persistence round-trip -------------------------------------- #
    save_path = "demo_ledger.json"
    ledger.save(save_path)
    loaded = ClientReputationLedger.load(save_path)
    assert loaded.all_reputations() == ledger.all_reputations(), "Round-trip mismatch!"
    print(f"\nJSON round-trip OK ({save_path})")

    # ---- Bridge to Part 1 -------------------------------------------- #
    basic = ledger.as_basic_ledger()
    print(f"\nBasic ReputationLedger bridge: {basic.summary()}")

    # Clean up demo file
    Path(save_path).unlink(missing_ok=True)

    print("\nDone.")


Writing client_reputation_ledger.py


### Part 5 — Evaluation Metrics & Report Generation


In [11]:
%%writefile evaluation_metrics.py
"""
Evaluation Metrics & Report Generation
=======================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Evaluates the global model after the federated learning cycle on:

* **Accuracy**
* **F1 Score** (macro, weighted, and per-class)
* **ROC-AUC**
* **Inference Latency** (mean, std, p95 over repeated batches)
* **Model Size** (parameter count + on-disk file size)

Generates structured reports (JSON + human-readable text) into a
dedicated ``reports/`` folder to keep the workspace organised.
"""

from __future__ import annotations

import json
import logging
import os
import tempfile
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Default report output directory (sibling of this file)
# ---------------------------------------------------------------------------
_MODULE_DIR = Path(__file__).resolve().parent
DEFAULT_REPORTS_DIR = _MODULE_DIR / "reports"


# ====================================================================== #
#  1.  DATA STRUCTURES                                                    #
# ====================================================================== #

@dataclass
class ClassificationMetrics:
    """Container for all classification-related evaluation metrics."""
    accuracy: float = 0.0
    f1_macro: float = 0.0
    f1_weighted: float = 0.0
    f1_per_class: Dict[str, float] = field(default_factory=dict)
    precision_macro: float = 0.0
    recall_macro: float = 0.0
    roc_auc: float = 0.0
    confusion_matrix: Optional[List[List[int]]] = None
    num_samples: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class LatencyMetrics:
    """Container for inference-latency measurements."""
    mean_ms: float = 0.0
    std_ms: float = 0.0
    min_ms: float = 0.0
    max_ms: float = 0.0
    p50_ms: float = 0.0
    p95_ms: float = 0.0
    p99_ms: float = 0.0
    num_batches: int = 0
    batch_size: int = 0
    total_samples: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ModelSizeMetrics:
    """Container for model-size information."""
    total_params: int = 0
    trainable_params: int = 0
    non_trainable_params: int = 0
    file_size_bytes: int = 0
    file_size_mb: float = 0.0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class EvaluationReport:
    """Full evaluation report aggregating all metric categories."""
    model_name: str = ""
    timestamp: str = ""
    federated_round: Optional[int] = None
    classification: ClassificationMetrics = field(default_factory=ClassificationMetrics)
    latency: LatencyMetrics = field(default_factory=LatencyMetrics)
    model_size: ModelSizeMetrics = field(default_factory=ModelSizeMetrics)
    extra: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "timestamp": self.timestamp,
            "federated_round": self.federated_round,
            "classification": self.classification.to_dict(),
            "latency": self.latency.to_dict(),
            "model_size": self.model_size.to_dict(),
            "extra": self.extra,
        }


# ====================================================================== #
#  2.  METRIC COMPUTATIONS                                                #
# ====================================================================== #

def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred_probs: np.ndarray,
    threshold: float = 0.5,
    class_names: Optional[List[str]] = None,
) -> ClassificationMetrics:
    """
    Compute accuracy, F1 (macro/weighted/per-class), precision, recall,
    ROC-AUC, and confusion matrix from ground-truth labels and predicted
    probabilities.

    Parameters
    ----------
    y_true : np.ndarray, shape ``(N,)``
        Ground-truth binary labels (0 or 1).
    y_pred_probs : np.ndarray, shape ``(N,)`` or ``(N, 1)``
        Predicted probabilities for the positive class.
    threshold : float
        Decision threshold for converting probabilities to hard labels.
    class_names : list[str] | None
        Human-readable names for class 0 and class 1.
        Defaults to ``["Real", "Fake"]``.

    Returns
    -------
    ClassificationMetrics
    """
    if class_names is None:
        class_names = ["Real", "Fake"]

    y_true = np.asarray(y_true).ravel().astype(int)
    y_probs = np.asarray(y_pred_probs).ravel().astype(float)
    y_pred = (y_probs >= threshold).astype(int)
    n = len(y_true)

    # --- Confusion matrix --------------------------------------------- #
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    cm = [[tn, fp], [fn, tp]]

    # --- Accuracy ----------------------------------------------------- #
    accuracy = (tp + tn) / max(n, 1)

    # --- Per-class precision / recall / F1 ---------------------------- #
    def _prf(tp_c: int, fp_c: int, fn_c: int):
        prec = tp_c / max(tp_c + fp_c, 1)
        rec = tp_c / max(tp_c + fn_c, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-12)
        return prec, rec, f1

    prec_0, rec_0, f1_0 = _prf(tn, fn, fp)   # class 0 = Real
    prec_1, rec_1, f1_1 = _prf(tp, fp, fn)   # class 1 = Fake

    support_0 = int(np.sum(y_true == 0))
    support_1 = int(np.sum(y_true == 1))

    f1_macro = (f1_0 + f1_1) / 2.0
    f1_weighted = (f1_0 * support_0 + f1_1 * support_1) / max(n, 1)
    precision_macro = (prec_0 + prec_1) / 2.0
    recall_macro = (rec_0 + rec_1) / 2.0

    f1_per_class = {
        class_names[0]: round(f1_0, 6),
        class_names[1]: round(f1_1, 6),
    }

    # --- ROC-AUC ------------------------------------------------------ #
    roc_auc = _compute_roc_auc(y_true, y_probs)

    return ClassificationMetrics(
        accuracy=round(accuracy, 6),
        f1_macro=round(f1_macro, 6),
        f1_weighted=round(f1_weighted, 6),
        f1_per_class=f1_per_class,
        precision_macro=round(precision_macro, 6),
        recall_macro=round(recall_macro, 6),
        roc_auc=round(roc_auc, 6),
        confusion_matrix=cm,
        num_samples=n,
    )


def _compute_roc_auc(y_true: np.ndarray, y_scores: np.ndarray) -> float:
    """
    Compute ROC-AUC using the trapezoidal rule (no sklearn dependency).

    Returns 0.0 when only one class is present.
    """
    if len(np.unique(y_true)) < 2:
        return 0.0

    # Sort by descending score
    desc = np.argsort(-y_scores)
    y_sorted = y_true[desc]
    scores_sorted = y_scores[desc]

    num_pos = np.sum(y_true == 1)
    num_neg = np.sum(y_true == 0)

    tp = 0
    fp = 0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Walk through thresholds (distinct score values)
    for i in range(len(y_sorted)):
        if y_sorted[i] == 1:
            tp += 1
        else:
            fp += 1

        # Only compute a point when the score changes or at the end
        if i == len(y_sorted) - 1 or scores_sorted[i] != scores_sorted[i + 1]:
            tpr = tp / max(num_pos, 1)
            fpr = fp / max(num_neg, 1)
            auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2.0
            prev_fpr = fpr
            prev_tpr = tpr

    return float(auc)


# ====================================================================== #
#  3.  INFERENCE LATENCY                                                  #
# ====================================================================== #

def measure_inference_latency(
    model: tf.keras.Model,
    test_data: tf.data.Dataset,
    batch_size: int = 32,
    warmup_batches: int = 3,
    max_batches: Optional[int] = None,
) -> LatencyMetrics:
    """
    Measure per-batch inference latency.

    Parameters
    ----------
    model : tf.keras.Model
    test_data : tf.data.Dataset
        Yields ``(images,)`` or ``(images, labels)``.
    batch_size : int
    warmup_batches : int
        Number of initial batches to discard (JIT warm-up).
    max_batches : int | None
        Cap the number of measured batches (``None`` = all).

    Returns
    -------
    LatencyMetrics
    """
    batched = test_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    times_ms: List[float] = []
    total_samples = 0
    batch_count = 0

    for batch in batched:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        batch_count += 1

        if batch_count <= warmup_batches:
            model(x, training=False)   # warm-up, discard timing
            continue

        t0 = time.perf_counter()
        model(x, training=False)
        t1 = time.perf_counter()

        times_ms.append((t1 - t0) * 1000.0)
        total_samples += int(x.shape[0])

        if max_batches is not None and len(times_ms) >= max_batches:
            break

    if not times_ms:
        logger.warning("No batches measured — dataset may be too small for warm-up.")
        return LatencyMetrics(batch_size=batch_size)

    arr = np.array(times_ms)
    return LatencyMetrics(
        mean_ms=round(float(arr.mean()), 4),
        std_ms=round(float(arr.std()), 4),
        min_ms=round(float(arr.min()), 4),
        max_ms=round(float(arr.max()), 4),
        p50_ms=round(float(np.percentile(arr, 50)), 4),
        p95_ms=round(float(np.percentile(arr, 95)), 4),
        p99_ms=round(float(np.percentile(arr, 99)), 4),
        num_batches=len(times_ms),
        batch_size=batch_size,
        total_samples=total_samples,
    )


# ====================================================================== #
#  4.  MODEL SIZE                                                         #
# ====================================================================== #

def measure_model_size(
    model: tf.keras.Model,
    save_path: Optional[str] = None,
) -> ModelSizeMetrics:
    """
    Count parameters and measure on-disk file size of the model.

    Parameters
    ----------
    model : tf.keras.Model
    save_path : str | None
        If ``None``, a temporary file is used and deleted afterwards.

    Returns
    -------
    ModelSizeMetrics
    """
    total = int(model.count_params())
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    non_trainable = total - trainable

    # On-disk size
    if save_path is None:
        tmp = tempfile.NamedTemporaryFile(suffix=".h5", delete=False)
        tmp.close()
        save_path = tmp.name
        cleanup = True
    else:
        cleanup = False

    try:
        model.save(save_path)
        file_bytes = os.path.getsize(save_path)
    finally:
        if cleanup and os.path.exists(save_path):
            os.unlink(save_path)

    return ModelSizeMetrics(
        total_params=total,
        trainable_params=trainable,
        non_trainable_params=non_trainable,
        file_size_bytes=file_bytes,
        file_size_mb=round(file_bytes / (1024 * 1024), 4),
    )


# ====================================================================== #
#  5.  FULL EVALUATOR                                                     #
# ====================================================================== #

class FederatedModelEvaluator:
    """
    One-stop evaluator that runs all metrics and produces reports.

    Parameters
    ----------
    model : tf.keras.Model
        The global model to evaluate.
    model_name : str
        Human-readable label (used in report filenames).
    reports_dir : str | Path
        Directory where reports are saved.
    class_names : list[str]
        Names for class 0 / class 1.  Defaults to ``["Real", "Fake"]``.
    """

    def __init__(
        self,
        model: tf.keras.Model,
        model_name: str = "effnet_global",
        reports_dir: str | Path = DEFAULT_REPORTS_DIR,
        class_names: Optional[List[str]] = None,
    ) -> None:
        self.model = model
        self.model_name = model_name
        self.reports_dir = Path(reports_dir)
        self.class_names = class_names or ["Real", "Fake"]

        # Ensure the reports directory exists
        self.reports_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ #
    #  Run full evaluation                                                #
    # ------------------------------------------------------------------ #

    def evaluate(
        self,
        test_data: tf.data.Dataset,
        batch_size: int = 32,
        threshold: float = 0.5,
        federated_round: Optional[int] = None,
        warmup_batches: int = 3,
        latency_max_batches: Optional[int] = None,
        extra_info: Optional[Dict[str, Any]] = None,
    ) -> EvaluationReport:
        """
        Run **all** evaluations and return a structured report.

        Parameters
        ----------
        test_data : tf.data.Dataset
            Yields ``(images, labels)``.
        batch_size : int
        threshold : float
            Classification decision threshold.
        federated_round : int | None
            If provided, included in the report metadata.
        warmup_batches : int
            Warm-up batches for latency measurement.
        latency_max_batches : int | None
            Cap measured batches for latency.
        extra_info : dict | None
            Arbitrary metadata to attach to the report.

        Returns
        -------
        EvaluationReport
        """
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        logger.info("Starting evaluation for '%s' …", self.model_name)

        # --- 1. Predictions ------------------------------------------- #
        y_true_list: List[np.ndarray] = []
        y_prob_list: List[np.ndarray] = []

        batched = test_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        for batch in batched:
            x, y = batch[0], batch[1]
            preds = self.model(x, training=False)
            y_true_list.append(y.numpy())
            y_prob_list.append(preds.numpy())

        y_true = np.concatenate(y_true_list).ravel()
        y_probs = np.concatenate(y_prob_list).ravel()

        # --- 2. Classification metrics -------------------------------- #
        cls_metrics = compute_classification_metrics(
            y_true, y_probs,
            threshold=threshold,
            class_names=self.class_names,
        )
        logger.info(
            "Classification — Acc: %.4f | F1-macro: %.4f | ROC-AUC: %.4f",
            cls_metrics.accuracy, cls_metrics.f1_macro, cls_metrics.roc_auc,
        )

        # --- 3. Inference latency ------------------------------------- #
        lat_metrics = measure_inference_latency(
            self.model, test_data,
            batch_size=batch_size,
            warmup_batches=warmup_batches,
            max_batches=latency_max_batches,
        )
        logger.info(
            "Latency — mean: %.2f ms | p95: %.2f ms | p99: %.2f ms",
            lat_metrics.mean_ms, lat_metrics.p95_ms, lat_metrics.p99_ms,
        )

        # --- 4. Model size -------------------------------------------- #
        size_metrics = measure_model_size(self.model)
        logger.info(
            "Model size — params: %s | disk: %.2f MB",
            f"{size_metrics.total_params:,}", size_metrics.file_size_mb,
        )

        report = EvaluationReport(
            model_name=self.model_name,
            timestamp=ts,
            federated_round=federated_round,
            classification=cls_metrics,
            latency=lat_metrics,
            model_size=size_metrics,
            extra=extra_info or {},
        )

        return report

    # ------------------------------------------------------------------ #
    #  Report persistence                                                 #
    # ------------------------------------------------------------------ #

    def save_report(
        self,
        report: EvaluationReport,
        tag: Optional[str] = None,
    ) -> Tuple[Path, Path]:
        """
        Save the report as both JSON and a human-readable text file.

        Parameters
        ----------
        report : EvaluationReport
        tag : str | None
            Optional suffix for the filename (e.g. ``"round_05"``).

        Returns
        -------
        (json_path, txt_path) : tuple[Path, Path]
        """
        ts_slug = datetime.now().strftime("%Y%m%d_%H%M%S")
        base = f"{self.model_name}_{ts_slug}"
        if tag:
            base += f"_{tag}"

        json_path = self.reports_dir / f"{base}.json"
        txt_path = self.reports_dir / f"{base}.txt"

        # --- JSON ----------------------------------------------------- #
        json_path.write_text(
            json.dumps(report.to_dict(), indent=2, default=str),
            encoding="utf-8",
        )

        # --- Human-readable text -------------------------------------- #
        txt_path.write_text(
            self._format_text_report(report),
            encoding="utf-8",
        )

        logger.info("Reports saved → %s  &  %s", json_path.name, txt_path.name)
        return json_path, txt_path

    # ------------------------------------------------------------------ #
    #  Text report formatter                                              #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _format_text_report(r: EvaluationReport) -> str:
        """Render a pretty-printed text report."""
        sep = "=" * 62
        cls = r.classification
        lat = r.latency
        sz = r.model_size

        lines = [
            sep,
            "  EVALUATION REPORT",
            sep,
            f"  Model:            {r.model_name}",
            f"  Timestamp:        {r.timestamp}",
        ]
        if r.federated_round is not None:
            lines.append(f"  Federated Round:  {r.federated_round}")
        lines.append("")

        # Classification
        lines += [
            "-" * 62,
            "  CLASSIFICATION METRICS",
            "-" * 62,
            f"  Samples evaluated:  {cls.num_samples:,}",
            f"  Accuracy:           {cls.accuracy:.4f}",
            f"  F1 (macro):         {cls.f1_macro:.4f}",
            f"  F1 (weighted):      {cls.f1_weighted:.4f}",
            f"  Precision (macro):  {cls.precision_macro:.4f}",
            f"  Recall (macro):     {cls.recall_macro:.4f}",
            f"  ROC-AUC:            {cls.roc_auc:.4f}",
            "",
            "  Per-class F1:",
        ]
        for name, f1 in cls.f1_per_class.items():
            lines.append(f"    {name:<16} {f1:.4f}")

        if cls.confusion_matrix is not None:
            cm = cls.confusion_matrix
            lines += [
                "",
                "  Confusion Matrix:     Pred=0   Pred=1",
                f"    Actual=0 (Real)    {cm[0][0]:>7,}  {cm[0][1]:>7,}",
                f"    Actual=1 (Fake)    {cm[1][0]:>7,}  {cm[1][1]:>7,}",
            ]

        # Latency
        lines += [
            "",
            "-" * 62,
            "  INFERENCE LATENCY",
            "-" * 62,
            f"  Batches measured:   {lat.num_batches}  (batch_size={lat.batch_size})",
            f"  Total samples:      {lat.total_samples:,}",
            f"  Mean:               {lat.mean_ms:.2f} ms",
            f"  Std:                {lat.std_ms:.2f} ms",
            f"  Min:                {lat.min_ms:.2f} ms",
            f"  Max:                {lat.max_ms:.2f} ms",
            f"  P50 (median):       {lat.p50_ms:.2f} ms",
            f"  P95:                {lat.p95_ms:.2f} ms",
            f"  P99:                {lat.p99_ms:.2f} ms",
        ]

        # Model size
        lines += [
            "",
            "-" * 62,
            "  MODEL SIZE",
            "-" * 62,
            f"  Total params:       {sz.total_params:,}",
            f"  Trainable params:   {sz.trainable_params:,}",
            f"  Non-trainable:      {sz.non_trainable_params:,}",
            f"  File size:          {sz.file_size_mb:.2f} MB  ({sz.file_size_bytes:,} bytes)",
        ]

        # Extra
        if r.extra:
            lines += [
                "",
                "-" * 62,
                "  ADDITIONAL INFO",
                "-" * 62,
            ]
            for k, v in r.extra.items():
                lines.append(f"  {k}: {v}")

        lines += ["", sep, "  END OF REPORT", sep, ""]
        return "\n".join(lines)

    # ------------------------------------------------------------------ #
    #  Comparative report across rounds                                   #
    # ------------------------------------------------------------------ #

    def save_comparison_report(
        self,
        reports: List[EvaluationReport],
        filename: str = "comparison",
    ) -> Tuple[Path, Path]:
        """
        Generate a comparison table across multiple evaluation reports
        (e.g. one per federated round).

        Returns ``(json_path, txt_path)``.
        """
        ts_slug = datetime.now().strftime("%Y%m%d_%H%M%S")
        base = f"{filename}_{ts_slug}"
        json_path = self.reports_dir / f"{base}.json"
        txt_path = self.reports_dir / f"{base}.txt"

        # JSON
        json_data = [r.to_dict() for r in reports]
        json_path.write_text(
            json.dumps(json_data, indent=2, default=str),
            encoding="utf-8",
        )

        # Text table
        header = (
            f"{'Round':>6} | {'Acc':>8} | {'F1-mac':>8} | {'ROC-AUC':>8} | "
            f"{'Lat(ms)':>8} | {'P95(ms)':>8} | {'Size(MB)':>9}"
        )
        sep = "-" * len(header)
        lines = [
            "=" * len(header),
            "  COMPARISON REPORT",
            "=" * len(header),
            "",
            header,
            sep,
        ]
        for r in reports:
            rnd = r.federated_round if r.federated_round is not None else "—"
            lines.append(
                f"{rnd:>6} | {r.classification.accuracy:>8.4f} | "
                f"{r.classification.f1_macro:>8.4f} | "
                f"{r.classification.roc_auc:>8.4f} | "
                f"{r.latency.mean_ms:>8.2f} | "
                f"{r.latency.p95_ms:>8.2f} | "
                f"{r.model_size.file_size_mb:>9.2f}"
            )
        lines += [sep, ""]

        txt_path.write_text("\n".join(lines), encoding="utf-8")
        logger.info("Comparison report saved → %s  &  %s", json_path.name, txt_path.name)
        return json_path, txt_path


# ====================================================================== #
#  6.  CONVENIENCE: evaluate + save in one call                           #
# ====================================================================== #

def evaluate_and_report(
    model: tf.keras.Model,
    test_data: tf.data.Dataset,
    model_name: str = "effnet_global",
    reports_dir: str | Path = DEFAULT_REPORTS_DIR,
    batch_size: int = 32,
    threshold: float = 0.5,
    federated_round: Optional[int] = None,
    extra_info: Optional[Dict[str, Any]] = None,
    tag: Optional[str] = None,
) -> EvaluationReport:
    """
    One-liner: evaluate the model and write JSON + text reports.

    Returns the ``EvaluationReport`` dataclass for programmatic use.
    """
    evaluator = FederatedModelEvaluator(
        model=model,
        model_name=model_name,
        reports_dir=reports_dir,
    )
    report = evaluator.evaluate(
        test_data,
        batch_size=batch_size,
        threshold=threshold,
        federated_round=federated_round,
        extra_info=extra_info,
    )
    evaluator.save_report(report, tag=tag)
    return report


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Evaluation Metrics & Report Generation — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model --------------------------------------- #
    INPUT_DIM = 16
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # ---- 2. Synthetic test data -------------------------------------- #
    NUM_SAMPLES = 500
    x_test = np.random.randn(NUM_SAMPLES, INPUT_DIM).astype(np.float32)
    y_test = np.random.randint(0, 2, size=(NUM_SAMPLES,)).astype(np.float32)
    test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

    # ---- 3. Run evaluation ------------------------------------------- #
    evaluator = FederatedModelEvaluator(
        model=model,
        model_name="demo_effnet",
        reports_dir=DEFAULT_REPORTS_DIR,
    )

    report = evaluator.evaluate(
        test_data=test_ds,
        batch_size=64,
        federated_round=5,
        extra_info={"dataset": "synthetic", "notes": "smoke-test"},
    )

    # ---- 4. Print text report to console ----------------------------- #
    print(evaluator._format_text_report(report))

    # ---- 5. Save reports to disk ------------------------------------- #
    json_path, txt_path = evaluator.save_report(report, tag="round_05")
    print(f"JSON report: {json_path}")
    print(f"Text report: {txt_path}")

    # ---- 6. Simulate multi-round comparison -------------------------- #
    reports: List[EvaluationReport] = []
    for rnd in range(1, 4):
        # Slightly perturb weights to simulate different rounds
        for w in model.trainable_weights:
            w.assign_add(tf.random.normal(w.shape, stddev=0.01))
        r = evaluator.evaluate(test_ds, batch_size=64, federated_round=rnd)
        reports.append(r)

    cmp_json, cmp_txt = evaluator.save_comparison_report(reports)
    print(f"\nComparison JSON: {cmp_json}")
    print(f"Comparison text: {cmp_txt}")

    print("\nDone.")


Writing evaluation_metrics.py


### Part 6 — Main FL Cycle Orchestrator (non-TFF, pure Keras)


In [12]:
%%writefile federated_learning_cycle.py
"""
Federated Learning Cycle — Main Orchestrator
==============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Integrates **all five** modules into one end-to-end pipeline:

 1. **Enhanced Client Selection**  (``enhanced_client_selection.py``)
 2. **Update Validation & Contribution Weighing** (``update_validation.py``)
 3. **Server-Side Knowledge Distillation**  (``knowledge_distillation.py``)
 4. **Client Reputation & Ledger**  (``client_reputation_ledger.py``)
 5. **Evaluation Metrics & Reporting**  (``evaluation_metrics.py``)

Cycle summary (per round)
-------------------------
1.  Select clients from the pool via multi-criteria scoring  (Part 1).
2.  Distribute global weights → clients train locally for 5 epochs.
3.  Validate updates, assign contribution weights, reject harmful ones,
    and perform weighted aggregation  (Part 2).
4.  Optionally refine the aggregated model with server-side knowledge
    distillation from the contribution-weighted client ensemble  (Part 3).
5.  Update the persistent reputation ledger based on gains & c_i  (Part 4).
6.  Every ``eval_every`` rounds, run full evaluation & save reports  (Part 5).
7.  After all rounds, convert the final model to TensorFlow Lite.

Configuration
-------------
- **Model**: ``effnet_ffpp_small_data.h5``   (EfficientNet‑based binary
  classifier — Real vs Fake)
- **Devices**: 100 simulated federated clients
- **Local epochs per round**: 5
- **Global aggregation rounds**: 50
- **Frameworks**: TensorFlow / TF Lite / TF Federated concepts
"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ====================================================================== #
#  Module imports  (Parts 1–5)                                            #
# ====================================================================== #

from enhanced_client_selection import (       # Part 1
    ClientMetrics,
    FederatedClient,
    ReputationLedger,
    SelectionWeights,
    EnhancedClientSelector,
)
from update_validation import (               # Part 2
    ContributionWeights,
    ClippingConfig,
    ClientUpdateRecord,
    UpdateValidator,
    flatten_weights,
    unflatten_weights,
    compute_update_delta,
    apply_update,
)
from knowledge_distillation import (          # Part 3
    DistillationConfig,
    run_distillation_round,
)
from client_reputation_ledger import (        # Part 4
    ReputationConfig,
    ClientReputationLedger,
    update_ledger_from_records,
)
from evaluation_metrics import (              # Part 5
    FederatedModelEvaluator,
    evaluate_and_report,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class FLCycleConfig:
    """
    Central configuration for the entire Federated Learning cycle.

    Parameters
    ----------
    model_path : str
        Path to the pre-trained EfficientNet HDF5 model.
    num_devices : int
        Total number of simulated federated client devices.
    local_epochs : int
        Number of local training epochs per client per round.
    global_rounds : int
        Total number of federated aggregation rounds.
    clients_per_round : int
        Target number of clients selected each round (``k``).
    local_batch_size : int
        Batch size for client-side local training.
    local_lr : float
        Client-side optimiser learning rate.
    eval_every : int
        Run full evaluation every N rounds (also round 1 & last).
    enable_distillation : bool
        Whether to run server-side knowledge distillation each round.
    distillation_config : DistillationConfig
        Hyper-parameters for the distillation loop.
    selection_weights : SelectionWeights
        Weights for the multi-criteria client scoring  (Part 1).
    contribution_weights : ContributionWeights
        α, β, γ, δ for the contribution scoring  (Part 2).
    clipping_config : ClippingConfig
        Norm-clipping parameters for update validation.
    harmful_threshold : float
        ε — reject updates with ``G_i < −ε``.
    reputation_config : ReputationConfig
        Parameters for the persistent reputation ledger  (Part 4).
    reports_dir : str
        Directory for saving evaluation reports  (Part 5).
    tflite_output_path : str
        Path for the final TF Lite model export.
    input_shape : tuple
        Input shape expected by the model  (H, W, C).
    """
    # -- Core FL settings ---------------------------------------------- #
    model_path: str = "effnet_ffpp_small_data.h5"
    num_devices: int = 100
    local_epochs: int = 5
    global_rounds: int = 50
    clients_per_round: int = 15
    local_batch_size: int = 32
    local_lr: float = 1e-4
    eval_every: int = 5

    # -- Distillation (Part 3) ---------------------------------------- #
    enable_distillation: bool = True
    distillation_config: DistillationConfig = field(
        default_factory=lambda: DistillationConfig(
            temperature=3.0,
            lam=0.7,
            epochs=3,
            batch_size=32,
            learning_rate=1e-4,
        )
    )

    # -- Client selection (Part 1) ------------------------------------- #
    selection_weights: SelectionWeights = field(
        default_factory=lambda: SelectionWeights(
            w_v=0.30,
            w_d=0.20,
            w_l=0.10,
            w_r=0.25,
            w_s=0.15,
        )
    )

    # -- Update validation (Part 2) ----------------------------------- #
    contribution_weights: ContributionWeights = field(
        default_factory=lambda: ContributionWeights(
            alpha=0.35,   # validation gain
            beta=0.20,    # similarity
            gamma=0.20,   # data volume
            delta=0.25,   # reputation
        )
    )
    clipping_config: ClippingConfig = field(
        default_factory=lambda: ClippingConfig(
            clip_threshold=10.0,
            clip_value=5.0,
        )
    )
    harmful_threshold: float = 0.02

    # -- Reputation ledger (Part 4) ------------------------------------ #
    reputation_config: ReputationConfig = field(
        default_factory=lambda: ReputationConfig(
            theta=0.0,
            gamma=0.10,
            decay_rate=0.99,
            floor=0.05,
            ceiling=1.0,
            initial_reputation=0.50,
            penalty_factor=0.05,
        )
    )

    # -- Evaluation & output (Part 5) --------------------------------- #
    reports_dir: str = "reports"
    tflite_output_path: str = "effnet_global_final.tflite"
    input_shape: Tuple[int, ...] = (224, 224, 3)


# ====================================================================== #
#  2.  DATA HELPERS  (simulation — replace with real FF++ loaders)        #
# ====================================================================== #

def _generate_synthetic_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Generate a synthetic (random) labelled dataset for smoke-testing.

    In production, replace this with a real FF++ c23 data loader that
    returns ``(image, label)`` pairs as ``tf.data.Dataset``.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    y = rng.randint(0, 2, size=(num_samples,)).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((x, y))


def _generate_proxy_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Generate unlabelled proxy data (images only) for knowledge
    distillation.  Replace with real FF++ c23 unlabelled images.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    return tf.data.Dataset.from_tensor_slices(x)


def partition_data_iid(
    full_dataset: tf.data.Dataset,
    num_clients: int,
    seed: int = 42,
) -> Dict[str, tf.data.Dataset]:
    """
    IID partition: shuffle the dataset and split evenly across clients.

    Returns
    -------
    dict[str, tf.data.Dataset]
        ``{client_id: local_dataset}``
    """
    all_data = list(full_dataset.shuffle(buffer_size=10_000, seed=seed))
    total = len(all_data)
    shard_size = max(1, total // num_clients)

    partitions: Dict[str, tf.data.Dataset] = {}
    for i in range(num_clients):
        cid = f"client_{i:03d}"
        start = i * shard_size
        end = min(start + shard_size, total)
        if start >= total:
            # Wrap around for the last few clients
            start = start % total
            end = start + shard_size

        shard_x = [elem[0].numpy() for elem in all_data[start:end]]
        shard_y = [elem[1].numpy() for elem in all_data[start:end]]

        if len(shard_x) == 0:
            shard_x = [all_data[0][0].numpy()]
            shard_y = [all_data[0][1].numpy()]

        partitions[cid] = tf.data.Dataset.from_tensor_slices(
            (np.stack(shard_x), np.array(shard_y))
        )
    return partitions


# ====================================================================== #
#  3.  TF LITE CONVERSION                                                 #
# ====================================================================== #

def convert_to_tflite(
    model: tf.keras.Model,
    output_path: str,
    quantise: bool = False,
) -> str:
    """
    Convert a Keras model to TensorFlow Lite format.

    Parameters
    ----------
    model : tf.keras.Model
    output_path : str
    quantise : bool
        If ``True``, apply dynamic-range (post-training) quantisation.

    Returns
    -------
    output_path : str
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantise:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    Path(output_path).write_bytes(tflite_model)
    size_mb = len(tflite_model) / (1024 * 1024)
    logger.info(
        "TF Lite model saved → %s  (%.2f MB, quantised=%s)",
        output_path, size_mb, quantise,
    )
    return output_path


# ====================================================================== #
#  4.  FEDERATED LEARNING CYCLE  (main orchestrator)                      #
# ====================================================================== #

class FederatedLearningCycle:
    """
    End-to-end federated learning cycle orchestrating Parts 1–5.

    Parameters
    ----------
    config : FLCycleConfig
        The full cycle configuration.
    """

    def __init__(self, config: Optional[FLCycleConfig] = None) -> None:
        self.config = config or FLCycleConfig()
        self.global_model: Optional[tf.keras.Model] = None
        self.clients: List[FederatedClient] = []
        self.reputation_ledger: Optional[ClientReputationLedger] = None
        self.basic_ledger: Optional[ReputationLedger] = None
        self.selector: Optional[EnhancedClientSelector] = None
        self.validator: Optional[UpdateValidator] = None
        self.evaluator: Optional[FederatedModelEvaluator] = None

        # Round history
        self.history: Dict[str, list] = {
            "round": [],
            "global_accuracy": [],
            "selected_clients": [],
            "num_accepted": [],
            "num_rejected": [],
            "distillation_loss": [],
        }

    # ------------------------------------------------------------------ #
    #  Initialisation helpers                                              #
    # ------------------------------------------------------------------ #

    def load_global_model(self) -> tf.keras.Model:
        """Load the pre-trained EfficientNet model from disk."""
        cfg = self.config
        logger.info("Loading global model from %s …", cfg.model_path)
        # Register EfficientNet preprocessing so Keras can deserialize the .h5
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        _custom = {"preprocess_input": _effnet_preprocess}
        model = tf.keras.models.load_model(
            cfg.model_path, custom_objects=_custom, compile=False,
        )
        model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        logger.info(
            "Global model loaded — %s params, input shape %s",
            f"{model.count_params():,}", model.input_shape,
        )
        self.global_model = model
        return model

    def create_clients(
        self,
        client_data: Dict[str, tf.data.Dataset],
    ) -> List[FederatedClient]:
        """
        Instantiate ``FederatedClient`` objects for every device.

        Parameters
        ----------
        client_data : dict[str, tf.data.Dataset]
            Pre-partitioned local data per client.
        """
        rng = np.random.RandomState(42)
        clients: List[FederatedClient] = []

        for cid, local_ds in client_data.items():
            n_samples = sum(1 for _ in local_ds)
            metrics = ClientMetrics(
                local_validation_metric=float(rng.uniform(0.4, 0.9)),
                data_volume=n_samples,
                inference_latency=float(rng.uniform(0.01, 0.15)),
                last_selected_round=0,
            )
            clients.append(FederatedClient(
                client_id=cid,
                local_data=local_ds,
                metrics=metrics,
            ))

        logger.info("Created %d federated clients.", len(clients))
        self.clients = clients
        return clients

    def setup_components(self) -> None:
        """
        Wire all the modules together: reputation ledger, client
        selector, update validator, and evaluator.
        """
        cfg = self.config
        assert self.global_model is not None, "Call load_global_model() first."
        assert len(self.clients) > 0, "Call create_clients() first."

        # -- Part 4: Reputation ledger --------------------------------- #
        self.reputation_ledger = ClientReputationLedger(config=cfg.reputation_config)
        for c in self.clients:
            self.reputation_ledger.register(c.client_id)
        # Create a basic ledger view for Part 1
        self.basic_ledger = self.reputation_ledger.as_basic_ledger()

        # -- Part 1: Enhanced client selector -------------------------- #
        self.selector = EnhancedClientSelector(
            clients=self.clients,
            reputation_ledger=self.basic_ledger,
            weights=cfg.selection_weights,
            target_k=cfg.clients_per_round,
        )

        # -- Part 2: Update validator ---------------------------------- #
        self.validator = UpdateValidator(
            global_model=self.global_model,
            reputation_ledger=self.basic_ledger,
            weights=cfg.contribution_weights,
            clipping=cfg.clipping_config,
            harmful_threshold=cfg.harmful_threshold,
            batch_size=cfg.local_batch_size,
        )

        # -- Part 5: Evaluator ---------------------------------------- #
        self.evaluator = FederatedModelEvaluator(
            model=self.global_model,
            model_name="effnet_global",
            reports_dir=cfg.reports_dir,
        )

        logger.info("All FL-cycle components initialised.")

    # ------------------------------------------------------------------ #
    #  Local training                                                     #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int]:
        """
        Train a local model replica for ``local_epochs`` on the client's
        private data, starting from the current global weights.

        Returns
        -------
        updated_weights : list[np.ndarray]
        num_samples : int
        """
        cfg = self.config

        local_model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            logger.warning(
                "Client %s has no local data — returning global weights.",
                client.client_id,
            )
            return global_weights, 0

        dataset = client.local_data.batch(cfg.local_batch_size)
        local_model.fit(dataset, epochs=cfg.local_epochs, verbose=0)
        return local_model.get_weights(), client.metrics.data_volume

    # ------------------------------------------------------------------ #
    #  Sync reputation bridge                                             #
    # ------------------------------------------------------------------ #

    def _sync_reputation_to_basic_ledger(self) -> None:
        """
        Copy the extended ledger scores into the basic ``ReputationLedger``
        view used by Parts 1 & 2.
        """
        updated_basic = self.reputation_ledger.as_basic_ledger()
        # Update the internal _scores dict of the existing basic ledger
        # so the selector and validator see fresh reputations.
        self.basic_ledger._scores = updated_basic._scores

    # ------------------------------------------------------------------ #
    #  Single round                                                       #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        current_round: int,
        server_val_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, Any]:
        """
        Execute one complete federated round.

        Pipeline
        --------
        1.  **Client selection**   (Part 1)
        2.  **Local training**     — each selected client trains for
            ``local_epochs`` on its private data.
        3.  **Update validation & weighted aggregation**  (Part 2)
        4.  **Knowledge distillation**  (Part 3, optional)
        5.  **Reputation ledger update**  (Part 4)
        6.  Quick accuracy check on the server validation set.

        Parameters
        ----------
        current_round : int
        server_val_data : tf.data.Dataset
            Held-out server validation set  ``(x, y)``  for baseline
            comparison and post-round evaluation.
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy inputs for distillation  (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for the combined distillation loss  (Part 3).

        Returns
        -------
        round_info : dict
        """
        cfg = self.config

        logger.info("=" * 70)
        logger.info("  ROUND %d / %d", current_round, cfg.global_rounds)
        logger.info("=" * 70)

        # ── 1. Client selection  (Part 1) ──────────────────────────── #
        selected: List[FederatedClient] = self.selector.select(
            current_round=current_round,
        )
        selected_ids = [c.client_id for c in selected]
        logger.info(
            "Selected %d / %d clients: %s",
            len(selected), len(self.clients), selected_ids,
        )

        global_weights = self.global_model.get_weights()

        # ── 2. Local training ──────────────────────────────────────── #
        client_updates: Dict[str, List[np.ndarray]] = {}
        data_volumes: Dict[str, int] = {}

        for client in selected:
            t0 = time.time()
            updated_w, n_samples = self._local_train(client, global_weights)
            elapsed = time.time() - t0

            client_updates[client.client_id] = updated_w
            data_volumes[client.client_id] = max(n_samples, 1)

            # Refresh client metrics
            client.metrics.last_selected_round = current_round
            client.metrics.inference_latency = elapsed / max(n_samples, 1)

            logger.debug(
                "Client %s trained — %d samples, %.2fs",
                client.client_id, n_samples, elapsed,
            )

        # ── 3. Update validation & aggregation  (Part 2) ──────────── #
        records: List[ClientUpdateRecord] = self.validator.validate_updates(
            client_updates=client_updates,
            data_volumes=data_volumes,
            server_val_data=server_val_data,
        )

        new_weights = self.validator.aggregate_weighted(
            records, global_weights,
        )
        self.global_model.set_weights(new_weights)
        self.validator.global_model.set_weights(new_weights)

        # Rejection statistics
        num_accepted = sum(1 for r in records if not r.rejected)
        num_rejected = sum(1 for r in records if r.rejected)
        logger.info(
            "Updates: %d accepted, %d rejected, out of %d total.",
            num_accepted, num_rejected, len(records),
        )

        # ── 4. Knowledge distillation  (Part 3, optional) ─────────── #
        distill_loss = None
        if cfg.enable_distillation and proxy_data is not None:
            # Only distill from clients with positive contribution
            contribution_weights = {
                r.client_id: r.contribution_weight
                for r in records
                if not r.rejected and r.contribution_weight > 0
            }

            if len(contribution_weights) >= 2:
                logger.info(
                    "Running knowledge distillation with %d teacher(s) …",
                    len(contribution_weights),
                )
                kd_history = run_distillation_round(
                    global_model=self.global_model,
                    client_weights={
                        cid: client_updates[cid]
                        for cid in contribution_weights
                    },
                    contribution_weights=contribution_weights,
                    proxy_data=proxy_data,
                    supervised_data=supervised_data,
                    config=cfg.distillation_config,
                )
                distill_loss = kd_history["loss_total"][-1]
                logger.info(
                    "Distillation complete — final loss %.5f", distill_loss,
                )
                # Update validator reference after distillation
                self.validator.global_model.set_weights(
                    self.global_model.get_weights()
                )
            else:
                logger.info(
                    "Skipping distillation — fewer than 2 contributing "
                    "clients (%d).", len(contribution_weights),
                )

        # ── 5. Reputation ledger update  (Part 4) ─────────────────── #
        updated_reps = update_ledger_from_records(
            ledger=self.reputation_ledger,
            records=records,
            current_round=current_round,
        )
        # Also feed reputation changes back via the basic (Part 1/2)
        # validator update_reputations for the simple ledger
        self.validator.update_reputations(records)

        # Sync extended ledger → basic ledger used by selector/validator
        self._sync_reputation_to_basic_ledger()

        logger.info(
            "Reputation update complete — top 5: %s",
            dict(list(self.reputation_ledger.ranked()[:5])),
        )

        # ── 6. Quick evaluation on val set ────────────────────────── #
        val_result = self.global_model.evaluate(
            server_val_data.batch(cfg.local_batch_size),
            verbose=0,
            return_dict=True,
        )
        acc = val_result.get("accuracy", 0.0)
        logger.info("Round %d — post-round accuracy: %.4f", current_round, acc)

        # ── Collect summary ────────────────────────────────────────── #
        round_info = {
            "round": current_round,
            "selected": selected_ids,
            "num_accepted": num_accepted,
            "num_rejected": num_rejected,
            "records": records,
            "global_accuracy": acc,
            "distillation_loss": distill_loss,
        }
        return round_info

    # ------------------------------------------------------------------ #
    #  Full FL cycle                                                      #
    # ------------------------------------------------------------------ #

    def run(
        self,
        server_val_data: tf.data.Dataset,
        test_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Run the full federated learning cycle for ``global_rounds``
        rounds.

        Parameters
        ----------
        server_val_data : tf.data.Dataset
            Held-out validation set used for update scoring & quick eval.
        test_data : tf.data.Dataset
            Independent test set for full evaluation (Part 5).
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy data for knowledge distillation (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for the combined distillation loss (Part 3).

        Returns
        -------
        history : dict
        """
        cfg = self.config

        logger.info("╔══════════════════════════════════════════════════════╗")
        logger.info("║  FEDERATED LEARNING CYCLE — START                   ║")
        logger.info("║  Devices: %3d  |  Rounds: %3d  |  Local epochs: %d  ║",
                     cfg.num_devices, cfg.global_rounds, cfg.local_epochs)
        logger.info("╚══════════════════════════════════════════════════════╝")

        # -- Pre-cycle evaluation (baseline) --------------------------- #
        logger.info("Evaluating baseline model before federated training …")
        baseline_report = self.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline"},
        )
        self.evaluator.save_report(baseline_report, tag="round_000_baseline")
        logger.info(
            "Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
            baseline_report.classification.accuracy,
            baseline_report.classification.f1_macro,
            baseline_report.classification.roc_auc,
        )

        all_reports = [baseline_report]
        cycle_start = time.time()

        # ============================================================== #
        #  MAIN LOOP                                                      #
        # ============================================================== #

        for t in range(1, cfg.global_rounds + 1):
            round_start = time.time()

            info = self.execute_round(
                current_round=t,
                server_val_data=server_val_data,
                proxy_data=proxy_data,
                supervised_data=supervised_data,
            )

            # Record history
            self.history["round"].append(t)
            self.history["global_accuracy"].append(info["global_accuracy"])
            self.history["selected_clients"].append(info["selected"])
            self.history["num_accepted"].append(info["num_accepted"])
            self.history["num_rejected"].append(info["num_rejected"])
            self.history["distillation_loss"].append(info["distillation_loss"])

            round_elapsed = time.time() - round_start
            logger.info("Round %d completed in %.1fs.", t, round_elapsed)

            # -- Periodic full evaluation (Part 5) --------------------- #
            is_eval_round = (
                t % cfg.eval_every == 0
                or t == 1
                or t == cfg.global_rounds
            )
            if is_eval_round:
                logger.info("Running full evaluation (round %d) …", t)
                report = self.evaluator.evaluate(
                    test_data=test_data,
                    batch_size=cfg.local_batch_size,
                    federated_round=t,
                    extra_info={
                        "accepted": info["num_accepted"],
                        "rejected": info["num_rejected"],
                        "distillation_loss": info["distillation_loss"],
                    },
                )
                self.evaluator.save_report(report, tag=f"round_{t:03d}")
                all_reports.append(report)
                logger.info(
                    "Round %d eval — Acc: %.4f, F1: %.4f, AUC: %.4f",
                    t,
                    report.classification.accuracy,
                    report.classification.f1_macro,
                    report.classification.roc_auc,
                )

        total_elapsed = time.time() - cycle_start

        # ============================================================== #
        #  POST-CYCLE                                                     #
        # ============================================================== #

        logger.info("╔══════════════════════════════════════════════════════╗")
        logger.info("║  FEDERATED LEARNING CYCLE — COMPLETE                ║")
        logger.info("║  Total time: %.1fs                                  ║", total_elapsed)
        logger.info("╚══════════════════════════════════════════════════════╝")

        # -- Save comparison report ------------------------------------ #
        if len(all_reports) > 1:
            self.evaluator.save_comparison_report(all_reports)

        # -- Save reputation ledger ------------------------------------ #
        ledger_path = Path(cfg.reports_dir) / "reputation_ledger_final.json"
        self.reputation_ledger.save(str(ledger_path))

        # -- Convert to TF Lite ---------------------------------------- #
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path,
            quantise=False,
        )
        # Also save a quantised version
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True,
        )

        # -- Print final summary --------------------------------------- #
        self._print_summary()

        return self.history

    # ------------------------------------------------------------------ #
    #  Summary                                                            #
    # ------------------------------------------------------------------ #

    def _print_summary(self) -> None:
        """Pretty-print a final training summary."""
        h = self.history
        if not h["round"]:
            return

        best_idx = int(np.argmax(h["global_accuracy"]))
        best_round = h["round"][best_idx]
        best_acc = h["global_accuracy"][best_idx]
        final_acc = h["global_accuracy"][-1]

        # Reputation statistics
        stats = self.reputation_ledger.statistics()

        print("\n" + "=" * 60)
        print("  FEDERATED LEARNING CYCLE — FINAL SUMMARY")
        print("=" * 60)
        print(f"  Total rounds:          {len(h['round'])}")
        print(f"  Best accuracy:         {best_acc:.4f}  (round {best_round})")
        print(f"  Final accuracy:        {final_acc:.4f}")
        print(f"  Mean accuracy:         {np.mean(h['global_accuracy']):.4f}")
        print(f"  Total accepted:        {sum(h['num_accepted'])}")
        print(f"  Total rejected:        {sum(h['num_rejected'])}")
        print(f"  Reputation — mean:     {stats.get('mean_reputation', 0):.4f}")
        print(f"  Reputation — std:      {stats.get('std_reputation', 0):.4f}")

        kd_losses = [l for l in h["distillation_loss"] if l is not None]
        if kd_losses:
            print(f"  Avg distillation loss: {np.mean(kd_losses):.5f}")
        print(f"  TF Lite model saved:   {self.config.tflite_output_path}")
        print("=" * 60 + "\n")


# ====================================================================== #
#  5.  ENTRY POINT                                                        #
# ====================================================================== #

def main() -> None:
    """
    Main entry point — runs the full Enhanced Federated Learning Cycle
    for DeepFake detection.
    """
    np.random.seed(42)
    tf.random.set_seed(42)

    # ------------------------------------------------------------------ #
    #  Configuration                                                      #
    # ------------------------------------------------------------------ #
    config = FLCycleConfig(
        model_path="effnet_ffpp_small_data.h5",
        num_devices=100,
        local_epochs=5,
        global_rounds=50,
        clients_per_round=15,
        local_batch_size=32,
        local_lr=1e-4,
        eval_every=10,
        enable_distillation=True,
    )

    cycle = FederatedLearningCycle(config)

    # ------------------------------------------------------------------ #
    #  Load global model                                                  #
    # ------------------------------------------------------------------ #
    model = cycle.load_global_model()
    input_shape = model.input_shape[1:]        # strip batch dim
    config.input_shape = input_shape
    logger.info("Model input shape: %s", input_shape)

    # ------------------------------------------------------------------ #
    #  Prepare data  (synthetic — replace with real FF++ c23 loaders)     #
    # ------------------------------------------------------------------ #
    #
    #  In production, replace these synthetic generators with actual
    #  FF++ c23 data loaders:
    #
    #    train_data = load_ffpp_c23_train(...)      # for client partitions
    #    val_data   = load_ffpp_c23_val(...)        # server validation
    #    test_data  = load_ffpp_c23_test(...)       # independent test set
    #    proxy_data = load_ffpp_c23_unlabelled(...) # for distillation
    #
    logger.info("Generating synthetic data for %d clients …", config.num_devices)

    TOTAL_TRAIN_SAMPLES = config.num_devices * 10   # 10 samples/client
    VAL_SAMPLES   = 200
    TEST_SAMPLES  = 300
    PROXY_SAMPLES = 150

    train_data = _generate_synthetic_data(
        TOTAL_TRAIN_SAMPLES, input_shape, seed=1,
    )
    server_val_data = _generate_synthetic_data(
        VAL_SAMPLES, input_shape, seed=2,
    )
    test_data = _generate_synthetic_data(
        TEST_SAMPLES, input_shape, seed=3,
    )
    proxy_data = _generate_proxy_data(
        PROXY_SAMPLES, input_shape, seed=4,
    )
    supervised_data = _generate_synthetic_data(
        100, input_shape, seed=5,
    )

    # Partition training data across clients (IID)
    client_data = partition_data_iid(
        train_data, config.num_devices, seed=42,
    )

    # ------------------------------------------------------------------ #
    #  Create clients & wire components                                   #
    # ------------------------------------------------------------------ #
    cycle.create_clients(client_data)
    cycle.setup_components()

    # ------------------------------------------------------------------ #
    #  Run the Federated Learning Cycle                                   #
    # ------------------------------------------------------------------ #
    history = cycle.run(
        server_val_data=server_val_data,
        test_data=test_data,
        proxy_data=proxy_data,
        supervised_data=supervised_data,
    )

    logger.info("Federated Learning Cycle finished. History keys: %s",
                list(history.keys()))


# ====================================================================== #
#  DEMO / SMOKE-TEST  (lightweight — reduced params for quick run)        #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Enhanced Federated Learning Cycle — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- Use a small synthetic model for smoke-testing --------------- #
    # (The real cycle uses effnet_ffpp_small_data.h5 via main())
    INPUT_DIM = 16
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    demo_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Demo model — {demo_model.count_params()} params, "
          f"input shape {demo_model.input_shape}")

    # ---- Configuration (reduced for speed) --------------------------- #
    demo_config = FLCycleConfig(
        model_path="(in-memory demo)",
        num_devices=8,
        local_epochs=1,
        global_rounds=3,
        clients_per_round=4,
        local_batch_size=16,
        local_lr=1e-3,
        eval_every=1,
        enable_distillation=True,
        distillation_config=DistillationConfig(
            temperature=3.0,
            lam=0.7,
            epochs=2,
            batch_size=16,
            learning_rate=1e-3,
        ),
    )

    # ---- Synthetic data ---------------------------------------------- #
    N_CLIENT = demo_config.num_devices
    SAMPLES_PER_CLIENT = 30
    TOTAL = N_CLIENT * SAMPLES_PER_CLIENT
    input_shape = (INPUT_DIM,)

    train_ds = _generate_synthetic_data(TOTAL, input_shape, seed=10)
    val_ds   = _generate_synthetic_data(100, input_shape, seed=20)
    test_ds  = _generate_synthetic_data(120, input_shape, seed=30)
    proxy_ds = _generate_proxy_data(80, input_shape, seed=40)
    sup_ds   = _generate_synthetic_data(60, input_shape, seed=50)

    client_data = partition_data_iid(train_ds, N_CLIENT, seed=42)

    # ---- Build cycle ------------------------------------------------- #
    cycle = FederatedLearningCycle(demo_config)
    cycle.global_model = demo_model         # skip load_model for demo
    cycle.create_clients(client_data)
    cycle.setup_components()

    # ---- Run --------------------------------------------------------- #
    history = cycle.run(
        server_val_data=val_ds,
        test_data=test_ds,
        proxy_data=proxy_ds,
        supervised_data=sup_ds,
    )

    # ---- Show history ------------------------------------------------ #
    print("\nRound-by-round accuracy:")
    for rnd, acc in zip(history["round"], history["global_accuracy"]):
        print(f"  Round {rnd}: {acc:.4f}")

    # ---- TF Lite check ----------------------------------------------- #
    tflite_path = demo_config.tflite_output_path
    if Path(tflite_path).exists():
        size_kb = Path(tflite_path).stat().st_size / 1024
        print(f"\nTF Lite model: {tflite_path} ({size_kb:.1f} KB)")
        # Clean up demo artifacts
        Path(tflite_path).unlink(missing_ok=True)
        q_path = tflite_path.replace(".tflite", "_quantised.tflite")
        Path(q_path).unlink(missing_ok=True)

    print("\nDone.")


Writing federated_learning_cycle.py


### TFF Wrapper — Federated Dataset Management


In [13]:
%%writefile tff_data_utils.py
"""
TFF Data Utilities — Federated Dataset Management
===================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Bridges existing ``tf.data.Dataset`` client partitions into
TensorFlow Federated (TFF) compatible data structures.

Provides
--------
* ``TFFDataManager``  — element specs, federated-data creation,
  optional wrapping into ``tff.simulation.datasets.ClientData``.
* ``partition_data_iid_tff``  — IID partition helper.
* ``generate_synthetic_data`` / ``generate_proxy_data``  — quick
  generators for smoke-testing.

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` for the exact compatible stack.
Recommended runtime: **Google Colab** (TFF pre-installed).
"""

from __future__ import annotations

import logging
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  Guard                                                                  #
# ====================================================================== #

def _require_tff() -> None:
    """Raise a clear error if neither the Flower adapter nor TFF is available."""
    if not TFF_AVAILABLE:
        raise RuntimeError(
            "Neither the Flower adapter (flwr_adapter) nor TensorFlow Federated\n"
            "is available in this environment.\n"
            "Install Flower with:  pip install flwr\n"
            "The flwr_adapter module provides a drop-in replacement for TFF.\n"
            "Alternatively, install TFF:  pip install tensorflow-federated==0.48.0"
        )


# ====================================================================== #
#  1.  TFF DATA MANAGER                                                   #
# ====================================================================== #

class TFFDataManager:
    """
    Manages federated data for TFF integration.

    Converts per-client ``tf.data.Dataset`` partitions into the formats
    expected by ``tff.learning.algorithms`` and provides the
    ``element_spec`` / ``input_spec`` required by
    ``tff.learning.models.from_keras_model()``.

    Parameters
    ----------
    input_shape : tuple[int, ...]
        Spatial input shape *without* the batch dimension,
        e.g. ``(224, 224, 3)`` for the EfficientNet model.
    """

    def __init__(self, input_shape: Tuple[int, ...]) -> None:
        self.input_shape = input_shape

    # ------------------------------------------------------------------ #
    #  Spec helpers                                                       #
    # ------------------------------------------------------------------ #

    def get_element_spec(self) -> Tuple[tf.TensorSpec, tf.TensorSpec]:
        """
        Return the **batched** element spec for ``from_keras_model()``.

        Matches datasets that yield ``(images, labels)`` with a leading
        ``None`` batch dimension.
        """
        return (
            tf.TensorSpec(shape=(None, *self.input_shape), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.float32),
        )

    def get_unbatched_spec(self) -> Tuple[tf.TensorSpec, tf.TensorSpec]:
        """Per-example spec (no batch dim) — useful for type annotations."""
        return (
            tf.TensorSpec(shape=self.input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        )

    # ------------------------------------------------------------------ #
    #  Federated data creation (for process.next())                       #
    # ------------------------------------------------------------------ #

    def make_federated_data(
        self,
        client_datasets: Dict[str, tf.data.Dataset],
        selected_ids: List[str],
        batch_size: int = 32,
        local_epochs: int = 1,
        shuffle_buffer: int = 1000,
    ) -> List[tf.data.Dataset]:
        """
        Create a **list of batched datasets** for TFF's
        ``process.next(state, federated_data)``.

        Parameters
        ----------
        client_datasets : dict[str, tf.data.Dataset]
            Pre-partitioned datasets keyed by client ID.
        selected_ids : list[str]
            Client IDs chosen for this round (from Part 1).
        batch_size : int
            Batch size for local training.
        local_epochs : int
            Number of local training epochs — implemented via
            ``dataset.repeat(local_epochs)`` which is the standard
            TFF pattern.
        shuffle_buffer : int
            Buffer size for per-epoch shuffling.

        Returns
        -------
        list[tf.data.Dataset]
            One **batched** dataset per selected client.
        """
        federated: List[tf.data.Dataset] = []
        for cid in selected_ids:
            if cid not in client_datasets:
                logger.warning("Client %s has no dataset — skipping.", cid)
                continue
            ds = (
                client_datasets[cid]
                .repeat(local_epochs)
                .shuffle(buffer_size=shuffle_buffer)
                .batch(batch_size)
                .prefetch(tf.data.AUTOTUNE)
            )
            federated.append(ds)
        return federated

    # ------------------------------------------------------------------ #
    #  TFF ClientData wrapper (optional — for TFF simulation tools)       #
    # ------------------------------------------------------------------ #

    def create_tff_client_data(
        self,
        client_datasets: Dict[str, tf.data.Dataset],
    ):
        """
        Wrap per-client datasets into ``tff.simulation.datasets.ClientData``
        for advanced TFF simulation tools (e.g. ``ClientData.preprocess``,
        sampling utilities, etc.).

        Returns
        -------
        tff.simulation.datasets.ClientData
        """
        _require_tff()

        client_ids = sorted(client_datasets.keys())
        local_ref = client_datasets  # captured by closure

        def create_dataset_fn(client_id):
            cid = (
                client_id.numpy().decode("utf-8")
                if isinstance(client_id, tf.Tensor)
                else client_id
            )
            return local_ref[cid]

        return tff.simulation.datasets.ClientData.from_clients_and_tf_fn(
            client_ids=client_ids,
            serializable_dataset_fn=create_dataset_fn,
        )

    # ------------------------------------------------------------------ #
    #  Preprocessing pipeline                                             #
    # ------------------------------------------------------------------ #

    @staticmethod
    def preprocess_dataset(
        dataset: tf.data.Dataset,
        batch_size: int = 32,
        local_epochs: int = 1,
        shuffle_buffer: int = 1000,
    ) -> tf.data.Dataset:
        """
        Standard preprocessing pipeline applied to each client dataset
        before passing to TFF.
        """
        return (
            dataset
            .repeat(local_epochs)
            .shuffle(buffer_size=shuffle_buffer)
            .batch(batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )


# ====================================================================== #
#  2.  PARTITIONING HELPERS                                               #
# ====================================================================== #

def partition_data_iid_tff(
    full_dataset: tf.data.Dataset,
    num_clients: int,
    seed: int = 42,
) -> Dict[str, tf.data.Dataset]:
    """
    IID partition: shuffle the dataset and split evenly across clients.

    Each shard is a ``tf.data.Dataset`` yielding ``(image, label)`` pairs.

    Parameters
    ----------
    full_dataset : tf.data.Dataset
        The complete labelled dataset.
    num_clients : int
        Number of federated client partitions.
    seed : int
        Random seed for reproducible shuffling.

    Returns
    -------
    dict[str, tf.data.Dataset]
        ``{client_id: local_dataset}``
    """
    all_data = list(full_dataset.shuffle(buffer_size=10_000, seed=seed))
    total = len(all_data)
    shard_size = max(1, total // num_clients)

    partitions: Dict[str, tf.data.Dataset] = {}
    for i in range(num_clients):
        cid = f"client_{i:03d}"
        start = i * shard_size
        end = min(start + shard_size, total)
        if start >= total:
            start = start % total
            end = start + shard_size

        shard_x = [elem[0].numpy() for elem in all_data[start:end]]
        shard_y = [elem[1].numpy() for elem in all_data[start:end]]

        if not shard_x:
            shard_x = [all_data[0][0].numpy()]
            shard_y = [all_data[0][1].numpy()]

        partitions[cid] = tf.data.Dataset.from_tensor_slices(
            (np.stack(shard_x), np.array(shard_y))
        )
    return partitions


# ====================================================================== #
#  3.  SYNTHETIC DATA GENERATORS  (for smoke-testing)                     #
# ====================================================================== #

def generate_synthetic_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Synthetic labelled dataset ``(image, label)`` for smoke-testing.

    Replace with real FF++ c23 data loaders in production.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    y = rng.randint(0, 2, size=(num_samples,)).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((x, y))


def generate_proxy_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """Unlabelled proxy data ``(image,)`` for knowledge distillation."""
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    return tf.data.Dataset.from_tensor_slices(x)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Data Utilities — Demo  ===\n")

    INPUT_SHAPE = (16,)          # tiny for fast demo
    NUM_CLIENTS = 8
    SAMPLES = NUM_CLIENTS * 30   # 30 per client

    # ---- 1. Synthetic dataset ---------------------------------------- #
    full_ds = generate_synthetic_data(SAMPLES, INPUT_SHAPE, seed=10)
    print(f"Full dataset: {SAMPLES} samples, shape {INPUT_SHAPE}")

    # ---- 2. Partition ------------------------------------------------ #
    client_data = partition_data_iid_tff(full_ds, NUM_CLIENTS)
    for cid, ds in sorted(client_data.items()):
        n = sum(1 for _ in ds)
        print(f"  {cid}: {n} samples")

    # ---- 3. TFF data manager ---------------------------------------- #
    dm = TFFDataManager(input_shape=INPUT_SHAPE)

    print(f"\nElement spec (batched): {dm.get_element_spec()}")
    print(f"Unbatched spec:        {dm.get_unbatched_spec()}")

    # ---- 4. Make federated data for a round -------------------------- #
    selected = ["client_001", "client_003", "client_005"]
    fed_data = dm.make_federated_data(
        client_data, selected, batch_size=16, local_epochs=2,
    )
    print(f"\nFederated data for {selected}: {len(fed_data)} datasets")
    for i, ds in enumerate(fed_data):
        n_batches = sum(1 for _ in ds)
        print(f"  Client {selected[i]}: {n_batches} batches (2 epochs)")

    # ---- 5. TFF ClientData (requires TFF) ---------------------------- #
    if TFF_AVAILABLE:
        tff_cd = dm.create_tff_client_data(client_data)
        print(f"\nTFF ClientData: {len(tff_cd.client_ids)} clients")
        print(f"  Client IDs: {tff_cd.client_ids[:5]} ...")
    else:
        print(
            "\n⚠  TFF not installed — skipping ClientData creation.\n"
            "   Install: pip install tensorflow-federated==0.48.0\n"
            "   See requirements_tff.txt for full compatible stack.\n"
            "   Recommended runtime: Google Colab."
        )



    # ---- 6. Proxy data ----------------------------------------------- #    print("\nDone.")

    proxy = generate_proxy_data(50, INPUT_SHAPE, seed=20)
    print(f"\nProxy data: 50 unlabelled samples, shape {INPUT_SHAPE}")

Writing tff_data_utils.py


### TFF Wrapper — Model Wrapping & Learning Process


In [23]:
%%writefile tff_learning_process.py
"""
TFF Learning Process — Model Wrapping & Federated Training
===========================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Wraps the EfficientNet Keras model into TFF's learning framework and
builds a customised Federated Averaging learning process.

Provides
--------
* ``TFFModelFactory``    — creates the ``model_fn`` callable required by
  ``tff.learning.algorithms.build_weighted_fed_avg()``.
* ``build_tff_learning_process``  — builds the TFF ``LearningProcess``.
* ``TFFRoundExecutor``   — executes TFF rounds and converts weights
  between TFF ``ModelWeights`` and Keras ``model.get_weights()`` format.
* Weight-conversion utilities: ``tff_weights_to_keras``,
  ``keras_weights_to_tff``.

Architecture
------------
TFF handles the federated computation (model broadcast → client-side
local training → data-weighted aggregation a.k.a. FedAvg).  Our custom
enhancements (contribution weighting, distillation, reputation) operate
as a **post-aggregation refinement** in the outer Python loop.

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` and ``tff_data_utils.py`` for details.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

from tff_data_utils import _require_tff

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  MODEL FACTORY                                                      #
# ====================================================================== #

class TFFModelFactory:
    """
    Creates the ``model_fn`` callable that TFF's learning algorithms
    require.

    Each call to ``model_fn()`` must return a **fresh** TFF-wrapped
    Keras model instance with the same architecture.  TFF traces the
    computation graph on first call and reuses it, so deterministic
    architecture is critical.

    Parameters
    ----------
    keras_model : tf.keras.Model
        Reference Keras model whose architecture will be cloned.
        Must **not** be compiled — TFF handles compilation internally.
    input_spec : tuple[tf.TensorSpec, tf.TensorSpec]
        Batched element spec ``(x_spec, y_spec)`` matching the dataset
        structure.  Obtain via ``TFFDataManager.get_element_spec()``.
    loss : tf.keras.losses.Loss | None
        Loss function; defaults to ``BinaryCrossentropy(from_logits=False)``.
    metrics : list[tf.keras.metrics.Metric] | None
        Metrics; defaults to ``[BinaryAccuracy()]``.
    """

    def __init__(
        self,
        keras_model: tf.keras.Model,
        input_spec: Tuple[tf.TensorSpec, tf.TensorSpec],
        loss: Optional[tf.keras.losses.Loss] = None,
        metrics: Optional[list] = None,
    ) -> None:
        self._ref_model = keras_model
        self._input_spec = input_spec
        self._loss = loss or tf.keras.losses.BinaryCrossentropy()
        self._metrics = metrics or [tf.keras.metrics.BinaryAccuracy()]

    # ------------------------------------------------------------------ #
    #  model_fn builder                                                   #
    # ------------------------------------------------------------------ #

    def create_model_fn(self) -> Callable[[], Any]:
        """
        Return a **no-args callable** compatible with
        ``tff.learning.algorithms.build_weighted_fed_avg(model_fn=...)``.

        The closure captures the reference model, spec, loss, and metrics
        so that every invocation produces an architecturally identical
        (but freshly initialised) TFF model.
        """
        _require_tff()

        ref = self._ref_model
        spec = self._input_spec
        loss = self._loss
        metrics_list = self._metrics

        def model_fn():
            # Clone architecture (fresh random weights — TFF will inject
            # the server weights before each client round).
            keras_clone = tf.keras.models.clone_model(ref)
            keras_clone.build(ref.input_shape)
            return tff.learning.models.from_keras_model(
                keras_model=keras_clone,
                input_spec=spec,
                loss=loss,
                metrics=metrics_list,
            )

        return model_fn


# ====================================================================== #
#  2.  WEIGHT CONVERSION  (TFF ↔ Keras)                                  #
# ====================================================================== #

def tff_weights_to_keras(
    model_weights,
    keras_model: tf.keras.Model,
) -> None:
    """
    Copy TFF ``ModelWeights`` into a Keras model's variables.

    Uses TFF's built-in ``assign_weights_to`` when available; falls back
    to manual assignment otherwise.

    Parameters
    ----------
    model_weights : tff.learning.models.ModelWeights
        ``ModelWeights(trainable=[...], non_trainable=[...])``.
    keras_model : tf.keras.Model
        Target Keras model — must share the same architecture.
    """
    _require_tff()

    try:
        # Preferred: TFF's built-in helper
        model_weights.assign_weights_to(keras_model)
    except AttributeError:
        # Fallback: manual assignment
        trainable = list(model_weights.trainable)
        non_trainable = list(model_weights.non_trainable)
        for var, val in zip(keras_model.trainable_variables, trainable):
            var.assign(val)
        for var, val in zip(keras_model.non_trainable_variables, non_trainable):
            var.assign(val)
    logger.debug("TFF → Keras weight transfer complete.")


def keras_weights_to_tff(
    keras_model: tf.keras.Model,
):
    """
    Convert Keras model weights to TFF ``ModelWeights``.

    Returns
    -------
    tff.learning.models.ModelWeights
    """
    _require_tff()

    trainable = [v.numpy() for v in keras_model.trainable_variables]
    non_trainable = [v.numpy() for v in keras_model.non_trainable_variables]
    return tff.learning.models.ModelWeights(
        trainable=trainable,
        non_trainable=non_trainable,
    )


# ====================================================================== #
#  3.  LEARNING-PROCESS BUILDER                                           #
# ====================================================================== #

@dataclass
class TFFProcessConfig:
    """Hyper-parameters for the TFF weighted-FedAvg process."""
    client_lr: float = 1e-4
    server_lr: float = 1.0
    client_optimizer: str = "adam"     # "adam" | "sgd"
    server_optimizer: str = "sgd"     # typically SGD with lr=1.0


def build_tff_learning_process(
    model_fn: Callable[[], Any],
    config: Optional[TFFProcessConfig] = None,
):
    """
    Build a TFF ``LearningProcess`` using weighted Federated Averaging.

    This wraps ``tff.learning.algorithms.build_weighted_fed_avg`` and
    returns a process with the following methods:

    * ``initialize()`` → initial server state
    * ``next(state, federated_data)`` → ``LearningProcessOutput``
    * ``get_model_weights(state)`` → ``ModelWeights``
    * ``set_model_weights(state, weights)`` → updated state

    Parameters
    ----------
    model_fn : callable
        No-args callable that returns a ``tff.learning.models.VariableModel``.
        Obtain from ``TFFModelFactory.create_model_fn()``.
    config : TFFProcessConfig | None
        Optimiser and learning-rate settings.

    Returns
    -------
    tff.learning.templates.LearningProcess
    """
    _require_tff()
    config = config or TFFProcessConfig()

    def client_optimizer_fn():
        if config.client_optimizer == "adam":
            return tf.keras.optimizers.Adam(config.client_lr)
        return tf.keras.optimizers.SGD(config.client_lr)

    def server_optimizer_fn():
        if config.server_optimizer == "adam":
            return tf.keras.optimizers.Adam(config.server_lr)
        return tf.keras.optimizers.SGD(config.server_lr)

    process = tff.learning.algorithms.build_weighted_fed_avg(
        model_fn=model_fn,
        client_optimizer_fn=client_optimizer_fn,
        server_optimizer_fn=server_optimizer_fn,
    )

    logger.info(
        "TFF LearningProcess built — client_opt=%s(lr=%g), "
        "server_opt=%s(lr=%g).",
        config.client_optimizer, config.client_lr,
        config.server_optimizer, config.server_lr,
    )
    return process


# ====================================================================== #
#  4.  ROUND EXECUTOR                                                     #
# ====================================================================== #

class TFFRoundExecutor:
    """
    Thin wrapper around a TFF ``LearningProcess`` that handles state
    management and weight conversion.

    Typical workflow
    ----------------
    >>> executor = TFFRoundExecutor(process, keras_model)
    >>> executor.initialize()
    >>> executor.inject_pretrained_weights()     # start from .h5 model
    >>> for t in range(num_rounds):
    ...     metrics = executor.execute_round(federated_data)
    ...     keras_weights = executor.get_keras_weights()
    ...     # ... apply enhancements ...
    ...     executor.set_keras_weights(enhanced_weights)

    Parameters
    ----------
    process : tff.learning.templates.LearningProcess
        TFF learning process (from ``build_tff_learning_process``).
    keras_model : tf.keras.Model
        Reference Keras model — used for weight conversion only.
    """

    def __init__(
        self,
        process,                        # tff.learning.templates.LearningProcess
        keras_model: tf.keras.Model,
    ) -> None:
        _require_tff()
        self.process = process
        self.keras_model = keras_model
        self.state = None
        self._round_count = 0

    # ------------------------------------------------------------------ #
    #  Initialisation                                                     #
    # ------------------------------------------------------------------ #

    def initialize(self) -> None:
        """
        Call TFF's ``process.initialize()`` to create the initial server
        state (with random model weights).
        """
        self.state = self.process.initialize()
        self._round_count = 0
        logger.info("TFF process initialised (random server weights).")

    def inject_pretrained_weights(self) -> None:
        """
        Inject the weights from ``self.keras_model`` into the TFF server
        state.  Call this after ``initialize()`` to start federated
        training from a pre-trained checkpoint (e.g. EfficientNet).
        """
        assert self.state is not None, "Call initialize() first."
        tff_weights = keras_weights_to_tff(self.keras_model)
        self.state = self.process.set_model_weights(self.state, tff_weights)
        logger.info("Pre-trained Keras weights injected into TFF state.")

    # ------------------------------------------------------------------ #
    #  Round execution                                                    #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        federated_data: List[tf.data.Dataset],
    ) -> Dict[str, Any]:
        """
        Execute one TFF federated round (broadcast → local training →
        FedAvg aggregation).

        Parameters
        ----------
        federated_data : list[tf.data.Dataset]
            Batched datasets for the selected clients.  Obtain from
            ``TFFDataManager.make_federated_data()``.

        Returns
        -------
        metrics : dict
            TFF round metrics (client loss, accuracy, num_examples, …).
        """
        assert self.state is not None, "Call initialize() first."

        result = self.process.next(self.state, federated_data)
        self.state = result.state
        self._round_count += 1

        # Extract metrics — TFF returns nested OrderedDicts
        metrics = _flatten_tff_metrics(result.metrics)
        logger.info(
            "TFF round %d complete — %s",
            self._round_count, _format_metrics(metrics),
        )
        return metrics

    # ------------------------------------------------------------------ #
    #  Weight access                                                      #
    # ------------------------------------------------------------------ #

    def get_keras_weights(self) -> List[np.ndarray]:
        """
        Extract model weights from the TFF state and set them on the
        Keras reference model.

        Returns the weights as a list of numpy arrays (same format
        as ``keras_model.get_weights()``).
        """
        model_weights = self.process.get_model_weights(self.state)
        tff_weights_to_keras(model_weights, self.keras_model)
        return self.keras_model.get_weights()

    def set_keras_weights(self, weights: List[np.ndarray]) -> None:
        """
        Inject (possibly enhanced) Keras weights back into the TFF
        server state so the next round broadcasts the updated model.
        """
        self.keras_model.set_weights(weights)
        tff_weights = keras_weights_to_tff(self.keras_model)
        self.state = self.process.set_model_weights(self.state, tff_weights)
        logger.debug("Enhanced weights injected into TFF state.")

    def get_tff_model_weights(self):
        """Return the raw TFF ``ModelWeights`` from the current state."""
        return self.process.get_model_weights(self.state)


# ====================================================================== #
#  5.  METRIC HELPERS                                                     #
# ====================================================================== #

def _flatten_tff_metrics(metrics_struct) -> Dict[str, float]:
    """
    Flatten TFF's nested ``OrderedDict`` metrics into a simple dict.

    TFF returns metrics like::

        OrderedDict([
            ('distributor', OrderedDict()),
            ('client_work', OrderedDict([
                ('train', OrderedDict([('loss', 0.693), ...])),
            ])),
            ...
        ])

    This flattens all leaf values into ``{'client_work/train/loss': 0.693}``.
    """
    flat: Dict[str, float] = {}

    def _walk(obj, prefix: str = "") -> None:
        if isinstance(obj, dict):
            for k, v in obj.items():
                _walk(v, f"{prefix}{k}/")
        elif hasattr(obj, "_asdict"):
            _walk(obj._asdict(), prefix)
        elif hasattr(obj, "items"):
            for k, v in obj.items():
                _walk(v, f"{prefix}{k}/")
        else:
            key = prefix.rstrip("/")
            try:
                flat[key] = float(obj)
            except (TypeError, ValueError):
                flat[key] = str(obj)

    try:
        _walk(metrics_struct)
    except Exception:
        flat["raw"] = str(metrics_struct)
    return flat


def _format_metrics(metrics: Dict[str, float], max_items: int = 5) -> str:
    """Compact single-line metrics string for logging."""
    items = list(metrics.items())[:max_items]
    return ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                     for k, v in items)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Learning Process — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny reference model ----------------------------- #
    INPUT_DIM = 16
    ref_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    # Do NOT compile — TFF handles compilation internally
    print(f"Reference model: {ref_model.count_params()} params, "
          f"input shape {ref_model.input_shape}")

    # ---- 2. Element spec --------------------------------------------- #
    from tff_data_utils import TFFDataManager, generate_synthetic_data, partition_data_iid_tff

    dm = TFFDataManager(input_shape=(INPUT_DIM,))
    input_spec = dm.get_element_spec()
    print(f"Input spec: {input_spec}")

    # ---- 3. Model factory -------------------------------------------- #
    factory = TFFModelFactory(
        keras_model=ref_model,
        input_spec=input_spec,
    )
    print("TFFModelFactory created.")

    # ---- 4. TFF-specific tests --------------------------------------- #
    if TFF_AVAILABLE:
        print("\n--- TFF available — running full demo ---\n")

        model_fn = factory.create_model_fn()
        print(f"model_fn created: {model_fn}")

        # Build learning process
        process = build_tff_learning_process(
            model_fn=model_fn,
            config=TFFProcessConfig(client_lr=1e-3, server_lr=1.0),
        )
        print("Learning process built.")

        # Create round executor
        executor = TFFRoundExecutor(process, ref_model)
        executor.initialize()
        executor.inject_pretrained_weights()
        print("Executor initialised with pre-trained weights.")

        # Synthesise federated data
        full_ds = generate_synthetic_data(240, (INPUT_DIM,), seed=10)
        client_data = partition_data_iid_tff(full_ds, 8)
        selected = ["client_001", "client_003", "client_005"]
        fed_data = dm.make_federated_data(
            client_data, selected, batch_size=16, local_epochs=2,
        )

        # Run 3 TFF rounds
        for rnd in range(1, 4):
            metrics = executor.execute_round(fed_data)
            keras_w = executor.get_keras_weights()
            print(f"  Round {rnd}: {len(keras_w)} weight arrays, "
                  f"metrics keys: {list(metrics.keys())[:4]}")

            # Simulate enhancement: perturb weights slightly
            enhanced = [w + np.random.randn(*w.shape).astype(np.float32) * 0.001
                        for w in keras_w]
            executor.set_keras_weights(enhanced)

        print("\nTFF demo complete.")

    else:
        print(
            "\n⚠  TFF not installed — running structural checks only.\n"
            "   Install: pip install tensorflow-federated==0.48.0\n"
            "   See requirements_tff.txt for the full stack.\n"
        )

        # Structural checks
        print("✓ TFFModelFactory instantiable (model_fn creation requires TFF)")
        print("✓ TFFProcessConfig:", TFFProcessConfig())

        try:
            _ = factory.create_model_fn()
        except RuntimeError as e:
            print(f"✓ model_fn correctly raises: {e.__class__.__name__}")

        # Weight conversion type check (without TFF, just show shapes)
        w = ref_model.get_weights()
        tr = [v.numpy() for v in ref_model.trainable_variables]
        ntr = [v.numpy() for v in ref_model.non_trainable_variables]
        print(f"✓ Keras → TFF mapping: {len(tr)} trainable, "

              f"{len(ntr)} non-trainable arrays")

    print("\nDone.")

    print("\nStructural checks passed.")

Overwriting tff_learning_process.py


### TFF Wrapper — Main TFF-Based Orchestrator


In [26]:
%%writefile tff_federated_cycle.py
"""
TFF Federated Learning Cycle — Main Orchestrator
==================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Integrates **TensorFlow Federated** with all five enhancement modules
into one end-to-end pipeline:

 1. **Enhanced Client Selection**   (``enhanced_client_selection.py``)
 2. **Update Validation & Weighing** (``update_validation.py``)
 3. **Knowledge Distillation**      (``knowledge_distillation.py``)
 4. **Client Reputation Ledger**    (``client_reputation_ledger.py``)
 5. **Evaluation Metrics**          (``evaluation_metrics.py``)

Architecture
------------
TFF handles the **core federated computation**: model broadcasting,
client-side local training, and data-weighted Federated Averaging.
Our thesis enhancements operate as a **post-aggregation refinement
layer** that runs in the outer Python loop after each TFF round.

Per-round pipeline
~~~~~~~~~~~~~~~~~~
1. **Client selection** (Part 1) — multi-criteria scoring to choose
   which clients participate.  Selected client IDs determine which
   data is passed to TFF's ``process.next()``.
2. **TFF round** — ``process.next(state, federated_data)`` performs
   broadcasting, local training (``local_epochs`` via dataset repeat),
   and weighted FedAvg aggregation.
3. **Per-client analysis** — local training is re-run *outside TFF* on
   the selected clients so that per-client model weights are available
   for contribution scoring, distillation, and validation.
4. **Update validation & contribution-weighted aggregation** (Part 2) —
   re-aggregate using contribution weights ``c_i`` (which account for
   validation gain, similarity, data volume, and reputation).  This
   replaces TFF's data-volume-only FedAvg result.
5. **Knowledge distillation** (Part 3) — refine the server model by
   distilling knowledge from the contribution-weighted client ensemble.
6. **Reputation ledger update** (Part 4) — feed validation gains and
   contribution scores into the persistent ledger.
7. **Evaluation & reporting** (Part 5) — periodic full evaluation with
   JSON + text reports.
8. **Inject enhanced weights** back into the TFF server state for the
   next round.

Comparison mode
~~~~~~~~~~~~~~~
When ``enable_comparison=True`` the cycle logs **both** TFF's standard
FedAvg accuracy and our enhanced accuracy each round, providing a
direct side-by-side comparison for the thesis.

Configuration
~~~~~~~~~~~~~
- **Model**: ``effnet_ffpp_small_data.h5`` (EfficientNet, binary)
- **Devices**: 100 simulated clients
- **Local epochs**: 5 per round
- **Global rounds**: 50
- **Frameworks**: TensorFlow Federated + TF Lite

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` for the exact compatible stack.
Recommended runtime: **Google Colab**.
"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

# ---------- Existing modules  (Parts 1–5) ----------------------------- #
from enhanced_client_selection import (          # Part 1
    ClientMetrics,
    FederatedClient,
    ReputationLedger,
    SelectionWeights,
    EnhancedClientSelector,
)
from update_validation import (                  # Part 2
    ContributionWeights,
    ClippingConfig,
    ClientUpdateRecord,
    UpdateValidator,
)
from knowledge_distillation import (             # Part 3
    DistillationConfig,
    run_distillation_round,
)
from client_reputation_ledger import (           # Part 4
    ReputationConfig,
    ClientReputationLedger,
    update_ledger_from_records,
)
from evaluation_metrics import (                 # Part 5
    FederatedModelEvaluator,
    evaluate_and_report,
)

# ---------- TFF wrappers ---------------------------------------------- #
from tff_data_utils import (
    TFFDataManager,
    _require_tff,
    partition_data_iid_tff,
    generate_synthetic_data,
    generate_proxy_data,
)
from tff_learning_process import (
    TFFModelFactory,
    TFFProcessConfig,
    build_tff_learning_process,
    TFFRoundExecutor,
    keras_weights_to_tff,
    tff_weights_to_keras,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class TFFCycleConfig:
    """
    Central configuration for the TFF-based Federated Learning cycle.
    """
    # -- Core FL settings ---------------------------------------------- #
    model_path: str = "effnet_ffpp_small_data.h5"
    num_devices: int = 100
    local_epochs: int = 5
    global_rounds: int = 50
    clients_per_round: int = 15
    local_batch_size: int = 32
    local_lr: float = 1e-4
    server_lr: float = 1.0
    eval_every: int = 5

    # -- TFF process settings ------------------------------------------ #
    client_optimizer: str = "adam"
    server_optimizer: str = "sgd"

    # -- Comparison mode ----------------------------------------------- #
    enable_comparison: bool = True     # Log TFF FedAvg vs enhanced

    # -- Distillation (Part 3) ---------------------------------------- #
    enable_distillation: bool = True
    distillation_config: DistillationConfig = field(
        default_factory=lambda: DistillationConfig(
            temperature=3.0,
            lam=0.7,
            epochs=3,
            batch_size=32,
            learning_rate=1e-4,
        )
    )

    # -- Client selection (Part 1) ------------------------------------- #
    selection_weights: SelectionWeights = field(
        default_factory=lambda: SelectionWeights(
            w_v=0.30,
            w_d=0.20,
            w_l=0.10,
            w_r=0.25,
            w_s=0.15,
        )
    )

    # -- Update validation (Part 2) ----------------------------------- #
    contribution_weights: ContributionWeights = field(
        default_factory=lambda: ContributionWeights(
            alpha=0.35,
            beta=0.20,
            gamma=0.20,
            delta=0.25,
        )
    )
    clipping_config: ClippingConfig = field(
        default_factory=lambda: ClippingConfig(
            clip_threshold=10.0,
            clip_value=5.0,
        )
    )
    harmful_threshold: float = 0.02

    # -- Reputation (Part 4) ------------------------------------------ #
    reputation_config: ReputationConfig = field(
        default_factory=lambda: ReputationConfig(
            theta=0.0,
            gamma=0.10,
            decay_rate=0.99,
            floor=0.05,
            ceiling=1.0,
            initial_reputation=0.50,
            penalty_factor=0.05,
        )
    )

    # -- Evaluation & output (Part 5) --------------------------------- #
    reports_dir: str = "reports"
    tflite_output_path: str = "effnet_global_tff_final.tflite"
    input_shape: Tuple[int, ...] = (224, 224, 3)


# ====================================================================== #
#  2.  TF LITE CONVERSION                                                 #
# ====================================================================== #

def convert_to_tflite(
    model: tf.keras.Model,
    output_path: str,
    quantise: bool = False,
) -> str:
    """Convert a Keras model to TF Lite format."""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantise:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    Path(output_path).write_bytes(tflite_bytes)
    size_mb = len(tflite_bytes) / (1024 * 1024)
    logger.info(
        "TF Lite model saved → %s  (%.2f MB, quantised=%s)",
        output_path, size_mb, quantise,
    )
    return output_path


# ====================================================================== #
#  3.  TFF FEDERATED LEARNING CYCLE                                       #
# ====================================================================== #

class TFFFederatedLearningCycle:
    """
    End-to-end Federated Learning cycle using TFF,
    integrating all five enhancement modules.

    See module docstring for the detailed per-round pipeline.
    """

    def __init__(self, config: Optional[TFFCycleConfig] = None) -> None:
        self.config = config or TFFCycleConfig()
        self.global_model: Optional[tf.keras.Model] = None
        self.clients: List[FederatedClient] = []
        self.client_datasets: Dict[str, tf.data.Dataset] = {}

        # TFF components
        self.data_manager: Optional[TFFDataManager] = None
        self.tff_executor: Optional[TFFRoundExecutor] = None

        # Enhancement components (Parts 1–5)
        self.reputation_ledger: Optional[ClientReputationLedger] = None
        self.basic_ledger: Optional[ReputationLedger] = None
        self.selector: Optional[EnhancedClientSelector] = None
        self.validator: Optional[UpdateValidator] = None
        self.evaluator: Optional[FederatedModelEvaluator] = None

        # History
        self.history: Dict[str, list] = {
            "round": [],
            "tff_fedavg_accuracy": [],
            "enhanced_accuracy": [],
            "selected_clients": [],
            "num_accepted": [],
            "num_rejected": [],
            "distillation_loss": [],
        }

    # ------------------------------------------------------------------ #
    #  Initialisation                                                     #
    # ------------------------------------------------------------------ #

    def load_global_model(self) -> tf.keras.Model:
        """Load the pre-trained EfficientNet model."""
        cfg = self.config
        logger.info("Loading global model from %s …", cfg.model_path)
        # Register EfficientNet preprocessing so Keras can deserialize the .h5
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        _custom = {"preprocess_input": _effnet_preprocess}
        model = tf.keras.models.load_model(
            cfg.model_path, custom_objects=_custom, compile=False,
        )
        model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        logger.info(
            "Global model loaded — %s params, input shape %s",
            f"{model.count_params():,}", model.input_shape,
        )
        self.global_model = model
        return model

    def create_clients(
        self,
        client_data: Dict[str, tf.data.Dataset],
    ) -> List[FederatedClient]:
        """Create ``FederatedClient`` objects and store the dataset dict."""
        rng = np.random.RandomState(42)
        clients: List[FederatedClient] = []

        for cid, local_ds in client_data.items():
            n_samples = sum(1 for _ in local_ds)
            metrics = ClientMetrics(
                local_validation_metric=float(rng.uniform(0.4, 0.9)),
                data_volume=n_samples,
                inference_latency=float(rng.uniform(0.01, 0.15)),
                last_selected_round=0,
            )
            clients.append(FederatedClient(
                client_id=cid,
                local_data=local_ds,
                metrics=metrics,
            ))

        logger.info("Created %d federated clients.", len(clients))
        self.clients = clients
        self.client_datasets = client_data
        return clients

    def setup_tff_process(self) -> None:
        """
        Build the TFF learning process and initialise the round executor.
        """
        _require_tff()
        cfg = self.config
        assert self.global_model is not None, "Call load_global_model() first."

        # Data manager
        input_shape = self.global_model.input_shape[1:]
        cfg.input_shape = input_shape
        self.data_manager = TFFDataManager(input_shape=input_shape)
        element_spec = self.data_manager.get_element_spec()

        # Model factory → model_fn
        factory = TFFModelFactory(
            keras_model=self.global_model,
            input_spec=element_spec,
        )
        model_fn = factory.create_model_fn()

        # Build TFF learning process
        process = build_tff_learning_process(
            model_fn=model_fn,
            config=TFFProcessConfig(
                client_lr=cfg.local_lr,
                server_lr=cfg.server_lr,
                client_optimizer=cfg.client_optimizer,
                server_optimizer=cfg.server_optimizer,
            ),
        )

        # Round executor
        self.tff_executor = TFFRoundExecutor(process, self.global_model)
        self.tff_executor.initialize()
        self.tff_executor.inject_pretrained_weights()

        logger.info("TFF process initialised with pre-trained weights.")

    def setup_enhancement_modules(self) -> None:
        """Wire Parts 1–5 (same logic as federated_learning_cycle.py)."""
        cfg = self.config
        assert self.global_model is not None
        assert len(self.clients) > 0

        # Part 4: Reputation ledger
        self.reputation_ledger = ClientReputationLedger(
            config=cfg.reputation_config,
        )
        for c in self.clients:
            self.reputation_ledger.register(c.client_id)
        self.basic_ledger = self.reputation_ledger.as_basic_ledger()

        # Part 1: Client selector
        self.selector = EnhancedClientSelector(
            clients=self.clients,
            reputation_ledger=self.basic_ledger,
            weights=cfg.selection_weights,
            target_k=cfg.clients_per_round,
        )

        # Part 2: Update validator
        self.validator = UpdateValidator(
            global_model=self.global_model,
            reputation_ledger=self.basic_ledger,
            weights=cfg.contribution_weights,
            clipping=cfg.clipping_config,
            harmful_threshold=cfg.harmful_threshold,
            batch_size=cfg.local_batch_size,
        )

        # Part 5: Evaluator
        self.evaluator = FederatedModelEvaluator(
            model=self.global_model,
            model_name="effnet_global_tff",
            reports_dir=cfg.reports_dir,
        )

        logger.info("Enhancement modules (Parts 1–5) initialised.")

    # ------------------------------------------------------------------ #
    #  Local training (manual — for per-client analysis)                  #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int]:
        """
        Manual local training (outside TFF) to obtain per-client model
        weights for contribution scoring and distillation.
        """
        cfg = self.config
        local_model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            return global_weights, 0

        dataset = client.local_data.batch(cfg.local_batch_size)
        local_model.fit(dataset, epochs=cfg.local_epochs, verbose=0)
        return local_model.get_weights(), client.metrics.data_volume

    # ------------------------------------------------------------------ #
    #  Reputation sync                                                    #
    # ------------------------------------------------------------------ #

    def _sync_reputation_to_basic_ledger(self) -> None:
        """Copy extended ledger → basic ledger used by Parts 1 & 2."""
        updated_basic = self.reputation_ledger.as_basic_ledger()
        self.basic_ledger._scores = updated_basic._scores

    # ------------------------------------------------------------------ #
    #  Single round                                                       #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        current_round: int,
        server_val_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, Any]:
        """
        Execute one complete TFF + enhanced federated round.

        Returns a summary dict with both TFF FedAvg and enhanced accuracy.
        """
        cfg = self.config

        logger.info("=" * 70)
        logger.info("  TFF ROUND %d / %d", current_round, cfg.global_rounds)
        logger.info("=" * 70)

        # ── 1. Client selection  (Part 1) ──────────────────────────── #
        selected: List[FederatedClient] = self.selector.select(
            current_round=current_round,
        )
        selected_ids = [c.client_id for c in selected]
        logger.info(
            "Selected %d / %d clients: %s",
            len(selected), len(self.clients), selected_ids,
        )

        # Save pre-round global weights for analysis
        global_weights_before = self.global_model.get_weights()

        # ── 2. TFF federated round (FedAvg baseline) ──────────────── #
        federated_data = self.data_manager.make_federated_data(
            self.client_datasets,
            selected_ids,
            batch_size=cfg.local_batch_size,
            local_epochs=cfg.local_epochs,
        )
        tff_metrics = self.tff_executor.execute_round(federated_data)

        # Extract TFF-aggregated weights
        tff_aggregated_weights = self.tff_executor.get_keras_weights()

        # Quick TFF FedAvg accuracy check
        tff_acc = None
        if cfg.enable_comparison:
            tff_model_tmp = tf.keras.models.clone_model(self.global_model)
            tff_model_tmp.build(self.global_model.input_shape)
            tff_model_tmp.compile(
                optimizer="adam", loss="binary_crossentropy",
                metrics=["accuracy"],
            )
            tff_model_tmp.set_weights(tff_aggregated_weights)
            tff_result = tff_model_tmp.evaluate(
                server_val_data.batch(cfg.local_batch_size),
                verbose=0, return_dict=True,
            )
            tff_acc = tff_result.get("accuracy", 0.0)
            logger.info(
                "TFF FedAvg accuracy (round %d): %.4f",
                current_round, tff_acc,
            )

        # ── 3. Per-client analysis (manual, for Parts 2/3/4) ──────── #
        client_updates: Dict[str, List[np.ndarray]] = {}
        data_volumes: Dict[str, int] = {}

        for client in selected:
            updated_w, n = self._local_train(client, global_weights_before)
            client_updates[client.client_id] = updated_w
            data_volumes[client.client_id] = max(n, 1)
            client.metrics.last_selected_round = current_round

        # ── 4. Update validation & contribution aggregation (Part 2) ─ #
        records: List[ClientUpdateRecord] = self.validator.validate_updates(
            client_updates=client_updates,
            data_volumes=data_volumes,
            server_val_data=server_val_data,
        )
        enhanced_weights = self.validator.aggregate_weighted(
            records, global_weights_before,
        )

        num_accepted = sum(1 for r in records if not r.rejected)
        num_rejected = sum(1 for r in records if r.rejected)
        logger.info(
            "Updates: %d accepted, %d rejected out of %d.",
            num_accepted, num_rejected, len(records),
        )

        # Apply enhanced weights to global model
        self.global_model.set_weights(enhanced_weights)
        self.validator.global_model.set_weights(enhanced_weights)

        # ── 5. Knowledge distillation  (Part 3) ───────────────────── #
        distill_loss = None
        if cfg.enable_distillation and proxy_data is not None:
            contribution_weights = {
                r.client_id: r.contribution_weight
                for r in records
                if not r.rejected and r.contribution_weight > 0
            }

            if len(contribution_weights) >= 2:
                logger.info(
                    "Running knowledge distillation with %d teacher(s) …",
                    len(contribution_weights),
                )
                kd_history = run_distillation_round(
                    global_model=self.global_model,
                    client_weights={
                        cid: client_updates[cid]
                        for cid in contribution_weights
                    },
                    contribution_weights=contribution_weights,
                    proxy_data=proxy_data,
                    supervised_data=supervised_data,
                    config=cfg.distillation_config,
                )
                distill_loss = kd_history["loss_total"][-1]
                logger.info(
                    "Distillation complete — final loss %.5f", distill_loss,
                )

        # ── 6. Reputation update  (Part 4) ────────────────────────── #
        update_ledger_from_records(
            self.reputation_ledger, records, current_round,
        )
        self.validator.update_reputations(records)
        self._sync_reputation_to_basic_ledger()

        # ── 7. Enhanced accuracy check ────────────────────────────── #
        enhanced_result = self.global_model.evaluate(
            server_val_data.batch(cfg.local_batch_size),
            verbose=0, return_dict=True,
        )
        enhanced_acc = enhanced_result.get("accuracy", 0.0)

        if cfg.enable_comparison and tff_acc is not None:
            delta = enhanced_acc - tff_acc
            logger.info(
                "Round %d — TFF FedAvg: %.4f | Enhanced: %.4f | Δ=%+.4f",
                current_round, tff_acc, enhanced_acc, delta,
            )
        else:
            logger.info(
                "Round %d — Enhanced accuracy: %.4f",
                current_round, enhanced_acc,
            )

        # ── 8. Inject enhanced weights into TFF state ─────────────── #
        self.tff_executor.set_keras_weights(
            self.global_model.get_weights(),
        )

        return {
            "round": current_round,
            "selected": selected_ids,
            "tff_fedavg_accuracy": tff_acc,
            "enhanced_accuracy": enhanced_acc,
            "num_accepted": num_accepted,
            "num_rejected": num_rejected,
            "records": records,
            "distillation_loss": distill_loss,
            "tff_metrics": tff_metrics,
        }

    # ------------------------------------------------------------------ #
    #  Full cycle                                                         #
    # ------------------------------------------------------------------ #

    def run(
        self,
        server_val_data: tf.data.Dataset,
        test_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Run the full TFF Federated Learning cycle.

        Parameters
        ----------
        server_val_data : tf.data.Dataset
            Server validation set for update scoring.
        test_data : tf.data.Dataset
            Independent test set for full evaluation (Part 5).
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy data for distillation (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for combined distillation loss (Part 3).

        Returns
        -------
        history : dict
        """
        cfg = self.config

        logger.info("╔══════════════════════════════════════════════════════════╗")
        logger.info("║  TFF FEDERATED LEARNING CYCLE — START                   ║")
        logger.info("║  Devices: %3d | Rounds: %3d | Local epochs: %d          ║",
                     cfg.num_devices, cfg.global_rounds, cfg.local_epochs)
        logger.info("║  TFF FedAvg: ON | Comparison: %-4s                      ║",
                     "ON" if cfg.enable_comparison else "OFF")
        logger.info("╚══════════════════════════════════════════════════════════╝")

        # -- Baseline evaluation --------------------------------------- #
        logger.info("Evaluating baseline model …")
        baseline_report = self.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline", "framework": "tff"},
        )
        self.evaluator.save_report(baseline_report, tag="round_000_baseline_tff")
        logger.info(
            "Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
            baseline_report.classification.accuracy,
            baseline_report.classification.f1_macro,
            baseline_report.classification.roc_auc,
        )
        all_reports = [baseline_report]
        cycle_start = time.time()

        # ============================================================== #
        #  MAIN LOOP                                                      #
        # ============================================================== #

        for t in range(1, cfg.global_rounds + 1):
            round_start = time.time()

            info = self.execute_round(
                current_round=t,
                server_val_data=server_val_data,
                proxy_data=proxy_data,
                supervised_data=supervised_data,
            )

            # Record history
            self.history["round"].append(t)
            self.history["tff_fedavg_accuracy"].append(info["tff_fedavg_accuracy"])
            self.history["enhanced_accuracy"].append(info["enhanced_accuracy"])
            self.history["selected_clients"].append(info["selected"])
            self.history["num_accepted"].append(info["num_accepted"])
            self.history["num_rejected"].append(info["num_rejected"])
            self.history["distillation_loss"].append(info["distillation_loss"])

            round_elapsed = time.time() - round_start
            logger.info("Round %d completed in %.1fs.", t, round_elapsed)

            # -- Periodic full evaluation (Part 5) --------------------- #
            is_eval_round = (
                t % cfg.eval_every == 0
                or t == 1
                or t == cfg.global_rounds
            )
            if is_eval_round:
                logger.info("Full evaluation (round %d) …", t)
                report = self.evaluator.evaluate(
                    test_data=test_data,
                    batch_size=cfg.local_batch_size,
                    federated_round=t,
                    extra_info={
                        "framework": "tff",
                        "tff_fedavg_acc": info["tff_fedavg_accuracy"],
                        "enhanced_acc": info["enhanced_accuracy"],
                        "accepted": info["num_accepted"],
                        "rejected": info["num_rejected"],
                    },
                )
                self.evaluator.save_report(report, tag=f"tff_round_{t:03d}")
                all_reports.append(report)
                logger.info(
                    "Round %d eval — Acc: %.4f, F1: %.4f, AUC: %.4f",
                    t,
                    report.classification.accuracy,
                    report.classification.f1_macro,
                    report.classification.roc_auc,
                )

        total_elapsed = time.time() - cycle_start

        # ============================================================== #
        #  POST-CYCLE                                                     #
        # ============================================================== #

        logger.info("╔══════════════════════════════════════════════════════════╗")
        logger.info("║  TFF FEDERATED LEARNING CYCLE — COMPLETE                ║")
        logger.info("║  Total time: %.1fs                                      ║", total_elapsed)
        logger.info("╚══════════════════════════════════════════════════════════╝")

        # Comparison report
        if len(all_reports) > 1:
            self.evaluator.save_comparison_report(all_reports)

        # Save reputation ledger
        ledger_path = Path(cfg.reports_dir) / "reputation_ledger_tff_final.json"
        self.reputation_ledger.save(str(ledger_path))

        # TF Lite export
        convert_to_tflite(self.global_model, cfg.tflite_output_path, quantise=False)
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True,
        )

        # Final summary
        self._print_summary()

        return self.history

    # ------------------------------------------------------------------ #
    #  Summary                                                            #
    # ------------------------------------------------------------------ #

    def _print_summary(self) -> None:
        """Pretty-print a final training summary."""
        h = self.history
        if not h["round"]:
            return

        best_idx = int(np.argmax(h["enhanced_accuracy"]))
        best_round = h["round"][best_idx]
        best_acc = h["enhanced_accuracy"][best_idx]
        final_acc = h["enhanced_accuracy"][-1]

        stats = self.reputation_ledger.statistics()

        print("\n" + "=" * 64)
        print("  TFF FEDERATED LEARNING CYCLE — FINAL SUMMARY")
        print("=" * 64)
        print(f"  Total rounds:              {len(h['round'])}")
        print(f"  Best enhanced accuracy:    {best_acc:.4f}  (round {best_round})")
        print(f"  Final enhanced accuracy:   {final_acc:.4f}")
        print(f"  Mean enhanced accuracy:    {np.mean(h['enhanced_accuracy']):.4f}")

        if h["tff_fedavg_accuracy"][0] is not None:
            mean_tff = np.mean([a for a in h["tff_fedavg_accuracy"] if a is not None])
            print(f"  Mean TFF FedAvg accuracy:  {mean_tff:.4f}")
            deltas = [
                e - t for e, t in zip(h["enhanced_accuracy"], h["tff_fedavg_accuracy"])
                if t is not None
            ]
            if deltas:
                print(f"  Mean improvement (Δ):      {np.mean(deltas):+.4f}")

        print(f"  Total accepted updates:    {sum(h['num_accepted'])}")
        print(f"  Total rejected updates:    {sum(h['num_rejected'])}")
        print(f"  Reputation — mean:         {stats.get('mean_reputation', 0):.4f}")
        print(f"  Reputation — std:          {stats.get('std_reputation', 0):.4f}")

        kd = [l for l in h["distillation_loss"] if l is not None]
        if kd:
            print(f"  Avg distillation loss:     {np.mean(kd):.5f}")

        print(f"  TF Lite model:             {self.config.tflite_output_path}")
        print("=" * 64 + "\n")


# ====================================================================== #
#  4.  ENTRY POINT  (full production run with .h5 model)                  #
# ====================================================================== #

def main() -> None:
    """
    Full TFF FL cycle with ``effnet_ffpp_small_data.h5``.

    Requires TFF and a compatible TF version.
    """
    _require_tff()

    np.random.seed(42)
    tf.random.set_seed(42)

    config = TFFCycleConfig(
        model_path="effnet_ffpp_small_data.h5",
        num_devices=100,
        local_epochs=5,
        global_rounds=50,
        clients_per_round=15,
        local_batch_size=32,
        local_lr=1e-4,
        server_lr=1.0,
        eval_every=10,
        enable_distillation=True,
        enable_comparison=True,
    )

    cycle = TFFFederatedLearningCycle(config)

    # Load model
    model = cycle.load_global_model()
    input_shape = model.input_shape[1:]
    config.input_shape = input_shape

    # Synthetic data (replace with real FF++ c23 loaders)
    logger.info("Generating synthetic data …")
    TOTAL = config.num_devices * 10
    train_ds = generate_synthetic_data(TOTAL, input_shape, seed=1)
    val_ds = generate_synthetic_data(200, input_shape, seed=2)
    test_ds = generate_synthetic_data(300, input_shape, seed=3)
    proxy_ds = generate_proxy_data(150, input_shape, seed=4)
    sup_ds = generate_synthetic_data(100, input_shape, seed=5)

    client_data = partition_data_iid_tff(train_ds, config.num_devices)

    # Build cycle
    cycle.create_clients(client_data)
    cycle.setup_tff_process()
    cycle.setup_enhancement_modules()

    # Run
    history = cycle.run(
        server_val_data=val_ds,
        test_data=test_ds,
        proxy_data=proxy_ds,
        supervised_data=sup_ds,
    )

    logger.info("Done. History keys: %s", list(history.keys()))


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Federated Learning Cycle — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model --------------------------------------- #
    INPUT_DIM = 16
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    demo_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Demo model — {demo_model.count_params()} params")

    # ---- 2. Config (reduced) ---------------------------------------- #
    demo_config = TFFCycleConfig(
        model_path="(in-memory demo)",
        num_devices=8,
        local_epochs=1,
        global_rounds=3,
        clients_per_round=4,
        local_batch_size=16,
        local_lr=1e-3,
        server_lr=1.0,
        eval_every=1,
        enable_distillation=True,
        enable_comparison=True,
        distillation_config=DistillationConfig(
            temperature=3.0, lam=0.7, epochs=2,
            batch_size=16, learning_rate=1e-3,
        ),
    )

    # ---- 3. Synthetic data ------------------------------------------- #
    input_shape = (INPUT_DIM,)
    N_CLI = demo_config.num_devices
    TOTAL = N_CLI * 30

    train_ds = generate_synthetic_data(TOTAL, input_shape, seed=10)
    val_ds = generate_synthetic_data(100, input_shape, seed=20)
    test_ds = generate_synthetic_data(120, input_shape, seed=30)
    proxy_ds = generate_proxy_data(80, input_shape, seed=40)
    sup_ds = generate_synthetic_data(60, input_shape, seed=50)
    client_data = partition_data_iid_tff(train_ds, N_CLI)

    # ---- 4. Build cycle ---------------------------------------------- #
    cycle = TFFFederatedLearningCycle(demo_config)
    cycle.global_model = demo_model

    cycle.create_clients(client_data)

    # ---- 5. TFF-dependent vs fallback -------------------------------- #
    if TFF_AVAILABLE:
        print("\n--- TFF available — running full TFF demo ---\n")
        cycle.setup_tff_process()
        cycle.setup_enhancement_modules()

        history = cycle.run(
            server_val_data=val_ds,
            test_data=test_ds,
            proxy_data=proxy_ds,
            supervised_data=sup_ds,
        )

        print("\nRound-by-round comparison:")
        for rnd, tff_a, enh_a in zip(
            history["round"],
            history["tff_fedavg_accuracy"],
            history["enhanced_accuracy"],
        ):
            tff_str = f"{tff_a:.4f}" if tff_a is not None else "  N/A "
            print(f"  Round {rnd}: TFF FedAvg={tff_str} | Enhanced={enh_a:.4f}")

        # Cleanup demo TFLite files
        for p in [demo_config.tflite_output_path,
                  demo_config.tflite_output_path.replace(".tflite", "_quantised.tflite")]:
            Path(p).unlink(missing_ok=True)

    else:
        print(
            "⚠  TFF not installed — running enhancement-only demo.\n"
            "   This exercises Parts 1–5 without TFF's federated layer.\n"
            "   For full TFF integration, see requirements_tff.txt.\n"
        )

        # Fallback: run the non-TFF orchestrator logic as a sanity check
        cycle.setup_enhancement_modules()

        # Simulate 3 rounds using manual local training + enhancements
        for t in range(1, demo_config.global_rounds + 1):
            selected = cycle.selector.select(current_round=t)
            selected_ids = [c.client_id for c in selected]
            global_w = cycle.global_model.get_weights()

            client_updates = {}
            data_volumes = {}
            for c in selected:
                w, n = cycle._local_train(c, global_w)
                client_updates[c.client_id] = w
                data_volumes[c.client_id] = max(n, 1)
                c.metrics.last_selected_round = t

            records = cycle.validator.validate_updates(
                client_updates, data_volumes, val_ds,
            )
            new_w = cycle.validator.aggregate_weighted(records, global_w)
            cycle.global_model.set_weights(new_w)
            cycle.validator.global_model.set_weights(new_w)

            # Distillation
            cw = {r.client_id: r.contribution_weight
                  for r in records if not r.rejected and r.contribution_weight > 0}
            if len(cw) >= 2:
                run_distillation_round(
                    cycle.global_model,
                    {cid: client_updates[cid] for cid in cw},
                    cw, proxy_ds, sup_ds,
                    demo_config.distillation_config,
                )

            update_ledger_from_records(cycle.reputation_ledger, records, t)
            cycle.validator.update_reputations(records)
            cycle._sync_reputation_to_basic_ledger()

            acc = cycle.global_model.evaluate(
                val_ds.batch(16), verbose=0, return_dict=True,
            ).get("accuracy", 0.0)
            print(f"  Round {t}: Enhanced accuracy = {acc:.4f} "
                  f"(selected: {selected_ids})")

        # Quick evaluation
        report = evaluate_and_report(
            cycle.global_model, test_ds,
            model_name="demo_no_tff", federated_round=t,
        )

        print(f"\nFinal - Acc: {report.classification.accuracy:.4f}, "
              f"F1: {report.classification.f1_macro:.4f}, "
              f"AUC: {report.classification.roc_auc:.4f}")

    print("\nDone.")

Overwriting tff_federated_cycle.py


## 3. Upload Pre-trained Model

Upload `effnet_ffpp_small_data.h5` (the EfficientNet binary classifier
for DeepFake detection). Choose **one** of the methods below.

### Option A: Upload from local machine


In [ ]:
# ── Option A: Upload from your local machine ──────────────────
# from google.colab import files
# import os

# if not os.path.exists('effnet_ffpp_small_data.h5'):
   # print('Please upload effnet_ffpp_small_data.h5')
   # uploaded = files.upload()
   # print(f'Uploaded: {list(uploaded.keys())}')
#else:
   # print('Model file already present.')

# Verify
#size_mb = os.path.getsize('effnet_ffpp_small_data.h5') / (1024**2)
#print(f'Model file: effnet_ffpp_small_data.h5 ({size_mb:.1f} MB)')


### Option B: Mount Google Drive

If the model is stored in your Google Drive, mount it and copy.


In [21]:
# ── Option B: Mount Google Drive ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
#
# # Adjust the path to where your model is stored in Drive:
DRIVE_MODEL_PATH = '/content/drive/MyDrive/effnet_ffpp_small_data.h5'
#
import shutil
shutil.copy(DRIVE_MODEL_PATH, 'effnet_ffpp_small_data.h5')
print('Model copied from Google Drive.')


Mounted at /content/drive
Model copied from Google Drive.


## 4. Quick Verification

Verify that all modules import correctly and the model loads.


In [34]:
# ── Verify module imports ─────────────────────────────────────
import enhanced_client_selection
import update_validation
import knowledge_distillation
import client_reputation_ledger
import evaluation_metrics
import federated_learning_cycle
import tff_data_utils
import tff_learning_process
import tff_federated_cycle

print('All 9 modules imported successfully.')

# Verify TFF is detected
from tff_data_utils import TFF_AVAILABLE
print(f'TFF Available: {TFF_AVAILABLE}')
assert TFF_AVAILABLE, 'TFF must be available for the full cycle!'

# Load and verify model
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as _effnet_preprocess
model = tf.keras.models.load_model(
    'effnet_ffpp_small_data.h5',
    custom_objects={'preprocess_input': _effnet_preprocess},
    compile=False,
)
print(f'\nModel loaded: {model.count_params():,} params')
print(f'Input shape:  {model.input_shape}')
print(f'Output shape: {model.output_shape}')
model.summary()

All 9 modules imported successfully.
TFF Available: True

Model loaded: 4,213,668 params
Input shape:  (None, 224, 224, 3)
Output shape: (None, 1)


Model: "EfficientNetB0_FFpp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,668 (16.07 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 5. Configuration

Configure the federated learning cycle parameters.
Adjust these values based on your experiment requirements.


In [28]:
# ── Experiment Configuration ──────────────────────────────────
from tff_federated_cycle import TFFCycleConfig
from knowledge_distillation import DistillationConfig
from enhanced_client_selection import SelectionWeights
from update_validation import ContributionWeights, ClippingConfig
from client_reputation_ledger import ReputationConfig

config = TFFCycleConfig(
    # ── Core FL settings ────────────────────────────────────
    model_path='effnet_ffpp_small_data.h5',
    num_devices=100,           # Total number of simulated clients
    local_epochs=5,            # Local training epochs per client per round
    global_rounds=50,          # Total federated aggregation rounds
    clients_per_round=15,      # Clients selected each round
    local_batch_size=32,
    local_lr=1e-4,             # Client-side learning rate
    server_lr=1.0,             # Server-side learning rate (FedAvg scale)
    eval_every=10,             # Full evaluation every N rounds

    # ── TFF process settings ────────────────────────────────
    client_optimizer='adam',
    server_optimizer='sgd',
    enable_comparison=True,    # Log TFF FedAvg vs enhanced accuracy

    # ── Knowledge Distillation (Part 3) ─────────────────────
    enable_distillation=True,
    distillation_config=DistillationConfig(
        temperature=3.0,
        lam=0.7,
        epochs=3,
        batch_size=32,
        learning_rate=1e-4,
    ),

    # ── Client Selection Weights (Part 1) ───────────────────
    selection_weights=SelectionWeights(
        w_v=0.30,   # Local validation performance
        w_d=0.20,   # Data volume
        w_l=0.10,   # Latency (applied to 1 - L_i)
        w_r=0.25,   # Reputation
        w_s=0.15,   # Staleness penalty
    ),

    # ── Contribution Weights (Part 2) ───────────────────────
    contribution_weights=ContributionWeights(
        alpha=0.35,   # Validation gain
        beta=0.20,    # Similarity to global update history
        gamma=0.20,   # Data volume
        delta=0.25,   # Reputation
    ),
    clipping_config=ClippingConfig(
        clip_threshold=10.0,
        clip_value=5.0,
    ),
    harmful_threshold=0.02,

    # ── Reputation Ledger (Part 4) ──────────────────────────
    reputation_config=ReputationConfig(
        theta=0.0,
        gamma=0.10,
        decay_rate=0.99,
        floor=0.05,
        ceiling=1.0,
        initial_reputation=0.50,
        penalty_factor=0.05,
    ),

    # ── Output ──────────────────────────────────────────────
    reports_dir='reports',
    tflite_output_path='effnet_global_tff_final.tflite',
)

print('Configuration created.')
print(f'  Devices:         {config.num_devices}')
print(f'  Rounds:          {config.global_rounds}')
print(f'  Local epochs:    {config.local_epochs}')
print(f'  Clients/round:   {config.clients_per_round}')
print(f'  Distillation:    {config.enable_distillation}')
print(f'  Comparison mode: {config.enable_comparison}')


Configuration created.
  Devices:         100
  Rounds:          50
  Local epochs:    5
  Clients/round:   15
  Distillation:    True
  Comparison mode: True


## 6. Data Preparation - FaceForensics++ c23

Load the **FaceForensics++ c23** dataset from Kaggle via `kagglehub`,
extract frames from videos, preprocess them (resize to 260×260, normalize
to [0, 1]), and partition across federated clients.

**Preprocessing pipeline** (per frame):
1. Sample evenly-spaced frames from each video with OpenCV
2. Resize to (260, 260) to match the EfficientNet model input
3. Normalize pixel values: `frame / 255.0`
4. Label: 0 = Real, 1 = Fake

**Dataset splits**: 70% train / 15% validation / 15% test

> **Kaggle dataset**: `xdxd003/ff-c23`
> Contains FF++ c23 videos organized by manipulation method
> (DeepFakeDetection, Deepfakes, Face2Face, etc.) and originals.

In [ ]:
import os, shutil, json, subprocess, sys

# ── Install kagglehub + OpenCV (needed for frame extraction) ──
for pkg in ['kagglehub', 'opencv-python-headless']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)
print('Installed kagglehub & opencv-python-headless')

# ── Configure Kaggle credentials ─────────────────────────────
def setup_kaggle_credentials():
    """
    Configure Kaggle API credentials.  Tries several sources in order:
      1. Environment variables KAGGLE_USERNAME / KAGGLE_KEY (already set)
      2. ~/.kaggle/kaggle.json (already in place)
      3. /content/drive/MyDrive/kaggle.json (Google Drive mount)
      4. Colab Secrets (google.colab.userdata)
      5. Interactive upload via google.colab.files
    """
    dest_dir = os.path.expanduser("~/.kaggle")
    dest_file = os.path.join(dest_dir, "kaggle.json")

    # 1. Already set via env vars?
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        print("Kaggle credentials found in environment variables.")
        return

    # 2. Already have ~/.kaggle/kaggle.json?
    if os.path.isfile(dest_file):
        with open(dest_file) as f:
            creds = json.load(f)
        if creds.get("username") and creds.get("key"):
            print(f"Kaggle credentials found at {dest_file}")
            return

    # 3. Try Google Drive path
    drive_path = "/content/drive/MyDrive/kaggle.json"
    if os.path.isfile(drive_path):
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(drive_path, dest_dir)
        os.chmod(dest_file, 0o600)
        print(f"Copied kaggle.json from Google Drive to {dest_dir}")
        return

    # 4. Try Colab Secrets (userdata)
    try:
        from google.colab import userdata
        username = userdata.get("KAGGLE_USERNAME")
        key = userdata.get("KAGGLE_KEY")
        if username and key:
            os.environ["KAGGLE_USERNAME"] = username
            os.environ["KAGGLE_KEY"] = key
            print("Kaggle credentials loaded from Colab Secrets.")
            return
    except Exception:
        pass

    # 5. Interactive upload
    try:
        from google.colab import files
        print("No Kaggle credentials found. Please upload your kaggle.json:")
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            os.makedirs(dest_dir, exist_ok=True)
            with open(dest_file, "wb") as f:
                f.write(uploaded["kaggle.json"])
            os.chmod(dest_file, 0o600)
            print(f"Saved uploaded kaggle.json to {dest_dir}")
            return
    except Exception:
        pass

    raise RuntimeError(
        "Could not configure Kaggle credentials. Options:\n"
        "  a) Set KAGGLE_USERNAME & KAGGLE_KEY env vars\n"
        "  b) Place kaggle.json in ~/.kaggle/\n"
        "  c) Mount Google Drive with kaggle.json at /content/drive/MyDrive/\n"
        "  d) Add KAGGLE_USERNAME & KAGGLE_KEY to Colab Secrets"
    )

setup_kaggle_credentials()

Kaggle credentials found at /root/.kaggle/kaggle.json


In [ ]:
import os, glob, pathlib, shutil
import numpy as np
import tensorflow as tf
import cv2
import kagglehub
from tff_data_utils import partition_data_iid_tff
from tff_federated_cycle import TFFFederatedLearningCycle

np.random.seed(42)
tf.random.set_seed(42)

# ── Register EfficientNet's preprocess_input globally ─────────
from tensorflow.keras.applications.efficientnet import preprocess_input as _effnet_preprocess
tf.keras.utils.get_custom_objects()['preprocess_input'] = _effnet_preprocess
print('Registered preprocess_input in Keras custom objects.')

# ══════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════
IMG_SIZE           = (260, 260)       # Must match model input
TRAIN_RATIO        = 0.70
VAL_RATIO          = 0.15
TEST_RATIO         = 0.15
MAX_FRAMES         = None             # Cap total frames (None = use all)
PROXY_FRACTION     = 0.10
SUP_FRACTION       = 0.08
FRAMES_PER_VIDEO   = 10               # Evenly-spaced frames sampled per video

KAGGLE_DATASET     = 'xdxd003/ff-c23'

# ══════════════════════════════════════════════════════════════
#  1. DOWNLOAD FROM KAGGLE VIA kagglehub
# ══════════════════════════════════════════════════════════════
def download_ffpp(dataset_slug: str) -> str:
    """Download the FF++ C23 dataset using kagglehub. Returns local cache path."""
    print(f'  Downloading {dataset_slug} via kagglehub ...')
    path = kagglehub.dataset_download(dataset_slug)
    print(f'  Dataset cached at: {path}')
    return path


# ══════════════════════════════════════════════════════════════
#  2. DISCOVER VIDEOS / IMAGES + LABELS
# ══════════════════════════════════════════════════════════════
# FF++ manipulation methods → fake (label 1)
_FAKE_KEYWORDS = {'deepfakedetection', 'deepfakes', 'face2face',
                  'faceswap', 'neuraltextures', 'faceshifter',
                  'fake', 'manipulated'}
# Original sequences → real (label 0)
_REAL_KEYWORDS = {'youtube', 'originalsequences', 'original',
                  'actors', 'real'}

def _label_from_path(filepath: str) -> int:
    """Determine label (0=real, 1=fake) by checking path components."""
    parts = [p.lower().replace('_', '').replace('-', '').replace(' ', '')
             for p in pathlib.Path(filepath).parts]
    for p in parts:
        for kw in _FAKE_KEYWORDS:
            if kw in p:
                return 1
        for kw in _REAL_KEYWORDS:
            if kw in p:
                return 0
    return -1

_IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
_VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}

def discover_samples(data_dir: str):
    """Walk dataset and return (image_samples, video_samples) as [(path, label)]."""
    images, videos = [], []
    for root, _, files in os.walk(data_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            ext = pathlib.Path(fname).suffix.lower()
            label = _label_from_path(fpath)
            if label == -1:
                continue
            if ext in _IMAGE_EXTS:
                images.append((fpath, label))
            elif ext in _VIDEO_EXTS:
                videos.append((fpath, label))
    return images, videos


# ══════════════════════════════════════════════════════════════
#  3. FRAME EXTRACTION + PREPROCESSING
# ══════════════════════════════════════════════════════════════
def extract_frames(video_path: str, n_frames: int = FRAMES_PER_VIDEO,
                   img_size=IMG_SIZE):
    """Extract n_frames evenly-spaced frames from a video, resize & normalize."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return []

    indices = np.linspace(0, total - 1, min(n_frames, total), dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (img_size[1], img_size[0]))  # (W, H) for cv2
        frame = frame.astype(np.float32) / 255.0               # Eq. (2): Normalize
        frames.append(frame)
    cap.release()
    return frames


def preprocess_image(path: str, img_size=IMG_SIZE):
    """Load a single image file, resize and normalize to [0,1]."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, img_size)
    img = img / 255.0
    return img.numpy().astype(np.float32)


def load_all_samples(image_samples, video_samples, img_size=IMG_SIZE,
                     max_frames=None, frames_per_video=FRAMES_PER_VIDEO):
    """Load images and/or extract video frames into numpy arrays."""
    all_images, all_labels = [], []
    errors = 0

    # ── Pre-extracted images ──────────────────────────────────
    for i, (path, label) in enumerate(image_samples):
        try:
            img = preprocess_image(path, img_size)
            all_images.append(img)
            all_labels.append(label)
        except Exception:
            errors += 1
        if (i + 1) % 5000 == 0:
            print(f'    Images: {i+1}/{len(image_samples)} ...')
    if image_samples:
        print(f'    Loaded {len(all_images)} images ({errors} errors)')

    # ── Video frame extraction ────────────────────────────────
    v_frames = 0
    v_errors = 0
    for i, (path, label) in enumerate(video_samples):
        try:
            frames = extract_frames(path, frames_per_video, img_size)
            for f in frames:
                all_images.append(f)
                all_labels.append(label)
                v_frames += 1
        except Exception:
            v_errors += 1
        if (i + 1) % 200 == 0:
            print(f'    Videos: {i+1}/{len(video_samples)}, '
                  f'frames extracted: {v_frames} ...')
    if video_samples:
        print(f'    Extracted {v_frames} frames from '
              f'{len(video_samples)} videos ({v_errors} errors)')

    images = np.array(all_images, dtype=np.float32)
    labels = np.array(all_labels, dtype=np.float32)

    # Optional cap on total frames
    if max_frames and len(images) > max_frames:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(images), max_frames, replace=False)
        images, labels = images[idx], labels[idx]

    return images, labels


# ══════════════════════════════════════════════════════════════
#  4. SPLIT INTO TRAIN / VAL / TEST (70:15:15)
# ══════════════════════════════════════════════════════════════
def split_dataset(images, labels, train_r=TRAIN_RATIO, val_r=VAL_RATIO):
    """Shuffle then split by ratio."""
    n = len(images)
    idx = np.random.permutation(n)
    images, labels = images[idx], labels[idx]
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    train_x, train_y = images[:n_train],              labels[:n_train]
    val_x,   val_y   = images[n_train:n_train+n_val], labels[n_train:n_train+n_val]
    test_x,  test_y  = images[n_train+n_val:],        labels[n_train+n_val:]
    return (train_x, train_y), (val_x, val_y), (test_x, test_y)


# ══════════════════════════════════════════════════════════════
#  EXECUTE THE PIPELINE
# ══════════════════════════════════════════════════════════════
print('=' * 60)
print('  FaceForensics++ c23 Data Preparation')
print('=' * 60)

# Step 1: Download
print('\n[1/5] Downloading FF++ c23 from Kaggle ...')
DATA_DIR = download_ffpp(KAGGLE_DATASET)

# Step 2: Discover
print('\n[2/5] Discovering samples ...')
image_samples, video_samples = discover_samples(DATA_DIR)
n_img_r = sum(1 for _, l in image_samples if l == 0)
n_img_f = sum(1 for _, l in image_samples if l == 1)
n_vid_r = sum(1 for _, l in video_samples if l == 0)
n_vid_f = sum(1 for _, l in video_samples if l == 1)
print(f'  Images: {len(image_samples)} ({n_img_r} real, {n_img_f} fake)')
print(f'  Videos: {len(video_samples)} ({n_vid_r} real, {n_vid_f} fake)')
if video_samples:
    est = len(video_samples) * FRAMES_PER_VIDEO
    print(f'  Estimated frames after extraction: ~{est}')

assert len(image_samples) + len(video_samples) > 0, (
    f'No images or videos found in {DATA_DIR}.\n'
    f'Check the dataset structure.'
)

# Step 3: Load & preprocess
print(f'\n[3/5] Loading and preprocessing '
      f'(resize to {IMG_SIZE}, normalize to [0,1]) ...')
if video_samples:
    print(f'  Sampling {FRAMES_PER_VIDEO} evenly-spaced frames per video')
images, labels = load_all_samples(
    image_samples, video_samples, IMG_SIZE, MAX_FRAMES, FRAMES_PER_VIDEO
)
print(f'  Total: {images.shape} (dtype={images.dtype}), labels: {labels.shape}')
print(f'  Label distribution: real={int((labels==0).sum())}, '
      f'fake={int((labels==1).sum())}')
print(f'  Pixel range: [{images.min():.3f}, {images.max():.3f}]')

# Step 4: Split
print(f'\n[4/5] Splitting into train/val/test '
      f'({TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%}) ...')
(train_x, train_y), (val_x, val_y), (test_x, test_y) = \
    split_dataset(images, labels)
print(f'  Train: {train_x.shape[0]} samples')
print(f'  Val:   {val_x.shape[0]} samples')
print(f'  Test:  {test_x.shape[0]} samples')

# Free the large combined array
del images, labels

# Create tf.data.Datasets
train_ds = tf.data.Dataset.from_tensor_slices((train_x, train_y))
val_ds   = tf.data.Dataset.from_tensor_slices((val_x, val_y))
test_ds  = tf.data.Dataset.from_tensor_slices((test_x, test_y))

# Proxy data (unlabelled) for knowledge distillation (Part 3)
n_proxy = max(1, int(train_x.shape[0] * PROXY_FRACTION))
proxy_ds = tf.data.Dataset.from_tensor_slices(train_x[:n_proxy])

# Supervised data for combined distillation loss
n_sup = max(1, int(train_x.shape[0] * SUP_FRACTION))
sup_ds = tf.data.Dataset.from_tensor_slices((train_x[:n_sup], train_y[:n_sup]))

print(f'  Proxy (unlabelled):  {n_proxy} samples')
print(f'  Supervised (for KD): {n_sup} samples')

# Step 5: Partition across federated clients (IID)
print(f'\n[5/5] Partitioning training data across '
      f'{config.num_devices} clients (IID) ...')
client_data = partition_data_iid_tff(train_ds, config.num_devices, seed=42)
samples_per_client = train_x.shape[0] // config.num_devices
print(f'  {len(client_data)} client shards, ~{samples_per_client} samples each')

# ── Initialize FL cycle ──────────────────────────────────────
cycle = TFFFederatedLearningCycle(config)
model = cycle.load_global_model()
input_shape = model.input_shape[1:]
config.input_shape = input_shape
print(f'\nModel input shape: {input_shape}')

assert input_shape == (*IMG_SIZE, 3), (
    f'Model expects {input_shape} but images are {(*IMG_SIZE, 3)}. '
    f'Adjust IMG_SIZE to match.'
)

cycle.create_clients(client_data)
print(f'Created {len(cycle.clients)} federated clients.')

# Free raw arrays (data lives in tf.data.Datasets now)
del train_x, train_y, val_x, val_y, test_x, test_y

print('\n' + '=' * 60)
print('  Data preparation complete')
print('=' * 60)

Registered preprocess_input in Keras custom objects.
  FaceForensics++ c23 Data Preparation

[1/5] Downloading FF++ c23 face images from Kaggle ...
  Kaggle config check: Configuration values from /root/.kaggle
- username: averageweebo101
- path: None
- proxy: None
- competition: None
  First attempt failed (--unzip). stdout: 403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/ondercaglar/faceforensics-face-images
  stderr: 


RuntimeError: Kaggle download failed.
stdout: 403 Client Error: Forbidden for url: https://www.kaggle.com/api/v1/datasets/metadata/ondercaglar/faceforensics-face-images

stderr: 
Make sure your Kaggle API key is configured:
  1. Go to kaggle.com -> Account -> Create API Token
  2. Upload kaggle.json or set KAGGLE_USERNAME & KAGGLE_KEY

## 7. Initialize TFF Process & Enhancement Modules

Build the TFF Weighted FedAvg learning process and wire up
all five enhancement modules.


In [ ]:
# ── Setup TFF federated learning process ─────────────────────
cycle.setup_tff_process()
print('TFF Learning Process initialised.')

# ── Setup enhancement modules (Parts 1-5) ────────────────────
cycle.setup_enhancement_modules()
print('Enhancement modules (Parts 1-5) initialised.')
print('\n✅ Ready to run the federated learning cycle.')


## 8. Run the Full TFF Federated Learning Cycle

Execute the complete federated training loop:
- Each round: Client Selection → TFF FedAvg → Update Validation →
  Knowledge Distillation → Reputation Update → Evaluation
- Comparison mode logs both TFF FedAvg and enhanced accuracy
- Full evaluation reports saved every `eval_every` rounds

> **Runtime:** With 100 clients, 50 rounds, and EfficientNet,
> this may take significant time. Consider reducing
> `global_rounds` or `num_devices` for initial testing.


In [ ]:
# ── Run the full federated learning cycle ────────────────────
history = cycle.run(
    server_val_data=val_ds,
    test_data=test_ds,
    proxy_data=proxy_ds,
    supervised_data=sup_ds,
)

print('\n✅ Federated Learning Cycle complete!')
print(f'History keys: {list(history.keys())}')


## 9. Results Visualization

Visualise the training history with matplotlib.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── 9a. Accuracy: TFF FedAvg vs Enhanced ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

rounds = history['round']
tff_acc = history['tff_fedavg_accuracy']
enh_acc = history['enhanced_accuracy']

# Plot 1: Accuracy comparison
ax = axes[0, 0]
if tff_acc[0] is not None:
    ax.plot(rounds, tff_acc, 'b--o', label='TFF FedAvg', markersize=3, alpha=0.7)
ax.plot(rounds, enh_acc, 'r-s', label='Enhanced (Ours)', markersize=3, alpha=0.7)
ax.set_xlabel('Federated Round')
ax.set_ylabel('Accuracy')
ax.set_title('TFF FedAvg vs Enhanced Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy improvement (delta)
ax = axes[0, 1]
if tff_acc[0] is not None:
    deltas = [e - t for e, t in zip(enh_acc, tff_acc) if t is not None]
    ax.bar(rounds[:len(deltas)], deltas, color=['green' if d >= 0 else 'red' for d in deltas], alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Federated Round')
    ax.set_ylabel('Accuracy Improvement (Δ)')
    ax.set_title('Enhanced − FedAvg Accuracy Delta')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Comparison mode disabled', ha='center', va='center', transform=ax.transAxes)

# Plot 3: Accepted vs Rejected updates
ax = axes[1, 0]
ax.bar(rounds, history['num_accepted'], label='Accepted', color='green', alpha=0.7)
ax.bar(rounds, history['num_rejected'], bottom=history['num_accepted'],
       label='Rejected', color='red', alpha=0.7)
ax.set_xlabel('Federated Round')
ax.set_ylabel('Number of Updates')
ax.set_title('Accepted vs Rejected Client Updates')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Distillation loss
ax = axes[1, 1]
kd_losses = history['distillation_loss']
kd_rounds = [r for r, l in zip(rounds, kd_losses) if l is not None]
kd_values = [l for l in kd_losses if l is not None]
if kd_values:
    ax.plot(kd_rounds, kd_values, 'g-^', label='KD Loss', markersize=4)
    ax.set_xlabel('Federated Round')
    ax.set_ylabel('Distillation Loss')
    ax.set_title('Knowledge Distillation Loss Over Rounds')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No distillation data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('results_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results plot saved to results_overview.png')


In [ ]:
# ── 9b. Reputation Distribution ──────────────────────────────
import matplotlib.pyplot as plt

reps = cycle.reputation_ledger.all_reputations()
rep_values = list(reps.values())
stats = cycle.reputation_ledger.statistics()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(rep_values, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(stats['mean_reputation'], color='red', linestyle='--',
           label=f"Mean: {stats['mean_reputation']:.3f}")
ax.axvline(stats['median_reputation'], color='orange', linestyle='--',
           label=f"Median: {stats['median_reputation']:.3f}")
ax.set_xlabel('Reputation Score')
ax.set_ylabel('Number of Clients')
ax.set_title('Final Client Reputation Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Top/Bottom clients
ax = axes[1]
ranked = cycle.reputation_ledger.ranked()
top_10 = ranked[:10]
bottom_10 = ranked[-10:]
combined = top_10 + bottom_10
names = [c[0] for c in combined]
values = [c[1] for c in combined]
colors = ['green'] * len(top_10) + ['red'] * len(bottom_10)
ax.barh(range(len(combined)), values, color=colors, alpha=0.7)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('Reputation Score')
ax.set_title('Top 10 & Bottom 10 Clients by Reputation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reputation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nReputation stats:')
for k, v in stats.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


In [ ]:
# ── 9c. Round-by-round Summary Table ─────────────────────────
print(f"{'Round':>6} | {'TFF Acc':>8} | {'Enh Acc':>8} | "
      f"{'Delta':>8} | {'Accepted':>8} | {'Rejected':>8} | {'KD Loss':>10}")
print('-' * 72)

for i, rnd in enumerate(history['round']):
    tff_a = history['tff_fedavg_accuracy'][i]
    enh_a = history['enhanced_accuracy'][i]
    tff_str = f'{tff_a:.4f}' if tff_a is not None else '  N/A '
    delta = f'{enh_a - tff_a:+.4f}' if tff_a is not None else '  N/A '
    kd = history['distillation_loss'][i]
    kd_str = f'{kd:.5f}' if kd is not None else '    N/A   '
    print(f'{rnd:>6} | {tff_str:>8} | {enh_a:>8.4f} | '
          f'{delta:>8} | {history["num_accepted"][i]:>8} | '
          f'{history["num_rejected"][i]:>8} | {kd_str:>10}')


## 10. Final Evaluation Report

Run a comprehensive final evaluation on the test set.


In [ ]:
from evaluation_metrics import evaluate_and_report

# Full evaluation on the test set
final_report = evaluate_and_report(
    model=cycle.global_model,
    test_data=test_ds,
    model_name='effnet_global_tff_final',
    reports_dir=config.reports_dir,
    federated_round=config.global_rounds,
    extra_info={
        'framework': 'tensorflow_federated',
        'num_devices': config.num_devices,
        'total_rounds': config.global_rounds,
        'local_epochs': config.local_epochs,
        'enhancement_modules': [
            'enhanced_client_selection',
            'update_validation',
            'knowledge_distillation',
            'client_reputation_ledger',
        ],
    },
    tag='final',
)

print('\n' + '=' * 60)
print('  FINAL EVALUATION RESULTS')
print('=' * 60)
cls = final_report.classification
print(f'  Accuracy:         {cls.accuracy:.4f}')
print(f'  F1 Score (macro): {cls.f1_macro:.4f}')
print(f'  F1 Score (wt.):   {cls.f1_weighted:.4f}')
print(f'  Precision:        {cls.precision_macro:.4f}')
print(f'  Recall:           {cls.recall_macro:.4f}')
print(f'  ROC-AUC:          {cls.roc_auc:.4f}')
print(f'\n  Confusion Matrix:')
if cls.confusion_matrix:
    print(f'    TN={cls.confusion_matrix[0][0]:>5}  FP={cls.confusion_matrix[0][1]:>5}')
    print(f'    FN={cls.confusion_matrix[1][0]:>5}  TP={cls.confusion_matrix[1][1]:>5}')

lat = final_report.latency
print(f'\n  Inference Latency:')
print(f'    Mean: {lat.mean_ms:.2f} ms | P95: {lat.p95_ms:.2f} ms')

sz = final_report.model_size
print(f'\n  Model Size:')
print(f'    Parameters: {sz.total_params:,}')
print(f'    File size:  {sz.file_size_mb:.2f} MB')
print('=' * 60)


## 11. Export & Download Results

Download the trained model, TF Lite exports, reports, and
reputation ledger.


In [ ]:
import os
import zipfile

# ── Save final Keras model ───────────────────────────────────
cycle.global_model.save('effnet_global_tff_trained.h5')
print('Saved: effnet_global_tff_trained.h5')

# ── List all output files ────────────────────────────────────
output_files = []

# Model files
for f in ['effnet_global_tff_trained.h5',
          config.tflite_output_path,
          config.tflite_output_path.replace('.tflite', '_quantised.tflite')]:
    if os.path.exists(f):
        output_files.append(f)

# Reports
if os.path.isdir('reports'):
    for f in os.listdir('reports'):
        output_files.append(os.path.join('reports', f))

# Visualisation plots
for f in ['results_overview.png', 'reputation_distribution.png']:
    if os.path.exists(f):
        output_files.append(f)

print(f'\nOutput files ({len(output_files)} total):')
for f in output_files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  {f} ({size / 1024:.1f} KB)')

# ── Create ZIP archive ───────────────────────────────────────
zip_name = 'fl_cycle_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        zf.write(f)
print(f'\nCreated: {zip_name} ({os.path.getsize(zip_name) / 1024:.1f} KB)')


In [ ]:
# ── Download results ─────────────────────────────────────────
from google.colab import files
files.download('fl_cycle_results.zip')
print('Download initiated.')


---

## Appendix A: Quick Demo (Reduced Settings)

If the full 100-device / 50-round cycle takes too long, use this
cell for a quick smoke-test with reduced parameters.


In [ ]:
# ── Quick Demo: 8 clients, 3 rounds ─────────────────────────
# Uncomment and run this cell for a faster test.
# Comment out Section 8 above to avoid running the full cycle.

# from tff_federated_cycle import TFFFederatedLearningCycle, TFFCycleConfig
# from tff_data_utils import generate_synthetic_data, generate_proxy_data, partition_data_iid_tff
# from knowledge_distillation import DistillationConfig
# import numpy as np, tensorflow as tf
#
# np.random.seed(42); tf.random.set_seed(42)
#
# quick_config = TFFCycleConfig(
#     model_path='effnet_ffpp_small_data.h5',
#     num_devices=8, local_epochs=1, global_rounds=3,
#     clients_per_round=4, local_batch_size=16,
#     local_lr=1e-3, eval_every=1,
#     enable_distillation=True, enable_comparison=True,
#     distillation_config=DistillationConfig(
#         temperature=3.0, lam=0.7, epochs=2,
#         batch_size=16, learning_rate=1e-3),
# )
#
# quick_cycle = TFFFederatedLearningCycle(quick_config)
# model = quick_cycle.load_global_model()
# input_shape = model.input_shape[1:]
# quick_config.input_shape = input_shape
#
# train = generate_synthetic_data(8 * 30, input_shape, seed=10)
# val   = generate_synthetic_data(100, input_shape, seed=20)
# test  = generate_synthetic_data(120, input_shape, seed=30)
# proxy = generate_proxy_data(80, input_shape, seed=40)
# sup   = generate_synthetic_data(60, input_shape, seed=50)
#
# client_data = partition_data_iid_tff(train, 8)
# quick_cycle.create_clients(client_data)
# quick_cycle.setup_tff_process()
# quick_cycle.setup_enhancement_modules()
#
# history = quick_cycle.run(
#     server_val_data=val, test_data=test,
#     proxy_data=proxy, supervised_data=sup,
# )
#
# for r, t_a, e_a in zip(history['round'], history['tff_fedavg_accuracy'], history['enhanced_accuracy']):
#     t_s = f'{t_a:.4f}' if t_a else 'N/A'
#     print(f'  Round {r}: FedAvg={t_s} | Enhanced={e_a:.4f}')
